# Adaptive Retrieval-Augmented Generation for Reducing Hallucinations in Large Language Models

## Research notebook

This notebook studies retrieval and answer-extraction strategies on a structured question-answering benchmark built from historical Indian Premier League (IPL) match data from Cricsheet. It compares fixed BM25 retrieval, metadata-assisted retrieval, FLAN-T5-small generation, and deterministic extraction from retrieved evidence.

## Experimental scope

- **Question types:** match winners, innings totals, and individual batter runs.
- **Evaluation:** 180 questions sampled from an existing held-out test partition; this is not an independent external benchmark.
- **Retrieval strategies:** fixed BM25 and task-specific metadata-assisted rules.
- **Answering strategies:** FLAN-T5-small generation and deterministic extraction from retrieved passages.

Retrieval quality, answer accuracy, and evidence support are evaluated as distinct outcomes. The extraction rules are tailored to the structured passage format, so high accuracy on this benchmark should not be interpreted as proof of general-purpose hallucination reduction.

> **Reproducibility note:** Run the notebook from top to bottom in a fresh Google Colab runtime before treating the results as independently reproduced. Keep a backup of the original notebook while validating execution.

## Environment and reproducibility

The recorded environment is Python 3.13.18 on Linux, with CPU-only PyTorch 2.11.0 and the package versions documented in `requirements.txt` and `ipl_research_environment_versions.csv`.

For a fresh runtime, install and verify dependencies before executing the notebook. Runtime package compatibility can change over time, so the recorded environment should be treated as a reproducibility reference rather than a guarantee of identical results.

The experiment uses a fixed random seed where specified and stores a frozen evaluation-question list and configuration. Do not tune the method using the frozen evaluation results. Run the notebook from top to bottom in a clean runtime when validating reproducibility.


## How to interpret the results

The principal retrieval metric is exact target-passage Recall@k. Answer accuracy is scored against the benchmark's expected answer. These are different outcomes: retrieving a passage does not guarantee that a generative model will extract the correct answer from it.

The reported results are specific to this template-generated IPL benchmark. Strong performance from rule-based extraction reflects the structured format of the passages and is not, by itself, evidence of broad hallucination reduction in open-domain language models.


In [1]:
# IPL Adaptive RAG Research
# Step 1: Import basic tools and record our experiment settings.

import os
import json
import random
import platform
from datetime import datetime, timezone

SEED = 42
random.seed(SEED)

PROJECT_NAME = "IPL Adaptive RAG Research"
DATA_SOURCE = "Cricsheet IPL historical match data"
DATA_SOURCE_URL = "https://cricsheet.org/downloads/"
DATA_LICENSE = "Open Data Commons Attribution License (ODC-BY)"

print("Project:", PROJECT_NAME)
print("Data source:", DATA_SOURCE)
print("Source URL:", DATA_SOURCE_URL)
print("Licence:", DATA_LICENSE)
print("Python version:", platform.python_version())
print("Experiment seed:", SEED)
print("Setup timestamp (UTC):", datetime.now(timezone.utc).isoformat())

Project: IPL Adaptive RAG Research
Data source: Cricsheet IPL historical match data
Source URL: https://cricsheet.org/downloads/
Licence: Open Data Commons Attribution License (ODC-BY)
Python version: 3.13.18
Experiment seed: 42
Setup timestamp (UTC): 2026-10-28T13:01:33.271746+00:00


In [2]:
# Step 2: Download historical IPL match data from Cricsheet

import urllib.request
import zipfile
import os

DATA_URL = "https://cricsheet.org/downloads/ipl_json.zip"
ZIP_PATH = "/content/ipl_json.zip"
DATA_DIR = "/content/ipl_json"

print("Downloading IPL dataset...")

urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

print("Download complete!")
print("ZIP file size:", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2), "MB")

# Extract the downloaded archive
os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

# Count the extracted JSON files
json_files = [
    filename for filename in os.listdir(DATA_DIR)
    if filename.lower().endswith(".json")
]

print("JSON files extracted:", len(json_files))
print("Example filenames:", json_files[:5])

Download complete!
ZIP file size: 4.95 MB
JSON files extracted: 1243
Example filenames: ['419118.json', '336012.json', '501240.json', '1359491.json', '548331.json']


In [3]:
# Step 3: Inspect the structure of one match file

import json
import os

# Select one match file
sample_file = os.path.join(DATA_DIR, json_files[0])

with open(sample_file, "r", encoding="utf-8") as f:
    sample_match = json.load(f)

print("Sample file:", json_files[0])
print("\nTop-level sections:")
print(list(sample_match.keys()))

print("\nMetadata:")
print(json.dumps(sample_match.get("meta", {}), indent=2))

print("\nInfo section:")
print(json.dumps(sample_match.get("info", {}), indent=2)[:4000])

print("\nNumber of innings:", len(sample_match.get("innings", [])))

Sample file: 419118.json

Top-level sections:
['meta', 'info', 'innings']

Metadata:
{
  "data_version": "1.2.0",
  "created": "2011-04-29",
  "revision": 2
}

Info section:
{
  "balls_per_over": 6,
  "city": "Ahmedabad",
  "dates": [
    "2010-03-20"
  ],
  "event": {
    "match_number": 13,
    "name": "Indian Premier League"
  },
  "gender": "male",
  "match_type": "T20",
  "officials": {
    "match_referees": [
      "AJ Pycroft"
    ],
    "reserve_umpires": [
      "S Dua"
    ],
    "tv_umpires": [
      "BG Jerling"
    ],
    "umpires": [
      "RE Koertzen",
      "RB Tiffin"
    ]
  },
  "outcome": {
    "by": {
      "runs": 34
    },
    "winner": "Rajasthan Royals"
  },
  "overs": 20,
  "player_of_match": [
    "AA Jhunjhunwala"
  ],
  "players": {
    "Kolkata Knight Riders": [
      "MK Tiwary",
      "BJ Hodge",
      "CA Pujara",
      "SC Ganguly",
      "OA Shah",
      "AD Mathews",
      "WP Saha",
      "M Kartik",
      "AB Dinda",
      "SE Bond",
      "I Shar

In [4]:
# Step 4: Inspect innings and ball-by-ball records

for innings_index, innings in enumerate(sample_match["innings"], start=1):
    print(f"\n{'=' * 50}")
    print("INNINGS:", innings_index)
    print("Batting team:", innings.get("team"))
    print("Available innings fields:", list(innings.keys()))

    overs = innings.get("overs", [])
    print("Number of overs recorded:", len(overs))

    if overs:
        first_over = overs[0]
        print("\nFirst over fields:")
        print(list(first_over.keys()))

        deliveries = first_over.get("deliveries", [])
        print("Deliveries in first over:", len(deliveries))

        if deliveries:
            print("\nFirst delivery:")
            print(json.dumps(deliveries[0], indent=2))

            print("\nSecond delivery (if available):")
            if len(deliveries) > 1:
                print(json.dumps(deliveries[1], indent=2))
            else:
                print("No second delivery in this over.")


INNINGS: 1
Batting team: Rajasthan Royals
Available innings fields: ['team', 'overs', 'powerplays']
Number of overs recorded: 20

First over fields:
['over', 'deliveries']
Deliveries in first over: 6

First delivery:
{
  "actual_delivery": "0.1",
  "batter": "MJ Lumb",
  "bowler": "AB Dinda",
  "non_striker": "NV Ojha",
  "runs": {
    "batter": 0,
    "extras": 0,
    "total": 0
  },
  "wickets": [
    {
      "kind": "lbw",
      "player_out": "MJ Lumb"
    }
  ]
}

Second delivery (if available):
{
  "actual_delivery": "0.2",
  "batter": "FY Fazal",
  "bowler": "AB Dinda",
  "non_striker": "NV Ojha",
  "runs": {
    "batter": 0,
    "extras": 0,
    "total": 0
  }
}

INNINGS: 2
Batting team: Kolkata Knight Riders
Available innings fields: ['team', 'overs', 'powerplays', 'target']
Number of overs recorded: 20

First over fields:
['over', 'deliveries']
Deliveries in first over: 6

First delivery:
{
  "actual_delivery": "0.1",
  "batter": "MK Tiwary",
  "bowler": "YK Pathan",
  "non_s

In [5]:
# Step 5: Extract match-level facts from all IPL JSON files

import pandas as pd
import os
import json

match_records = []
failed_files = []

for filename in json_files:
    filepath = os.path.join(DATA_DIR, filename)

    try:
        with open(filepath, "r", encoding="utf-8") as f:
            match = json.load(f)

        info = match.get("info", {})
        outcome = info.get("outcome", {})
        winner = outcome.get("winner")

        match_records.append({
            "match_id": filename.replace(".json", ""),
            "date": info.get("dates", [None])[0],
            "season": info.get("season"),
            "city": info.get("city"),
            "venue": info.get("venue"),
            "teams": info.get("teams", []),
            "winner": winner,
            "player_of_match": info.get("player_of_match", []),
            "toss_winner": info.get("toss", {}).get("winner"),
            "toss_decision": info.get("toss", {}).get("decision"),
            "innings_count": len(match.get("innings", []))
        })

    except Exception as e:
        failed_files.append((filename, str(e)))

matches_df = pd.DataFrame(match_records)

print("Successfully processed matches:", len(matches_df))
print("Files that failed:", len(failed_files))
print("Unique seasons:", matches_df["season"].nunique())

print("\nSeasons available:")
print(sorted(matches_df["season"].dropna().astype(str).unique(), key=str))

print("\nSample records:")
display(matches_df.head(5))

Successfully processed matches: 1243
Files that failed: 0
Unique seasons: 24

Seasons available:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

Sample records:


,match_id,date,season,city,venue,teams,winner,player_of_match,toss_winner,toss_decision,innings_count
0,419118,2010-03-20,2009/10,Ahmedabad,"Sardar Patel Stadium, Motera","[Rajasthan Royals, Kolkata Knight Riders]",Rajasthan Royals,[AA Jhunjhunwala],Rajasthan Royals,bat,2
1,336012,2008-05-28,2007/08,Bangalore,M Chinnaswamy Stadium,"[Royal Challengers Bangalore, Mumbai Indians]",Mumbai Indians,[CRD Fernando],Mumbai Indians,field,2
2,501240,2011-05-04,2011,Chennai,"MA Chidambaram Stadium, Chepauk","[Chennai Super Kings, Rajasthan Royals]",Chennai Super Kings,[MEK Hussey],Rajasthan Royals,bat,2
3,1359491,2023-04-12,2023,Chennai,"MA Chidambaram Stadium, Chepauk, Chennai","[Rajasthan Royals, Chennai Super Kings]",Rajasthan Royals,[R Ashwin],Chennai Super Kings,field,2
4,548331,2012-04-20,2012,Chandigarh,"Punjab Cricket Association Stadium, Mohali","[Kings XI Punjab, Royal Challengers Bangalore]",Royal Challengers Bangalore,[CH Gayle],Royal Challengers Bangalore,field,2


In [6]:
# STEP 5A: FIX SEASON DATA TYPES AND VERIFY THE MATCH DATASET
# Purpose:
# 1. Convert season values to a consistent text format.
# 2. Display the number of successfully processed matches.
# 3. Display all available seasons in sorted order.
# 4. Preview five match records to check the extracted data.

# Step 1: Convert season values to strings so mixed numbers and text can be sorted.
matches_df["season"] = matches_df["season"].astype(str)

# Step 2: Verify the number of processed matches and unique seasons.
print("Successfully processed matches:", len(matches_df))
print("Unique seasons:", matches_df["season"].nunique())

# Step 3: Display the available seasons in sorted order.
print("\nSeasons available:")
print(sorted(matches_df["season"].dropna().unique()))

# Step 4: Preview the first five match records.
print("\nSample records:")
display(matches_df.head(5))

Successfully processed matches: 1243
Unique seasons: 19

Seasons available:
['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020/21', '2021', '2022', '2023', '2024', '2025', '2026']

Sample records:


,match_id,date,season,city,venue,teams,winner,player_of_match,toss_winner,toss_decision,innings_count
0,419118,2010-03-20,2009/10,Ahmedabad,"Sardar Patel Stadium, Motera","[Rajasthan Royals, Kolkata Knight Riders]",Rajasthan Royals,[AA Jhunjhunwala],Rajasthan Royals,bat,2
1,336012,2008-05-28,2007/08,Bangalore,M Chinnaswamy Stadium,"[Royal Challengers Bangalore, Mumbai Indians]",Mumbai Indians,[CRD Fernando],Mumbai Indians,field,2
2,501240,2011-05-04,2011,Chennai,"MA Chidambaram Stadium, Chepauk","[Chennai Super Kings, Rajasthan Royals]",Chennai Super Kings,[MEK Hussey],Rajasthan Royals,bat,2
3,1359491,2023-04-12,2023,Chennai,"MA Chidambaram Stadium, Chepauk, Chennai","[Rajasthan Royals, Chennai Super Kings]",Rajasthan Royals,[R Ashwin],Chennai Super Kings,field,2
4,548331,2012-04-20,2012,Chandigarh,"Punjab Cricket Association Stadium, Mohali","[Kings XI Punjab, Royal Challengers Bangalore]",Royal Challengers Bangalore,[CH Gayle],Royal Challengers Bangalore,field,2


In [7]:
# STEP 6: CALCULATE INNINGS TOTALS AND PLAYER BATTING STATISTICS
# Purpose:
# 1. Load each match's ball-by-ball data.
# 2. Calculate each innings' total runs and wickets.
# 3. Calculate batter runs and balls faced.
# 4. Store the results in separate tables for later QA generation.
#
# Note:
# - Batter runs come from runs["batter"].
# - Innings totals come from runs["total"].
# - Wickets are counted from recorded wicket events.
# - Balls faced require special handling for wides, so we will not
#   calculate balls faced in this step.

innings_records = []
batter_records = []

# Step 1: Process every match JSON file.
for filename in json_files:
    filepath = os.path.join(DATA_DIR, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        match = json.load(f)

    match_id = filename.replace(".json", "")
    info = match.get("info", {})
    season = str(info.get("season", "Unknown"))
    date = info.get("dates", [None])[0]

    # Step 2: Process each innings separately.
    for innings_number, innings in enumerate(match.get("innings", []), start=1):
        batting_team = innings.get("team")
        innings_runs = 0
        innings_wickets = 0
        batter_stats = {}

        # Step 3: Read every delivery in every over.
        for over in innings.get("overs", []):
            for delivery in over.get("deliveries", []):
                runs = delivery.get("runs", {})

                # Add total runs to the innings total.
                innings_runs += runs.get("total", 0)

                # Add batter runs to the relevant player's total.
                batter = delivery.get("batter")
                if batter:
                    batter_stats.setdefault(batter, 0)
                    batter_stats[batter] += runs.get("batter", 0)

                # Count each recorded wicket event.
                innings_wickets += len(delivery.get("wickets", []))

        # Step 4: Save the innings-level summary.
        innings_records.append({
            "match_id": match_id,
            "date": date,
            "season": season,
            "innings_number": innings_number,
            "batting_team": batting_team,
            "innings_runs": innings_runs,
            "innings_wickets": innings_wickets
        })

        # Step 5: Save the batter-level run totals for this innings.
        for batter, batter_runs in batter_stats.items():
            batter_records.append({
                "match_id": match_id,
                "date": date,
                "season": season,
                "innings_number": innings_number,
                "batting_team": batting_team,
                "batter": batter,
                "batter_runs": batter_runs
            })

# Step 6: Convert the records into pandas DataFrames.
innings_df = pd.DataFrame(innings_records)
batter_df = pd.DataFrame(batter_records)

# Step 7: Display basic validation checks and sample rows.
print("Innings records:", len(innings_df))
print("Batter-innings records:", len(batter_df))

print("\nSample innings totals:")
display(innings_df.head(10))

print("\nSample batter statistics:")
display(batter_df.head(10))

Innings records: 2514
Batter-innings records: 18842

Sample innings totals:


,match_id,date,season,innings_number,batting_team,innings_runs,innings_wickets
0,419118,2010-03-20,2009/10,1,Rajasthan Royals,168,7
1,419118,2010-03-20,2009/10,2,Kolkata Knight Riders,134,5
2,336012,2008-05-28,2007/08,1,Royal Challengers Bangalore,122,9
3,336012,2008-05-28,2007/08,2,Mumbai Indians,126,1
4,501240,2011-05-04,2011,1,Rajasthan Royals,147,6
5,501240,2011-05-04,2011,2,Chennai Super Kings,149,2
6,1359491,2023-04-12,2023,1,Rajasthan Royals,175,8
7,1359491,2023-04-12,2023,2,Chennai Super Kings,172,6
8,548331,2012-04-20,2012,1,Kings XI Punjab,163,6
9,548331,2012-04-20,2012,2,Royal Challengers Bangalore,166,5



Sample batter statistics:


,match_id,date,season,innings_number,batting_team,batter,batter_runs
0,419118,2010-03-20,2009/10,1,Rajasthan Royals,MJ Lumb,0
1,419118,2010-03-20,2009/10,1,Rajasthan Royals,FY Fazal,31
2,419118,2010-03-20,2009/10,1,Rajasthan Royals,NV Ojha,27
3,419118,2010-03-20,2009/10,1,Rajasthan Royals,AA Jhunjhunwala,45
4,419118,2010-03-20,2009/10,1,Rajasthan Royals,YK Pathan,15
5,419118,2010-03-20,2009/10,1,Rajasthan Royals,AC Voges,37
6,419118,2010-03-20,2009/10,1,Rajasthan Royals,P Dogra,0
7,419118,2010-03-20,2009/10,1,Rajasthan Royals,SK Warne,1
8,419118,2010-03-20,2009/10,1,Rajasthan Royals,MM Patel,4
9,419118,2010-03-20,2009/10,2,Kolkata Knight Riders,MK Tiwary,11


In [8]:
# STEP 7: VALIDATE CALCULATED INNINGS TOTALS AGAINST SOURCE DATA
# Purpose:
# 1. Recalculate each innings' total runs directly from the original JSON.
# 2. Compare the source totals with the values in innings_df.
# 3. Identify any mismatches before using these statistics for QA.
#
# Important:
# We are validating the runs calculation only in this step.
# Wicket counts and batter statistics will need separate validation.

# Step 1: Create a list to store validation results.
validation_records = []

# Step 2: Reopen every original match file.
for filename in json_files:
    filepath = os.path.join(DATA_DIR, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        match = json.load(f)

    match_id = filename.replace(".json", "")

    # Step 3: Recalculate the total runs for every innings.
    for innings_number, innings in enumerate(
        match.get("innings", []), start=1
    ):
        source_total = 0

        for over in innings.get("overs", []):
            for delivery in over.get("deliveries", []):
                source_total += delivery.get("runs", {}).get("total", 0)

        # Step 4: Find the corresponding calculated record.
        calculated_row = innings_df[
            (innings_df["match_id"] == match_id)
            & (innings_df["innings_number"] == innings_number)
        ]

        # Step 5: Record the comparison.
        if len(calculated_row) == 1:
            calculated_total = int(
                calculated_row.iloc[0]["innings_runs"]
            )
            validation_records.append({
                "match_id": match_id,
                "innings_number": innings_number,
                "source_total": source_total,
                "calculated_total": calculated_total,
                "matches": source_total == calculated_total
            })
        else:
            validation_records.append({
                "match_id": match_id,
                "innings_number": innings_number,
                "source_total": source_total,
                "calculated_total": None,
                "matches": False
            })

# Step 6: Convert the results into a DataFrame.
validation_df = pd.DataFrame(validation_records)

# Step 7: Summarise the validation outcome.
print("Innings records checked:", len(validation_df))
print("Matching totals:", int(validation_df["matches"].sum()))
print("Mismatched totals:", int((~validation_df["matches"]).sum()))

# Step 8: Display mismatches, if any.
mismatches_df = validation_df[~validation_df["matches"]]
if len(mismatches_df) > 0:
    print("\nFirst mismatches:")
    display(mismatches_df.head(10))
else:
    print("\nAll innings totals match the original JSON records.")

Innings records checked: 2514
Matching totals: 2514
Mismatched totals: 0

All innings totals match the original JSON records.


In [9]:
# STEP 8: VALIDATE BATTER RUNS AGAINST THE ORIGINAL JSON DATA
# Purpose:
# 1. Recalculate each batter's runs directly from the source deliveries.
# 2. Compare the results with the existing batter_df records.
# 3. Report missing, extra, or incorrect batter-innings records.
#
# Note:
# This validates batter runs only, not balls faced, strike rate, or wickets.

from collections import defaultdict

# Step 1: Create a dictionary to store independently recalculated runs.
source_batter_runs = defaultdict(int)

# Step 2: Read every match file and accumulate each batter's runs.
for filename in json_files:
    filepath = os.path.join(DATA_DIR, filename)
    match_id = filename.replace(".json", "")

    with open(filepath, "r", encoding="utf-8") as f:
        match = json.load(f)

    for innings_number, innings in enumerate(
        match.get("innings", []), start=1
    ):
        for over in innings.get("overs", []):
            for delivery in over.get("deliveries", []):
                batter = delivery.get("batter")
                runs = delivery.get("runs", {}).get("batter", 0)

                if batter:
                    key = (match_id, innings_number, batter)
                    source_batter_runs[key] += runs

# Step 3: Convert our existing batter_df into a comparable dictionary.
calculated_batter_runs = {
    (row["match_id"], int(row["innings_number"]), row["batter"]):
        int(row["batter_runs"])
    for _, row in batter_df.iterrows()
}

# Step 4: Compare the source and calculated records.
all_keys = set(source_batter_runs) | set(calculated_batter_runs)

batter_mismatches = []

for key in all_keys:
    source_runs = source_batter_runs.get(key)
    calculated_runs = calculated_batter_runs.get(key)

    if source_runs != calculated_runs:
        batter_mismatches.append({
            "match_id": key[0],
            "innings_number": key[1],
            "batter": key[2],
            "source_runs": source_runs,
            "calculated_runs": calculated_runs
        })

# Step 5: Report the validation outcome.
print("Unique batter-innings checked:", len(all_keys))
print("Matching records:", len(all_keys) - len(batter_mismatches))
print("Mismatched records:", len(batter_mismatches))

# Step 6: Display any mismatches for investigation.
if batter_mismatches:
    print("\nFirst mismatches:")
    display(pd.DataFrame(batter_mismatches).head(10))
else:
    print("\nAll batter-run totals match the original JSON records.")

Unique batter-innings checked: 18842
Matching records: 18842
Mismatched records: 0

All batter-run totals match the original JSON records.


In [10]:
# STEP 9: VALIDATE INNINGS WICKET COUNTS
# Purpose:
# 1. Recount wicket events directly from the original JSON deliveries.
# 2. Compare source counts with the values in innings_df.
# 3. Identify any mismatches before using wicket statistics in questions.
#
# Important:
# This checks recorded wicket events. It does not yet calculate
# bowler-credit wickets, which require dismissal-type rules.

# Step 1: Create a list to store the validation results.
wicket_validation_records = []

# Step 2: Read each original match file.
for filename in json_files:
    filepath = os.path.join(DATA_DIR, filename)
    match_id = filename.replace(".json", "")

    with open(filepath, "r", encoding="utf-8") as f:
        match = json.load(f)

    # Step 3: Recount wicket events for every innings.
    for innings_number, innings in enumerate(
        match.get("innings", []), start=1
    ):
        source_wickets = 0

        for over in innings.get("overs", []):
            for delivery in over.get("deliveries", []):
                source_wickets += len(delivery.get("wickets", []))

        # Step 4: Retrieve the corresponding innings_df record.
        calculated_row = innings_df[
            (innings_df["match_id"] == match_id)
            & (innings_df["innings_number"] == innings_number)
        ]

        # Step 5: Compare the source and calculated counts.
        if len(calculated_row) == 1:
            calculated_wickets = int(
                calculated_row.iloc[0]["innings_wickets"]
            )
            wicket_validation_records.append({
                "match_id": match_id,
                "innings_number": innings_number,
                "source_wickets": source_wickets,
                "calculated_wickets": calculated_wickets,
                "matches": source_wickets == calculated_wickets
            })
        else:
            wicket_validation_records.append({
                "match_id": match_id,
                "innings_number": innings_number,
                "source_wickets": source_wickets,
                "calculated_wickets": None,
                "matches": False
            })

# Step 6: Convert the results to a DataFrame.
wicket_validation_df = pd.DataFrame(wicket_validation_records)

# Step 7: Print the validation summary.
print("Innings records checked:", len(wicket_validation_df))
print("Matching wicket counts:",
      int(wicket_validation_df["matches"].sum()))
print("Mismatched wicket counts:",
      int((~wicket_validation_df["matches"]).sum()))

# Step 8: Display any mismatches for investigation.
wicket_mismatches_df = wicket_validation_df[
    ~wicket_validation_df["matches"]
]

if len(wicket_mismatches_df) > 0:
    print("\nFirst mismatches:")
    display(wicket_mismatches_df.head(10))
else:
    print("\nAll wicket-event counts match the original JSON records.")

Innings records checked: 2514
Matching wicket counts: 2514
Mismatched wicket counts: 0

All wicket-event counts match the original JSON records.


In [11]:
# STEP 10: CREATE SEARCHABLE MATCH-LEVEL TEXT PASSAGES
# Purpose:
# 1. Combine match metadata and validated innings totals.
# 2. Create one readable text passage per match.
# 3. Include batter-run records for future player-stat questions.
# 4. Preserve match_id so passages can be split without data leakage.

# Step 1: Create a list to store the searchable passages.
match_passages = []

# Step 2: Process each match record.
for _, match_row in matches_df.iterrows():
    match_id = str(match_row["match_id"])

    # Step 3: Retrieve the innings summaries for this match.
    match_innings = innings_df[
        innings_df["match_id"] == match_id
    ].sort_values("innings_number")

    # Step 4: Retrieve the batter statistics for this match.
    match_batters = batter_df[
        batter_df["match_id"] == match_id
    ]

    # Step 5: Build a readable summary of the innings.
    innings_text = []
    for _, innings_row in match_innings.iterrows():
        innings_text.append(
            f"{innings_row['batting_team']} scored "
            f"{innings_row['innings_runs']} runs and lost "
            f"{innings_row['innings_wickets']} wickets "
            f"in innings {innings_row['innings_number']}."
        )

    # Step 6: Build a player-runs summary for each innings.
    batter_text = []
    for _, batter_row in match_batters.iterrows():
        batter_text.append(
            f"{batter_row['batter']} scored "
            f"{batter_row['batter_runs']} runs for "
            f"{batter_row['batting_team']}."
        )

    # Step 7: Format the match metadata.
    teams_text = ", ".join(match_row["teams"])
    player_of_match_text = ", ".join(match_row["player_of_match"])

    summary = (
        f"IPL match ID: {match_id}. "
        f"Date: {match_row['date']}. "
        f"Season: {match_row['season']}. "
        f"Teams: {teams_text}. "
        f"Venue: {match_row['venue']}. "
        f"City: {match_row['city']}. "
        f"Winner: {match_row['winner']}. "
        f"Toss winner: {match_row['toss_winner']}. "
        f"Toss decision: {match_row['toss_decision']}. "
        f"Player of the match: {player_of_match_text}. "
        + " ".join(innings_text)
        + " Player batting runs: "
        + " ".join(batter_text)
    )

    # Step 8: Store the passage alongside its match ID.
    match_passages.append({
        "match_id": match_id,
        "date": match_row["date"],
        "season": match_row["season"],
        "text": summary
    })

# Step 9: Convert passages into a DataFrame.
passages_df = pd.DataFrame(match_passages)

# Step 10: Verify passage count and inspect one example.
print("Searchable passages created:", len(passages_df))
print("Unique match IDs:", passages_df["match_id"].nunique())

print("\nExample passage:")
print(passages_df.iloc[0]["text"][:2500])

Searchable passages created: 1243
Unique match IDs: 1243

Example passage:
IPL match ID: 419118. Date: 2010-03-20. Season: 2009/10. Teams: Rajasthan Royals, Kolkata Knight Riders. Venue: Sardar Patel Stadium, Motera. City: Ahmedabad. Winner: Rajasthan Royals. Toss winner: Rajasthan Royals. Toss decision: bat. Player of the match: AA Jhunjhunwala. Rajasthan Royals scored 168 runs and lost 7 wickets in innings 1. Kolkata Knight Riders scored 134 runs and lost 5 wickets in innings 2. Player batting runs: MJ Lumb scored 0 runs for Rajasthan Royals. FY Fazal scored 31 runs for Rajasthan Royals. NV Ojha scored 27 runs for Rajasthan Royals. AA Jhunjhunwala scored 45 runs for Rajasthan Royals. YK Pathan scored 15 runs for Rajasthan Royals. AC Voges scored 37 runs for Rajasthan Royals. P Dogra scored 0 runs for Rajasthan Royals. SK Warne scored 1 runs for Rajasthan Royals. MM Patel scored 4 runs for Rajasthan Royals. MK Tiwary scored 11 runs for Kolkata Knight Riders. BJ Hodge scored 36 runs fo

In [12]:
# STEP 11: MEASURE SEARCHABLE PASSAGE LENGTHS
# Purpose:
# 1. Measure the length of every match passage.
# 2. Summarise typical and unusually long passages.
# 3. Decide whether we need to split passages into smaller units.

# Step 1: Count words and characters in each passage.
passages_df["word_count"] = passages_df["text"].apply(
    lambda text: len(text.split())
)
passages_df["character_count"] = passages_df["text"].apply(len)

# Step 2: Print summary statistics for passage word counts.
print("Number of passages:", len(passages_df))
print("\nWord-count statistics:")
print(passages_df["word_count"].describe().round(2))

# Step 3: Print useful percentiles to understand passage sizes.
print("\nPassage word-count percentiles:")
print(
    passages_df["word_count"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
)

# Step 4: Display the five longest passages.
print("\nFive longest passages:")
display(
    passages_df[
        ["match_id", "season", "word_count", "character_count"]
    ]
    .sort_values("word_count", ascending=False)
    .head(5)
)

Number of passages: 1243

Word-count statistics:
count    1243.00
mean      194.81
std        26.75
min        68.00
25%       177.00
50%       194.00
75%       212.00
max       320.00
Name: word_count, dtype: float64

Passage word-count percentiles:
0.50    194.00
0.75    212.00
0.90    228.00
0.95    237.00
0.99    259.58
Name: word_count, dtype: float64

Five longest passages:


,match_id,season,word_count,character_count
522,1216517,2020/21,320,1732
1007,1529281,2026,306,1778
512,419121,2009/10,300,1642
391,1082625,2017,294,1680
575,598004,2013,282,1793


In [13]:
# STEP 12: CREATE SEPARATE MATCH AND PLAYER-STAT PASSAGES
# Purpose:
# 1. Keep match-level facts separate from individual batter statistics.
# 2. Create smaller, more targeted searchable passages.
# 3. Preserve match IDs for leakage-safe evaluation later.

# Step 1: Build one concise match-summary passage per match.
match_summary_passages = []

for _, match_row in matches_df.iterrows():
    match_id = str(match_row["match_id"])

    # Step 2: Retrieve the innings totals for this match.
    match_innings = innings_df[
        innings_df["match_id"] == match_id
    ].sort_values("innings_number")

    innings_summary = " ".join(
        f"{row['batting_team']} scored {row['innings_runs']} runs "
        f"and lost {row['innings_wickets']} wickets in innings "
        f"{row['innings_number']}."
        for _, row in match_innings.iterrows()
    )

    # Step 3: Combine match metadata with innings summaries.
    summary_text = (
        f"IPL match ID: {match_id}. "
        f"Date: {match_row['date']}. "
        f"Season: {match_row['season']}. "
        f"Teams: {', '.join(match_row['teams'])}. "
        f"Venue: {match_row['venue']}. "
        f"City: {match_row['city']}. "
        f"Winner: {match_row['winner']}. "
        f"Toss winner: {match_row['toss_winner']}. "
        f"Toss decision: {match_row['toss_decision']}. "
        f"Player of the match: {', '.join(match_row['player_of_match'])}. "
        f"{innings_summary}"
    )

    match_summary_passages.append({
        "passage_id": f"{match_id}_summary",
        "match_id": match_id,
        "passage_type": "match_summary",
        "text": summary_text
    })

# Step 4: Build one player-stat passage per batter per innings.
player_stat_passages = []

for _, row in batter_df.iterrows():
    match_id = str(row["match_id"])

    player_text = (
        f"IPL match ID: {match_id}. "
        f"Date: {row['date']}. "
        f"Season: {row['season']}. "
        f"Innings: {row['innings_number']}. "
        f"Batting team: {row['batting_team']}. "
        f"Player: {row['batter']}. "
        f"Runs scored by {row['batter']}: {row['batter_runs']}."
    )

    player_stat_passages.append({
        "passage_id": (
            f"{match_id}_innings{row['innings_number']}_"
            f"{row['batter']}"
        ),
        "match_id": match_id,
        "innings_number": int(row["innings_number"]),
        "batter": str(row["batter"]),
        "passage_type": "player_stat",
        "text": player_text
    })

# Step 5: Convert both passage types into DataFrames.
match_summary_df = pd.DataFrame(match_summary_passages)
player_stat_df = pd.DataFrame(player_stat_passages)

# Step 6: Check counts and inspect examples.
print("Match-summary passages:", len(match_summary_df))
print("Player-stat passages:", len(player_stat_df))

print("\nExample match-summary passage:")
print(match_summary_df.iloc[0]["text"])

print("\nExample player-stat passage:")
print(player_stat_df.iloc[0]["text"])

Match-summary passages: 1243
Player-stat passages: 18842

Example match-summary passage:
IPL match ID: 419118. Date: 2010-03-20. Season: 2009/10. Teams: Rajasthan Royals, Kolkata Knight Riders. Venue: Sardar Patel Stadium, Motera. City: Ahmedabad. Winner: Rajasthan Royals. Toss winner: Rajasthan Royals. Toss decision: bat. Player of the match: AA Jhunjhunwala. Rajasthan Royals scored 168 runs and lost 7 wickets in innings 1. Kolkata Knight Riders scored 134 runs and lost 5 wickets in innings 2.

Example player-stat passage:
IPL match ID: 419118. Date: 2010-03-20. Season: 2009/10. Innings: 1. Batting team: Rajasthan Royals. Player: MJ Lumb. Runs scored by MJ Lumb: 0.


In [14]:
# STEP 13: CREATE A REPRODUCIBLE MATCH-LEVEL DATA SPLIT
# Purpose:
# 1. Split unique IPL matches into train, validation, and test sets.
# 2. Keep all passages from the same match in the same split.
# 3. Use a fixed random seed so the split is reproducible.
# 4. Check that no match IDs overlap between splits.

# Step 1: Import the tools needed for a reproducible split.
from sklearn.model_selection import train_test_split

# Step 2: Get the unique match IDs and sort them for stable input ordering.
all_match_ids = sorted(matches_df["match_id"].astype(str).unique())

# Step 3: Reserve 20% of matches for the final test set.
train_val_ids, test_ids = train_test_split(
    all_match_ids,
    test_size=0.20,
    random_state=SEED
)

# Step 4: Split the remaining matches into training and validation sets.
# This gives approximately 60% train, 20% validation, and 20% test overall.
train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.25,
    random_state=SEED
)

# Step 5: Convert the split lists to sets for overlap checks.
train_ids = set(train_ids)
val_ids = set(val_ids)
test_ids = set(test_ids)

# Step 6: Verify that the three splits do not share any match IDs.
print("Training matches:", len(train_ids))
print("Validation matches:", len(val_ids))
print("Test matches:", len(test_ids))

print("\nTrain/validation overlap:", len(train_ids & val_ids))
print("Train/test overlap:", len(train_ids & test_ids))
print("Validation/test overlap:", len(val_ids & test_ids))

# Step 7: Confirm that every match was assigned to exactly one split.
all_split_ids = train_ids | val_ids | test_ids

print("\nTotal unique matches:", len(all_match_ids))
print("Matches assigned to a split:", len(all_split_ids))
print("Unassigned matches:", len(set(all_match_ids) - all_split_ids))

# Step 8: Save the split assignment for reproducibility.
split_df = pd.DataFrame([
    {"match_id": match_id, "split": split_name}
    for split_name, match_set in [
        ("train", train_ids),
        ("validation", val_ids),
        ("test", test_ids)
    ]
    for match_id in sorted(match_set)
])

split_df.to_csv("/content/ipl_match_split.csv", index=False)

print("\nSplit assignment saved to /content/ipl_match_split.csv")

Training matches: 745
Validation matches: 249
Test matches: 249

Train/validation overlap: 0
Train/test overlap: 0
Validation/test overlap: 0

Total unique matches: 1243
Matches assigned to a split: 1243
Unassigned matches: 0

Split assignment saved to /content/ipl_match_split.csv


In [15]:
# STEP 14: LABEL ALL PASSAGES WITH THEIR MATCH-LEVEL SPLIT
# Purpose:
# 1. Attach train, validation, or test labels to every match.
# 2. Apply the same split to both passage collections.
# 3. Verify that no match appears in multiple splits.
# 4. Check that all passages received a split label.

# Step 1: Create a lookup table from match ID to split name.
split_lookup = split_df.set_index("match_id")["split"].to_dict()

# Step 2: Add split labels to match-summary passages.
match_summary_df["split"] = (
    match_summary_df["match_id"].astype(str).map(split_lookup)
)

# Step 3: Add split labels to player-stat passages.
player_stat_df["split"] = (
    player_stat_df["match_id"].astype(str).map(split_lookup)
)

# Step 4: Report how many passages belong to each split.
print("Match-summary passages by split:")
print(match_summary_df["split"].value_counts(dropna=False))

print("\nPlayer-stat passages by split:")
print(player_stat_df["split"].value_counts(dropna=False))

# Step 5: Check whether any passages are missing a split assignment.
print("\nUnassigned match-summary passages:",
      int(match_summary_df["split"].isna().sum()))

print("Unassigned player-stat passages:",
      int(player_stat_df["split"].isna().sum()))

# Step 6: Verify that each match ID has exactly one split.
combined_passages_df = pd.concat(
    [match_summary_df[["match_id", "split"]],
     player_stat_df[["match_id", "split"]]],
    ignore_index=True
)

splits_per_match = combined_passages_df.groupby(
    "match_id"
)["split"].nunique()

print("\nMatches appearing in multiple splits:",
      int((splits_per_match > 1).sum()))

Match-summary passages by split:
split
train         745
validation    249
test          249
Name: count, dtype: int64

Player-stat passages by split:
split
train         11280
test           3797
validation     3765
Name: count, dtype: int64

Unassigned match-summary passages: 0
Unassigned player-stat passages: 0

Matches appearing in multiple splits: 0


In [16]:
# STEP 15: GENERATE DETERMINISTIC IPL QUESTIONS AND GOLD ANSWERS
# Purpose:
# 1. Generate factual questions from structured source records.
# 2. Create gold answers without using a language model.
# 3. Store the source match ID and split for leakage-safe evaluation.
# 4. Keep match facts, innings statistics, and player statistics identifiable.

# Step 1: Create a list to hold all generated questions.
qa_records = []

# Step 2: Generate match-winner questions from match metadata.
for _, row in matches_df.iterrows():
    match_id = str(row["match_id"])
    teams = row["teams"]

    # Only create a winner question when a winner is recorded.
    if pd.notna(row["winner"]) and row["winner"]:
        qa_records.append({
            "question_id": f"{match_id}_winner",
            "match_id": match_id,
            "split": split_lookup[match_id],
            "question_type": "match_winner",
            "question": (
                f"Which team won the IPL match between "
                f"{teams[0]} and {teams[1]} played on {row['date']}?"
            ),
            "gold_answer": str(row["winner"])
        })

# Step 3: Generate innings-total questions from validated innings records.
for _, row in innings_df.iterrows():
    match_id = str(row["match_id"])

    qa_records.append({
        "question_id": (
            f"{match_id}_innings{row['innings_number']}_runs"
        ),
        "match_id": match_id,
        "split": split_lookup[match_id],
        "question_type": "innings_runs",
        "question": (
            f"How many runs did {row['batting_team']} score "
            f"in innings {row['innings_number']} of the IPL match "
            f"played on {row['date']}?"
        ),
        "gold_answer": str(int(row["innings_runs"]))
    })

# Step 4: Generate batter-run questions from validated batter records.
for _, row in batter_df.iterrows():
    match_id = str(row["match_id"])

    qa_records.append({
        "question_id": (
            f"{match_id}_innings{row['innings_number']}_"
            f"{row['batter']}_runs"
        ),
        "match_id": match_id,
        "split": split_lookup[match_id],
        "question_type": "batter_runs",
        "question": (
            f"How many runs did {row['batter']} score for "
            f"{row['batting_team']} in innings {row['innings_number']} "
            f"of the IPL match played on {row['date']}?"
        ),
        "gold_answer": str(int(row["batter_runs"]))
    })

# Step 5: Convert the generated questions into a DataFrame.
qa_df = pd.DataFrame(qa_records)

# Step 6: Report dataset size and question-type counts.
print("Total questions generated:", len(qa_df))

print("\nQuestions by type:")
print(qa_df["question_type"].value_counts())

print("\nQuestions by split:")
print(qa_df["split"].value_counts())

# Step 7: Check for duplicate question IDs.
print("\nDuplicate question IDs:", int(qa_df["question_id"].duplicated().sum()))

# Step 8: Preview a few questions and their gold answers.
display(
    qa_df[
        ["question_id", "split", "question_type", "question", "gold_answer"]
    ].head(10)
)

Total questions generated: 22574

Questions by type:
question_type
batter_runs     18842
innings_runs     2514
match_winner     1218
Name: count, dtype: int64

Questions by split:
split
train         13518
test           4545
validation     4511
Name: count, dtype: int64

Duplicate question IDs: 0


,question_id,split,question_type,question,gold_answer
0,419118_winner,train,match_winner,Which team won the IPL match between Rajasthan...,Rajasthan Royals
1,336012_winner,train,match_winner,Which team won the IPL match between Royal Cha...,Mumbai Indians
2,501240_winner,train,match_winner,Which team won the IPL match between Chennai S...,Chennai Super Kings
3,1359491_winner,validation,match_winner,Which team won the IPL match between Rajasthan...,Rajasthan Royals
4,548331_winner,test,match_winner,Which team won the IPL match between Kings XI ...,Royal Challengers Bangalore
5,419108_winner,train,match_winner,Which team won the IPL match between Kings XI ...,Delhi Daredevils
6,598063_winner,train,match_winner,Which team won the IPL match between Mumbai In...,Mumbai Indians
7,1136564_winner,train,match_winner,Which team won the IPL match between Rajasthan...,Sunrisers Hyderabad
8,501230_winner,test,match_winner,Which team won the IPL match between Delhi Dar...,Kolkata Knight Riders
9,1082637_winner,train,match_winner,Which team won the IPL match between Kings XI ...,Gujarat Lions


In [17]:
# STEP 16: VALIDATE QUESTION-TO-PASSAGE COVERAGE
# Purpose:
# 1. Check that every question has a source passage for its match.
# 2. Check that the source passage belongs to the same split as the question.
# 3. Check that the gold answer appears in the expected evidence passage.
# 4. Identify questions that need fixing before retrieval evaluation.

# Step 1: Create lookup dictionaries for match-summary and player-stat passages.
summary_lookup = match_summary_df.set_index("match_id")["text"].to_dict()
player_lookup = player_stat_df.set_index("passage_id")["text"].to_dict()

# Step 2: Create a lookup for individual player-stat passages.
player_passage_lookup = {
    (str(row["match_id"]), str(row["passage_id"])): row["text"]
    for _, row in player_stat_df.iterrows()
}

# Step 3: Check each question against its expected evidence.
coverage_results = []

for _, question_row in qa_df.iterrows():
    match_id = str(question_row["match_id"])
    question_type = question_row["question_type"]
    gold_answer = str(question_row["gold_answer"])

    if question_type in ["match_winner", "innings_runs"]:
        evidence_text = summary_lookup.get(match_id, "")
    else:
        # Find the exact player-stat passage for this question.
        matching_player_rows = player_stat_df[
            (player_stat_df["match_id"].astype(str) == match_id)
            & (
                player_stat_df["text"].str.contains(
                    gold_answer, regex=False, na=False
                )
            )
        ]
        evidence_text = (
            matching_player_rows.iloc[0]["text"]
            if len(matching_player_rows) > 0 else ""
        )

    # Step 4: Check passage existence and gold-answer presence.
    passage_exists = bool(evidence_text)
    answer_present = (
        gold_answer.lower() in evidence_text.lower()
        if passage_exists else False
    )

    coverage_results.append({
        "question_id": question_row["question_id"],
        "question_type": question_type,
        "split": question_row["split"],
        "passage_exists": passage_exists,
        "gold_answer_present": answer_present
    })

# Step 5: Summarise the coverage checks.
coverage_df = pd.DataFrame(coverage_results)

print("Questions checked:", len(coverage_df))
print("\nPassage existence:")
print(coverage_df["passage_exists"].value_counts(dropna=False))

print("\nGold-answer presence:")
print(coverage_df["gold_answer_present"].value_counts(dropna=False))

print("\nCoverage by question type:")
display(
    coverage_df.groupby("question_type")[
        ["passage_exists", "gold_answer_present"]
    ].sum()
)

# Step 6: Show a few questions that failed either check.
coverage_failures = coverage_df[
    (~coverage_df["passage_exists"])
    | (~coverage_df["gold_answer_present"])
]

print("\nQuestions needing investigation:", len(coverage_failures))
display(coverage_failures.head(10))

Questions checked: 22574

Passage existence:
passage_exists
True    22574
Name: count, dtype: int64

Gold-answer presence:
gold_answer_present
True    22574
Name: count, dtype: int64

Coverage by question type:


,passage_exists,gold_answer_present
question_type,,
batter_runs,18842,18842
innings_runs,2514,2514
match_winner,1218,1218



Questions needing investigation: 0


,question_id,question_type,split,passage_exists,gold_answer_present


In [18]:
# STEP 17: INSTALL AND PREPARE BM25 RETRIEVAL
# Purpose:
# 1. Install rank_bm25, a lightweight keyword-retrieval library.
# 2. Import the tokenizer and BM25 implementation.
# 3. Confirm that the retrieval library is ready.
#
# Research note:
# We will later compare fixed top-1, fixed top-3, and adaptive retrieval
# using the same held-out test questions.

# Step 1: Install the BM25 library in the current Colab runtime.
%pip -q install rank_bm25

# Step 2: Import the library and supporting tools.
import re
from rank_bm25 import BM25Okapi

# Step 3: Define a simple, reproducible lowercase word tokenizer.
def tokenize_text(text):
    return re.findall(r"\b[a-z0-9]+\b", str(text).lower())

# Step 4: Confirm that the library and tokenizer work.
example_tokens = tokenize_text(
    "Rajasthan Royals scored 168 runs in the IPL."
)

print("BM25 library imported successfully.")
print("Tokenizer test:", example_tokens)
print("Match-summary passages available:", len(match_summary_df))
print("Player-stat passages available:", len(player_stat_df))

BM25 library imported successfully.
Tokenizer test: ['rajasthan', 'royals', 'scored', '168', 'runs', 'in', 'the', 'ipl']
Match-summary passages available: 1243
Player-stat passages available: 18842


In [19]:
# STEP 18: BUILD SPLIT-SPECIFIC BM25 RETRIEVAL INDEXES
# Purpose:
# 1. Build separate indexes for match facts and player statistics.
# 2. Build each index independently for train, validation, and test.
# 3. Keep the evidence passages and their metadata aligned.
# 4. Verify the number of indexed passages in each collection.

# Step 1: Create dictionaries to store the indexes and their passages.
bm25_indexes = {}
bm25_passage_stores = {}

# Step 2: Loop through each data split.
for split_name in ["train", "validation", "test"]:

    # Step 3: Select match-summary passages for this split.
    summary_subset = match_summary_df[
        match_summary_df["split"] == split_name
    ].reset_index(drop=True)

    # Step 4: Tokenize the match-summary passages.
    summary_tokens = [
        tokenize_text(text) for text in summary_subset["text"]
    ]

    # Step 5: Build the match-summary BM25 index.
    bm25_indexes[(split_name, "match_summary")] = BM25Okapi(
        summary_tokens
    )
    bm25_passage_stores[(split_name, "match_summary")] = (
        summary_subset
    )

    # Step 6: Select player-stat passages for this split.
    player_subset = player_stat_df[
        player_stat_df["split"] == split_name
    ].reset_index(drop=True)

    # Step 7: Tokenize the player-stat passages.
    player_tokens = [
        tokenize_text(text) for text in player_subset["text"]
    ]

    # Step 8: Build the player-stat BM25 index.
    bm25_indexes[(split_name, "player_stat")] = BM25Okapi(
        player_tokens
    )
    bm25_passage_stores[(split_name, "player_stat")] = (
        player_subset
    )

# Step 9: Print index sizes to verify successful construction.
print("BM25 indexes built:", len(bm25_indexes))

for key, passage_store in bm25_passage_stores.items():
    print(
        f"Split: {key[0]:10s} | "
        f"Collection: {key[1]:13s} | "
        f"Passages: {len(passage_store)}"
    )

# Step 10: Confirm that all expected indexes exist.
expected_indexes = {
    (split_name, collection_name)
    for split_name in ["train", "validation", "test"]
    for collection_name in ["match_summary", "player_stat"]
}

print("\nAll six indexes created:",
      set(bm25_indexes.keys()) == expected_indexes)

BM25 indexes built: 6
Split: train      | Collection: match_summary | Passages: 745
Split: train      | Collection: player_stat   | Passages: 11280
Split: validation | Collection: match_summary | Passages: 249
Split: validation | Collection: player_stat   | Passages: 3765
Split: test       | Collection: match_summary | Passages: 249
Split: test       | Collection: player_stat   | Passages: 3797

All six indexes created: True


In [20]:
# STEP 19: TEST BM25 RETRIEVAL ON THREE HELD-OUT QUESTIONS
# Purpose:
# 1. Select one test question from each question category.
# 2. Retrieve the top three passages from the appropriate test index.
# 3. Display the gold answer and retrieved evidence for inspection.
# 4. Confirm that retrieval works before running a full evaluation.

# Step 1: Define which passage collection each question type should search.
collection_for_question = {
    "match_winner": "match_summary",
    "innings_runs": "match_summary",
    "batter_runs": "player_stat"
}

# Step 2: Select one test question from each available category.
sample_questions = (
    qa_df[qa_df["split"] == "test"]
    .groupby("question_type", group_keys=False)
    .head(1)
)

print("Sample questions selected:", len(sample_questions))

# Step 3: Retrieve and display the top three passages per question.
for _, question_row in sample_questions.iterrows():
    split_name = question_row["split"]
    question_type = question_row["question_type"]
    collection_name = collection_for_question[question_type]

    # Step 4: Get the appropriate index and aligned passage store.
    index = bm25_indexes[(split_name, collection_name)]
    passage_store = bm25_passage_stores[
        (split_name, collection_name)
    ]

    # Step 5: Tokenize the question and score all passages.
    query_tokens = tokenize_text(question_row["question"])
    scores = index.get_scores(query_tokens)

    # Step 6: Select the three highest-scoring passages.
    top_indices = scores.argsort()[::-1][:3]

    print("\n" + "=" * 80)
    print("Question type:", question_type)
    print("Question:", question_row["question"])
    print("Gold answer:", question_row["gold_answer"])

    # Step 7: Print each retrieved passage and its score.
    for rank, passage_index in enumerate(top_indices, start=1):
        retrieved_row = passage_store.iloc[int(passage_index)]

        print(f"\nRank {rank} | BM25 score: {scores[passage_index]:.3f}")
        print("Match ID:", retrieved_row["match_id"])
        print("Passage:", retrieved_row["text"][:1000])

Sample questions selected: 3

Question type: match_winner
Question: Which team won the IPL match between Kings XI Punjab and Royal Challengers Bangalore played on 2012-04-20?
Gold answer: Royal Challengers Bangalore

Rank 1 | BM25 score: 22.752
Match ID: 548331
Passage: IPL match ID: 548331. Date: 2012-04-20. Season: 2012. Teams: Kings XI Punjab, Royal Challengers Bangalore. Venue: Punjab Cricket Association Stadium, Mohali. City: Chandigarh. Winner: Royal Challengers Bangalore. Toss winner: Royal Challengers Bangalore. Toss decision: field. Player of the match: CH Gayle. Kings XI Punjab scored 163 runs and lost 6 wickets in innings 1. Royal Challengers Bangalore scored 166 runs and lost 5 wickets in innings 2.

Rank 2 | BM25 score: 16.429
Match ID: 419112
Passage: IPL match ID: 419112. Date: 2010-03-16. Season: 2009/10. Teams: Royal Challengers Bangalore, Kings XI Punjab. Venue: M Chinnaswamy Stadium. City: Bangalore. Winner: Royal Challengers Bangalore. Toss winner: Kings XI Punjab. 

In [21]:
# STEP 20: EVALUATE BM25 EVIDENCE RETRIEVAL ON THE HELD-OUT TEST SET
# Purpose:
# 1. Evaluate every question assigned to the test split.
# 2. Check whether the correct match's evidence is retrieved at Rank 1 or Top 3.
# 3. Report results separately for each question type.
# 4. Save per-question retrieval results for reproducible analysis.
#
# Important:
# This measures whether the correct match is retrieved.
# It does not yet measure whether a generated answer is correct or supported.

# Step 1: Select the held-out test questions.
test_questions_df = qa_df[
    qa_df["split"] == "test"
].reset_index(drop=True)

# Step 2: Map each question type to its appropriate passage collection.
collection_for_question = {
    "match_winner": "match_summary",
    "innings_runs": "match_summary",
    "batter_runs": "player_stat"
}

# Step 3: Prepare a list to store the retrieval results.
retrieval_results = []

# Step 4: Evaluate each test question using its split-specific BM25 index.
for _, question_row in test_questions_df.iterrows():
    question_type = question_row["question_type"]
    match_id = str(question_row["match_id"])
    collection_name = collection_for_question[question_type]

    # Retrieve the correct index and its aligned passage store.
    index = bm25_indexes[("test", collection_name)]
    passage_store = bm25_passage_stores[
        ("test", collection_name)
    ]

    # Tokenize the question and score the indexed passages.
    query_tokens = tokenize_text(question_row["question"])
    scores = index.get_scores(query_tokens)

    # Get the highest-scoring passages, ordered by score.
    top_indices = scores.argsort()[::-1][:3]

    # Record the retrieved match IDs.
    retrieved_match_ids = [
        str(passage_store.iloc[int(i)]["match_id"])
        for i in top_indices
    ]

    # Check whether the correct source match was retrieved.
    recall_at_1 = (
        len(retrieved_match_ids) >= 1
        and retrieved_match_ids[0] == match_id
    )
    recall_at_3 = match_id in retrieved_match_ids

    retrieval_results.append({
        "question_id": question_row["question_id"],
        "question_type": question_type,
        "match_id": match_id,
        "gold_answer": question_row["gold_answer"],
        "retrieved_match_ids": retrieved_match_ids,
        "recall_at_1": recall_at_1,
        "recall_at_3": recall_at_3
    })

# Step 5: Convert the retrieval results to a DataFrame.
retrieval_results_df = pd.DataFrame(retrieval_results)

# Step 6: Report overall evidence-retrieval performance.
print("Test questions evaluated:", len(retrieval_results_df))
print(
    "Overall Evidence Recall@1:",
    round(retrieval_results_df["recall_at_1"].mean(), 4)
)
print(
    "Overall Evidence Recall@3:",
    round(retrieval_results_df["recall_at_3"].mean(), 4)
)

# Step 7: Report retrieval performance by question type.
print("\nResults by question type:")
type_summary = retrieval_results_df.groupby("question_type").agg(
    questions=("question_id", "count"),
    evidence_recall_at_1=("recall_at_1", "mean"),
    evidence_recall_at_3=("recall_at_3", "mean")
).round(4)

display(type_summary)

# Step 8: Save the per-question results for later analysis.
retrieval_results_df.to_csv(
    "/content/ipl_bm25_test_retrieval_results.csv",
    index=False
)

print(
    "\nPer-question results saved to "
    "/content/ipl_bm25_test_retrieval_results.csv"
)

Test questions evaluated: 4545
Overall Evidence Recall@1: 0.9894
Overall Evidence Recall@3: 0.9991

Results by question type:


,questions,evidence_recall_at_1,evidence_recall_at_3
question_type,,,
batter_runs,3797,0.9932,1.0000
innings_runs,505,0.9564,0.9921
match_winner,243,1.0000,1.0000



Per-question results saved to /content/ipl_bm25_test_retrieval_results.csv


In [22]:
# STEP 21: EVALUATE EXACT EVIDENCE-PASSAGE RETRIEVAL
# Purpose:
# 1. Evaluate whether the exact evidence passage is retrieved.
# 2. Use match summaries for match-winner and innings-run questions.
# 3. Use the exact batter-and-innings passage for batter-run questions.
# 4. Measure top-1 and top-3 exact passage recall.
# 5. Check whether the top-ranked passage contains the gold answer.

# Step 1: Prepare a list for exact-passage evaluation results.
exact_retrieval_results = []

# Step 2: Evaluate each held-out test question.
for _, question_row in test_questions_df.iterrows():
    question_type = question_row["question_type"]
    match_id = str(question_row["match_id"])
    collection_name = collection_for_question[question_type]

    # Step 3: Get the appropriate BM25 index and passage store.
    index = bm25_indexes[("test", collection_name)]
    passage_store = bm25_passage_stores[
        ("test", collection_name)
    ]

    # Step 4: Tokenize the question and rank the passages.
    query_tokens = tokenize_text(question_row["question"])
    scores = index.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:3]

    # Step 5: Identify the exact passage required for the question.
    if question_type in ["match_winner", "innings_runs"]:
        target_passage_id = f"{match_id}_summary"

    else:
        # Find the batter and innings associated with this question.
        batter_row = batter_df[
            (batter_df["match_id"].astype(str) == match_id)
            & (
                batter_df["innings_number"]
                == int(
                    question_row["question_id"].split("_innings")[1]
                    .split("_")[0]
                )
            )
        ]

        # Extract the batter name from the unique question ID.
        question_suffix = question_row["question_id"].split("_runs")[0]
        batter_name = question_suffix.split("_innings")[-1]

        # Match the exact batter passage using match, innings, and player.
        exact_batter_row = batter_row[
            batter_row["batter"] == batter_name
        ]

        if len(exact_batter_row) == 1:
            target_passage_id = (
                f"{match_id}_innings"
                f"{int(exact_batter_row.iloc[0]['innings_number'])}_"
                f"{exact_batter_row.iloc[0]['batter']}"
            )
        else:
            target_passage_id = None

    # Step 6: Retrieve the IDs and text of the top-ranked passages.
    top_rows = passage_store.iloc[top_indices]
    retrieved_passage_ids = top_rows["passage_id"].astype(str).tolist()
    top1_text = str(top_rows.iloc[0]["text"])

    # Step 7: Calculate exact passage recall and answer presence.
    exact_recall_at_1 = (
        target_passage_id is not None
        and retrieved_passage_ids[0] == target_passage_id
    )
    exact_recall_at_3 = (
        target_passage_id is not None
        and target_passage_id in retrieved_passage_ids
    )
    gold_answer_present_top1 = (
        str(question_row["gold_answer"]).lower() in top1_text.lower()
    )

    # Step 8: Save the result for later analysis.
    exact_retrieval_results.append({
        "question_id": question_row["question_id"],
        "question_type": question_type,
        "target_passage_id": target_passage_id,
        "retrieved_passage_ids": retrieved_passage_ids,
        "exact_recall_at_1": exact_recall_at_1,
        "exact_recall_at_3": exact_recall_at_3,
        "gold_answer_present_top1": gold_answer_present_top1
    })

# Step 9: Convert results into a DataFrame.
exact_retrieval_df = pd.DataFrame(exact_retrieval_results)

# Step 10: Print overall metrics.
print("Test questions evaluated:", len(exact_retrieval_df))
print(
    "Exact Passage Recall@1:",
    round(exact_retrieval_df["exact_recall_at_1"].mean(), 4)
)
print(
    "Exact Passage Recall@3:",
    round(exact_retrieval_df["exact_recall_at_3"].mean(), 4)
)
print(
    "Gold answer present in top-1 passage:",
    round(exact_retrieval_df["gold_answer_present_top1"].mean(), 4)
)

# Step 11: Print metrics separately for each question type.
print("\nExact retrieval results by question type:")
display(
    exact_retrieval_df.groupby("question_type").agg(
        questions=("question_id", "count"),
        exact_recall_at_1=("exact_recall_at_1", "mean"),
        exact_recall_at_3=("exact_recall_at_3", "mean"),
        gold_answer_present_top1=("gold_answer_present_top1", "mean")
    ).round(4)
)

# Step 12: Save the detailed results.
exact_retrieval_df.to_csv(
    "/content/ipl_bm25_exact_passage_results.csv",
    index=False
)
print("\nDetailed results saved to /content/ipl_bm25_exact_passage_results.csv")

Test questions evaluated: 4545
Exact Passage Recall@1: 0.1597
Exact Passage Recall@3: 0.1637
Gold answer present in top-1 passage: 0.9923

Exact retrieval results by question type:


,questions,exact_recall_at_1,exact_recall_at_3,gold_answer_present_top1
question_type,,,,
batter_runs,3797,0.0000,0.0000,0.9966
innings_runs,505,0.9564,0.9921,0.9564
match_winner,243,1.0000,1.0000,1.0000



Detailed results saved to /content/ipl_bm25_exact_passage_results.csv


In [23]:
# STEP 22: FIX QUESTION-TO-EVIDENCE MAPPING
# Purpose:
# 1. Rebuild questions with explicit target passage IDs.
# 2. Link each batter question to the exact batter-and-innings passage.
# 3. Link match-winner and innings-run questions to match summaries.
# 4. Preserve the existing match-level train/validation/test split.
# 5. Verify that every target passage exists in its correct split.

# Step 1: Create a lookup of match-summary passage IDs.
summary_passage_ids = {
    str(row["match_id"]): str(row["passage_id"])
    for _, row in match_summary_df.iterrows()
}

# Step 2: Create a lookup of player-stat passage IDs.
player_passage_ids = {
    (
        str(row["match_id"]),
        int(row["innings_number"]),
        str(row["batter"])
    ): str(row["passage_id"])
    for _, row in player_stat_df.iterrows()
}

# Step 3: Initialise a fresh question list.
qa_records_fixed = []

# Step 4: Rebuild match-winner questions with their exact evidence IDs.
for _, row in matches_df.iterrows():
    match_id = str(row["match_id"])
    winner = row["winner"]

    if pd.notna(winner) and winner:
        qa_records_fixed.append({
            "question_id": f"{match_id}_winner",
            "match_id": match_id,
            "split": split_lookup[match_id],
            "question_type": "match_winner",
            "question": (
                f"Which team won the IPL match between "
                f"{row['teams'][0]} and {row['teams'][1]} "
                f"played on {row['date']}?"
            ),
            "gold_answer": str(winner),
            "target_passage_id": summary_passage_ids[match_id]
        })

# Step 5: Rebuild innings-run questions with their exact evidence IDs.
for _, row in innings_df.iterrows():
    match_id = str(row["match_id"])

    qa_records_fixed.append({
        "question_id": (
            f"{match_id}_innings{row['innings_number']}_runs"
        ),
        "match_id": match_id,
        "split": split_lookup[match_id],
        "question_type": "innings_runs",
        "question": (
            f"How many runs did {row['batting_team']} score "
            f"in innings {row['innings_number']} of the IPL match "
            f"played on {row['date']}?"
        ),
        "gold_answer": str(int(row["innings_runs"])),
        "target_passage_id": summary_passage_ids[match_id]
    })

# Step 6: Rebuild batter-run questions using the exact batter record.
for _, row in batter_df.iterrows():
    match_id = str(row["match_id"])
    innings_number = int(row["innings_number"])
    batter = str(row["batter"])

    target_id = player_passage_ids.get(
        (match_id, innings_number, batter)
    )

    qa_records_fixed.append({
        "question_id": (
            f"{match_id}_innings{innings_number}_{batter}_runs"
        ),
        "match_id": match_id,
        "split": split_lookup[match_id],
        "question_type": "batter_runs",
        "question": (
            f"How many runs did {batter} score for "
            f"{row['batting_team']} in innings {innings_number} "
            f"of the IPL match played on {row['date']}?"
        ),
        "gold_answer": str(int(row["batter_runs"])),
        "target_passage_id": target_id
    })

# Step 7: Replace the working question DataFrame with the corrected version.
qa_df = pd.DataFrame(qa_records_fixed)

# Step 8: Verify that all question IDs and target passage IDs are present.
print("Total questions:", len(qa_df))
print("Duplicate question IDs:", int(qa_df["question_id"].duplicated().sum()))
print("Missing target passage IDs:", int(qa_df["target_passage_id"].isna().sum()))

# Step 9: Verify each target passage exists in the appropriate collection.
summary_id_set = set(match_summary_df["passage_id"].astype(str))
player_id_set = set(player_stat_df["passage_id"].astype(str))

def target_passage_exists(row):
    target_id = str(row["target_passage_id"])

    if row["question_type"] == "batter_runs":
        return target_id in player_id_set

    return target_id in summary_id_set

qa_df["target_passage_exists"] = qa_df.apply(
    target_passage_exists, axis=1
)

print(
    "Valid target passages:",
    int(qa_df["target_passage_exists"].sum())
)
print(
    "Invalid target passages:",
    int((~qa_df["target_passage_exists"]).sum())
)

# Step 10: Preview batter questions and their explicit target passage IDs.
display(
    qa_df[qa_df["question_type"] == "batter_runs"][
        ["question", "gold_answer", "target_passage_id"]
    ].head(5)
)

Total questions: 22574
Duplicate question IDs: 0
Missing target passage IDs: 0
Valid target passages: 22574
Invalid target passages: 0


,question,gold_answer,target_passage_id
3732,How many runs did MJ Lumb score for Rajasthan ...,0,419118_innings1_MJ Lumb
3733,How many runs did FY Fazal score for Rajasthan...,31,419118_innings1_FY Fazal
3734,How many runs did NV Ojha score for Rajasthan ...,27,419118_innings1_NV Ojha
3735,How many runs did AA Jhunjhunwala score for Ra...,45,419118_innings1_AA Jhunjhunwala
3736,How many runs did YK Pathan score for Rajastha...,15,419118_innings1_YK Pathan


In [24]:
# STEP 22A: RESTORE MISSING PLAYER-PASSAGE METADATA
# Purpose:
# 1. Extract the innings number from each player-stat passage ID.
# 2. Add it as a dedicated DataFrame column.
# 3. Verify that match ID, innings number, and batter name are available.
# 4. Preserve the existing passage text and train/validation/test labels.

# Step 1: Extract the innings number from IDs such as "548331_innings1_N Saini".
player_stat_df["innings_number"] = (
    player_stat_df["passage_id"]
    .str.extract(r"_innings(\d+)_")[0]
    .astype(int)
)

# Step 2: Check that the required metadata columns are present.
required_columns = [
    "match_id",
    "innings_number",
    "batter",
    "passage_id",
    "split"
]

print("Required columns available:")
print({
    column: column in player_stat_df.columns
    for column in required_columns
})

# Step 3: Display a few records for verification.
display(player_stat_df[required_columns].head(5))

# Step 4: Check for missing innings numbers.
print(
    "\nMissing innings numbers:",
    int(player_stat_df["innings_number"].isna().sum())
)

Required columns available:
{'match_id': True, 'innings_number': True, 'batter': True, 'passage_id': True, 'split': True}


,match_id,innings_number,batter,passage_id,split
0,419118,1,MJ Lumb,419118_innings1_MJ Lumb,train
1,419118,1,FY Fazal,419118_innings1_FY Fazal,train
2,419118,1,NV Ojha,419118_innings1_NV Ojha,train
3,419118,1,AA Jhunjhunwala,419118_innings1_AA Jhunjhunwala,train
4,419118,1,YK Pathan,419118_innings1_YK Pathan,train



Missing innings numbers: 0


In [25]:
# STEP 22B: RESTORE BATTER NAMES FROM PLAYER-STAT PASSAGE IDS
# Purpose:
# 1. Extract the batter name from each passage ID.
# 2. Keep the existing innings_number column from Step 22A.
# 3. Verify the metadata required for exact-passage evaluation.
# 4. Check that no batter names or innings numbers are missing.

# Step 1: Extract the batter name from IDs such as "548331_innings1_N Saini".
player_stat_df["batter"] = player_stat_df["passage_id"].str.extract(
    r"_innings\d+_(.+)$"
)[0]

# Step 2: List the required metadata columns.
required_columns = [
    "match_id",
    "innings_number",
    "batter",
    "passage_id",
    "split"
]

# Step 3: Display the restored metadata.
print("Required columns available:")
print({
    column: column in player_stat_df.columns
    for column in required_columns
})

display(player_stat_df[required_columns].head(5))

# Step 4: Check for missing values.
print("\nMissing innings numbers:",
      int(player_stat_df["innings_number"].isna().sum()))

print("Missing batter names:",
      int(player_stat_df["batter"].isna().sum()))

# Step 5: Check whether the restored batter names match the original data.
print(
    "Player-stat passage count:",
    len(player_stat_df)
)

Required columns available:
{'match_id': True, 'innings_number': True, 'batter': True, 'passage_id': True, 'split': True}


,match_id,innings_number,batter,passage_id,split
0,419118,1,MJ Lumb,419118_innings1_MJ Lumb,train
1,419118,1,FY Fazal,419118_innings1_FY Fazal,train
2,419118,1,NV Ojha,419118_innings1_NV Ojha,train
3,419118,1,AA Jhunjhunwala,419118_innings1_AA Jhunjhunwala,train
4,419118,1,YK Pathan,419118_innings1_YK Pathan,train



Missing innings numbers: 0
Missing batter names: 0
Player-stat passage count: 18842


In [26]:
# STEP 22C: REBUILD QUESTIONS WITH EXACT TARGET PASSAGE IDS
# Purpose:
# 1. Link match-winner questions to match-summary passages.
# 2. Link innings-run questions to match-summary passages.
# 3. Link batter-run questions to the exact batter-and-innings passage.
# 4. Preserve the original match-level data split.
# 5. Validate that every target passage exists in the correct collection.

# Step 1: Create a lookup for match-summary passage IDs.
summary_passage_ids = {
    str(row["match_id"]): str(row["passage_id"])
    for _, row in match_summary_df.iterrows()
}

# Step 2: Create a lookup for player-stat passage IDs.
player_passage_ids = {
    (
        str(row["match_id"]),
        int(row["innings_number"]),
        str(row["batter"])
    ): str(row["passage_id"])
    for _, row in player_stat_df.iterrows()
}

# Step 3: Start a fresh list for the corrected questions.
qa_records_fixed = []

# Step 4: Rebuild match-winner questions.
for _, row in matches_df.iterrows():
    match_id = str(row["match_id"])
    winner = row["winner"]

    if pd.notna(winner) and winner:
        qa_records_fixed.append({
            "question_id": f"{match_id}_winner",
            "match_id": match_id,
            "split": split_lookup[match_id],
            "question_type": "match_winner",
            "question": (
                f"Which team won the IPL match between "
                f"{row['teams'][0]} and {row['teams'][1]} "
                f"played on {row['date']}?"
            ),
            "gold_answer": str(winner),
            "target_passage_id": summary_passage_ids[match_id]
        })

# Step 5: Rebuild innings-run questions.
for _, row in innings_df.iterrows():
    match_id = str(row["match_id"])

    qa_records_fixed.append({
        "question_id": (
            f"{match_id}_innings{row['innings_number']}_runs"
        ),
        "match_id": match_id,
        "split": split_lookup[match_id],
        "question_type": "innings_runs",
        "question": (
            f"How many runs did {row['batting_team']} score "
            f"in innings {row['innings_number']} of the IPL match "
            f"played on {row['date']}?"
        ),
        "gold_answer": str(int(row["innings_runs"])),
        "target_passage_id": summary_passage_ids[match_id]
    })

# Step 6: Rebuild batter-run questions.
for _, row in batter_df.iterrows():
    match_id = str(row["match_id"])
    innings_number = int(row["innings_number"])
    batter = str(row["batter"])

    target_id = player_passage_ids.get(
        (match_id, innings_number, batter)
    )

    qa_records_fixed.append({
        "question_id": (
            f"{match_id}_innings{innings_number}_{batter}_runs"
        ),
        "match_id": match_id,
        "split": split_lookup[match_id],
        "question_type": "batter_runs",
        "question": (
            f"How many runs did {batter} score for "
            f"{row['batting_team']} in innings {innings_number} "
            f"of the IPL match played on {row['date']}?"
        ),
        "gold_answer": str(int(row["batter_runs"])),
        "target_passage_id": target_id
    })

# Step 7: Replace qa_df with the corrected question dataset.
qa_df = pd.DataFrame(qa_records_fixed)

# Step 8: Verify question IDs and target passage IDs.
print("Total questions:", len(qa_df))
print("Duplicate question IDs:",
      int(qa_df["question_id"].duplicated().sum()))
print("Missing target passage IDs:",
      int(qa_df["target_passage_id"].isna().sum()))

# Step 9: Verify that each target passage exists in the appropriate collection.
summary_id_set = set(match_summary_df["passage_id"].astype(str))
player_id_set = set(player_stat_df["passage_id"].astype(str))

def target_passage_exists(row):
    target_id = str(row["target_passage_id"])

    if row["question_type"] == "batter_runs":
        return target_id in player_id_set

    return target_id in summary_id_set

qa_df["target_passage_exists"] = qa_df.apply(
    target_passage_exists, axis=1
)

print("Valid target passages:",
      int(qa_df["target_passage_exists"].sum()))
print("Invalid target passages:",
      int((~qa_df["target_passage_exists"]).sum()))

# Step 10: Check that the target passage belongs to the same split as its question.
player_split_lookup = player_stat_df.set_index(
    "passage_id"
)["split"].to_dict()

summary_split_lookup = match_summary_df.set_index(
    "passage_id"
)["split"].to_dict()

def target_split_matches(row):
    target_id = str(row["target_passage_id"])

    if row["question_type"] == "batter_runs":
        target_split = player_split_lookup.get(target_id)
    else:
        target_split = summary_split_lookup.get(target_id)

    return target_split == row["split"]

qa_df["target_split_matches"] = qa_df.apply(
    target_split_matches, axis=1
)

print("Targets in the correct split:",
      int(qa_df["target_split_matches"].sum()))
print("Targets in the wrong split:",
      int((~qa_df["target_split_matches"]).sum()))

# Step 11: Preview the corrected batter question mappings.
display(
    qa_df[qa_df["question_type"] == "batter_runs"][
        ["question", "gold_answer", "target_passage_id"]
    ].head(5)
)

Total questions: 22574
Duplicate question IDs: 0
Missing target passage IDs: 0
Valid target passages: 22574
Invalid target passages: 0
Targets in the correct split: 22574
Targets in the wrong split: 0


,question,gold_answer,target_passage_id
3732,How many runs did MJ Lumb score for Rajasthan ...,0,419118_innings1_MJ Lumb
3733,How many runs did FY Fazal score for Rajasthan...,31,419118_innings1_FY Fazal
3734,How many runs did NV Ojha score for Rajasthan ...,27,419118_innings1_NV Ojha
3735,How many runs did AA Jhunjhunwala score for Ra...,45,419118_innings1_AA Jhunjhunwala
3736,How many runs did YK Pathan score for Rajastha...,15,419118_innings1_YK Pathan


In [27]:
# STEP 23: EVALUATE EXACT-PASSAGE RETRIEVAL ON THE TEST SET
# Purpose:
# 1. Evaluate only held-out test questions.
# 2. Retrieve the top 3 passages from the appropriate test-split index.
# 3. Compare retrieved passage IDs with the verified target passage ID.
# 4. Calculate Exact Passage Recall@1 and Recall@3.
# 5. Save question-level results for reproducibility.

# Step 1: Select held-out test questions.
test_questions_df = (
    qa_df[qa_df["split"] == "test"]
    .reset_index(drop=True)
)

# Step 2: Map question types to their evidence collections.
collection_for_question = {
    "match_winner": "match_summary",
    "innings_runs": "match_summary",
    "batter_runs": "player_stat"
}

# Step 3: Evaluate each question against the matching test-split index.
exact_results = []

for _, row in test_questions_df.iterrows():
    question_type = row["question_type"]
    collection_name = collection_for_question[question_type]

    index = bm25_indexes[("test", collection_name)]
    passage_store = bm25_passage_stores[("test", collection_name)]

    # Step 4: Retrieve the top 3 passages for this question.
    query_tokens = tokenize_text(row["question"])
    scores = index.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:3]

    retrieved_passages = passage_store.iloc[top_indices]
    retrieved_ids = retrieved_passages["passage_id"].astype(str).tolist()

    # Step 5: Compare results against the verified target passage.
    target_id = str(row["target_passage_id"])

    exact_recall_at_1 = (
        len(retrieved_ids) > 0 and retrieved_ids[0] == target_id
    )
    exact_recall_at_3 = target_id in retrieved_ids

    exact_results.append({
        "question_id": row["question_id"],
        "match_id": row["match_id"],
        "split": row["split"],
        "question_type": question_type,
        "question": row["question"],
        "gold_answer": row["gold_answer"],
        "target_passage_id": target_id,
        "top1_passage_id": retrieved_ids[0] if retrieved_ids else None,
        "top3_passage_ids": " | ".join(retrieved_ids),
        "exact_recall_at_1": int(exact_recall_at_1),
        "exact_recall_at_3": int(exact_recall_at_3)
    })

# Step 6: Convert results into a DataFrame.
exact_results_df = pd.DataFrame(exact_results)

# Step 7: Report overall exact-passage retrieval metrics.
print("Test questions evaluated:", len(exact_results_df))
print(
    "Exact Passage Recall@1:",
    round(exact_results_df["exact_recall_at_1"].mean(), 4)
)
print(
    "Exact Passage Recall@3:",
    round(exact_results_df["exact_recall_at_3"].mean(), 4)
)

# Step 8: Report metrics separately for each question type.
exact_summary_df = (
    exact_results_df
    .groupby("question_type")
    .agg(
        questions=("question_id", "count"),
        exact_recall_at_1=("exact_recall_at_1", "mean"),
        exact_recall_at_3=("exact_recall_at_3", "mean")
    )
    .reset_index()
)

print("\nMetrics by question type:")
display(exact_summary_df)

# Step 9: Save the full question-level evaluation results.
exact_results_path = "/content/ipl_bm25_exact_passage_results.csv"
exact_results_df.to_csv(exact_results_path, index=False)

print("\nSaved results to:", exact_results_path)

# Step 10: Inspect examples where the exact passage was not ranked first.
display(
    exact_results_df[
        exact_results_df["exact_recall_at_1"] == 0
    ][
        [
            "question_type",
            "question",
            "target_passage_id",
            "top1_passage_id",
            "exact_recall_at_3"
        ]
    ].head(10)
)

Test questions evaluated: 4545
Exact Passage Recall@1: 0.9892
Exact Passage Recall@3: 0.9991

Metrics by question type:


,question_type,questions,exact_recall_at_1,exact_recall_at_3
0,batter_runs,3797,0.992889,1.000000
1,innings_runs,505,0.956436,0.992079
2,match_winner,243,1.000000,1.000000



Saved results to: /content/ipl_bm25_exact_passage_results.csv


,question_type,question,target_passage_id,top1_passage_id,exact_recall_at_3
262,innings_runs,How many runs did Deccan Chargers score in inn...,392236_summary,392237_summary,1
275,innings_runs,How many runs did Rajasthan Royals score in in...,1473455_summary,1473487_summary,1
276,innings_runs,How many runs did Punjab Kings score in inning...,1473455_summary,1473481_summary,1
286,innings_runs,How many runs did Rajasthan Royals score in in...,1359482_summary,1359526_summary,0
307,innings_runs,How many runs did Kochi Tuskers Kerala score i...,501242_summary,501258_summary,1
369,innings_runs,How many runs did Rising Pune Supergiants scor...,980919_summary,980997_summary,1
371,innings_runs,How many runs did Sunrisers Hyderabad score in...,1359484_summary,1359526_summary,1
389,innings_runs,How many runs did Kolkata Knight Riders score ...,1178431_summary,1175372_summary,1
408,innings_runs,How many runs did Kings XI Punjab score in inn...,1216523_summary,1216531_summary,1
430,innings_runs,How many runs did Chennai Super Kings score in...,598046_summary,598070_summary,1


In [28]:
# STEP 24: INSPECT EXACT-PASSAGE RETRIEVAL ERRORS
# Purpose:
# 1. Isolate test questions where the correct passage was not ranked first.
# 2. Focus on innings-run questions, the weakest category.
# 3. Compare the question, gold passage, and top-ranked passage.
# 4. Check whether the correct passage was still retrieved in the top 3.

# Step 1: Filter for innings-run questions with an incorrect top-ranked passage.
innings_errors_df = exact_results_df[
    (exact_results_df["question_type"] == "innings_runs") &
    (exact_results_df["exact_recall_at_1"] == 0)
].copy()

# Step 2: Display the number of errors.
print("Innings-run questions missed at rank 1:", len(innings_errors_df))
print(
    "Correct passage still in top 3:",
    int(innings_errors_df["exact_recall_at_3"].sum())
)
print(
    "Correct passage absent from top 3:",
    int((innings_errors_df["exact_recall_at_3"] == 0).sum())
)

# Step 3: Retrieve the actual passage text for a small sample.
summary_text_lookup = match_summary_df.set_index(
    match_summary_df["passage_id"].astype(str)
)["text"].to_dict()

innings_errors_df["target_passage_text"] = (
    innings_errors_df["target_passage_id"]
    .map(summary_text_lookup)
)

innings_errors_df["top1_passage_text"] = (
    innings_errors_df["top1_passage_id"]
    .map(summary_text_lookup)
)

# Step 4: Display five examples for manual error analysis.
display(
    innings_errors_df[
        [
            "question",
            "gold_answer",
            "target_passage_id",
            "top1_passage_id",
            "exact_recall_at_3",
            "target_passage_text",
            "top1_passage_text"
        ]
    ].head(5)
)

Innings-run questions missed at rank 1: 22
Correct passage still in top 3: 18
Correct passage absent from top 3: 4


,question,gold_answer,target_passage_id,top1_passage_id,exact_recall_at_3,target_passage_text,top1_passage_text
262,How many runs did Deccan Chargers score in inn...,158,392236_summary,392237_summary,1,IPL match ID: 392236. Date: 2009-05-21. Season...,IPL match ID: 392237. Date: 2009-05-22. Season...
275,How many runs did Rajasthan Royals score in in...,205,1473455_summary,1473487_summary,1,IPL match ID: 1473455. Date: 2025-04-05. Seaso...,IPL match ID: 1473487. Date: 2025-05-01. Seaso...
276,How many runs did Punjab Kings score in inning...,155,1473455_summary,1473481_summary,1,IPL match ID: 1473455. Date: 2025-04-05. Seaso...,IPL match ID: 1473481. Date: 2025-04-26. Seaso...
286,How many runs did Rajasthan Royals score in in...,192,1359482_summary,1359526_summary,0,IPL match ID: 1359482. Date: 2023-04-05. Seaso...,IPL match ID: 1359526. Date: 2023-05-07. Seaso...
307,How many runs did Kochi Tuskers Kerala score i...,156,501242_summary,501258_summary,1,IPL match ID: 501242. Date: 2011-05-05. Season...,IPL match ID: 501258. Date: 2011-05-15. Season...


In [29]:
# STEP 25: DIAGNOSE WRONG-MATCH RETRIEVAL
# Purpose:
# 1. Extract dates from the target and top-ranked passages.
# 2. Check whether the question date matches the target passage date.
# 3. Inspect the four cases where the correct passage was absent from the top 3.
# 4. Avoid changing the retrieval algorithm before understanding the errors.

# Step 1: Extract a date from a passage using its standard text format.
def extract_passage_date(text):
    # Step 1a: Find dates formatted as YYYY-MM-DD.
    match = re.search(r"Date:\s*(\d{4}-\d{2}-\d{2})", str(text))
    return match.group(1) if match else None

# Step 2: Extract the date stated in each question.
def extract_question_date(text):
    # Step 2a: Find dates formatted as YYYY-MM-DD.
    match = re.search(r"(\d{4}-\d{2}-\d{2})", str(text))
    return match.group(1) if match else None

# Step 3: Create a diagnostic copy of the rank-1 errors.
diagnostic_df = innings_errors_df.copy()

# Step 4: Compare dates across the question and passages.
diagnostic_df["question_date"] = diagnostic_df["question"].apply(
    extract_question_date
)
diagnostic_df["target_date"] = diagnostic_df["target_passage_text"].apply(
    extract_passage_date
)
diagnostic_df["top1_date"] = diagnostic_df["top1_passage_text"].apply(
    extract_passage_date
)

diagnostic_df["question_matches_target_date"] = (
    diagnostic_df["question_date"] == diagnostic_df["target_date"]
)

# Step 5: Summarize the date checks.
print("Rank-1 errors examined:", len(diagnostic_df))
print(
    "Question date matches target passage date:",
    int(diagnostic_df["question_matches_target_date"].sum())
)
print(
    "Question date missing:",
    int(diagnostic_df["question_date"].isna().sum())
)

# Step 6: Display four examples where the target was not in the top 3.
missed_top3_df = diagnostic_df[
    diagnostic_df["exact_recall_at_3"] == 0
]

display(
    missed_top3_df[
        [
            "question",
            "gold_answer",
            "question_date",
            "target_date",
            "top1_date",
            "target_passage_id",
            "top1_passage_id",
            "target_passage_text",
            "top1_passage_text"
        ]
    ].head(4)
)

Rank-1 errors examined: 22
Question date matches target passage date: 22
Question date missing: 0


,question,gold_answer,question_date,target_date,top1_date,target_passage_id,top1_passage_id,target_passage_text,top1_passage_text
286,How many runs did Rajasthan Royals score in in...,192,2023-04-05,2023-04-05,2023-05-07,1359482_summary,1359526_summary,IPL match ID: 1359482. Date: 2023-04-05. Seaso...,IPL match ID: 1359526. Date: 2023-05-07. Seaso...
452,How many runs did Deccan Chargers score in inn...,150,2012-05-04,2012-05-04,2012-04-26,548352_summary,548341_summary,IPL match ID: 548352. Date: 2012-05-04. Season...,IPL match ID: 548341. Date: 2012-04-26. Season...
608,How many runs did Mumbai Indians score in inni...,162,2026-04-04,2026-04-04,2026-05-24,1527681_summary,1529312_summary,IPL match ID: 1527681. Date: 2026-04-04. Seaso...,IPL match ID: 1529312. Date: 2026-05-24. Seaso...
654,How many runs did Chennai Super Kings score in...,165,2024-04-05,2024-04-05,2024-03-26,1422136_summary,1422125_summary,IPL match ID: 1422136. Date: 2024-04-05. Seaso...,IPL match ID: 1422125. Date: 2024-03-26. Seaso...


In [30]:
# STEP 26: TEST DATE-FILTERED BM25 RETRIEVAL
# Purpose:
# 1. Preserve the original BM25 baseline.
# 2. Restrict summary candidates to the question's exact date.
# 3. Rank the remaining candidates using BM25.
# 4. Compare exact-passage Recall@1 and Recall@3.
# 5. Report questions for which no date-matched candidate exists.

# Step 1: Prepare the test questions for this diagnostic.
date_filter_results = []

# Step 2: Build a date lookup for the summary passages.
summary_store = bm25_passage_stores[("test", "match_summary")].copy()

summary_store["passage_date"] = summary_store["text"].apply(
    extract_passage_date
)

# Step 3: Evaluate date filtering on innings-run questions.
innings_test_df = test_questions_df[
    test_questions_df["question_type"] == "innings_runs"
]

for _, row in innings_test_df.iterrows():
    # Step 3a: Extract the question's date.
    question_date = extract_question_date(row["question"])

    # Step 3b: Keep only passages from that date.
    candidates = summary_store[
        summary_store["passage_date"] == question_date
    ].copy()

    # Step 3c: Rank the date-matched candidates with BM25.
    if len(candidates) > 0:
        query_tokens = tokenize_text(row["question"])
        candidate_tokens = [
            tokenize_text(text) for text in candidates["text"]
        ]

        candidate_index = BM25Okapi(candidate_tokens)
        scores = candidate_index.get_scores(query_tokens)
        top_indices = scores.argsort()[::-1][:3]

        retrieved_ids = (
            candidates.iloc[top_indices]["passage_id"]
            .astype(str)
            .tolist()
        )
    else:
        retrieved_ids = []

    # Step 3d: Compare retrieval against the verified target passage.
    target_id = str(row["target_passage_id"])

    date_filter_results.append({
        "question_id": row["question_id"],
        "question": row["question"],
        "question_date": question_date,
        "target_passage_id": target_id,
        "candidate_count": len(candidates),
        "top1_passage_id": (
            retrieved_ids[0] if retrieved_ids else None
        ),
        "exact_recall_at_1": int(
            len(retrieved_ids) > 0 and retrieved_ids[0] == target_id
        ),
        "exact_recall_at_3": int(target_id in retrieved_ids)
    })

# Step 4: Summarize the diagnostic results.
date_filter_df = pd.DataFrame(date_filter_results)

print("Innings-run test questions:", len(date_filter_df))
print(
    "Questions with date-matched candidates:",
    int((date_filter_df["candidate_count"] > 0).sum())
)
print(
    "Date-filtered Exact Recall@1:",
    round(date_filter_df["exact_recall_at_1"].mean(), 4)
)
print(
    "Date-filtered Exact Recall@3:",
    round(date_filter_df["exact_recall_at_3"].mean(), 4)
)

# Step 5: Inspect any remaining rank-1 errors.
display(
    date_filter_df[
        date_filter_df["exact_recall_at_1"] == 0
    ].head(10)
)

# Step 6: Save the diagnostic results separately from the baseline.
date_filter_df.to_csv(
    "/content/ipl_bm25_date_filtered_diagnostic.csv",
    index=False
)

print("Saved: /content/ipl_bm25_date_filtered_diagnostic.csv")

Innings-run test questions: 505
Questions with date-matched candidates: 505
Date-filtered Exact Recall@1: 0.9723
Date-filtered Exact Recall@3: 1.0


,question_id,question,question_date,target_passage_id,candidate_count,top1_passage_id,exact_recall_at_1,exact_recall_at_3
220,598016_innings1_runs,How many runs did Kings XI Punjab score in inn...,2013-04-16,598016_summary,2,598017_summary,0,1
248,1359478_innings1_runs,How many runs did Rajasthan Royals score in in...,2023-04-02,1359478_summary,2,1359479_summary,0,1
291,1216509_innings1_runs,How many runs did Chennai Super Kings score in...,2020-10-17,1216509_summary,2,1216522_summary,0,1
292,1216509_innings2_runs,How many runs did Delhi Capitals score in inni...,2020-10-17,1216509_summary,2,1216522_summary,0,1
293,548310_innings1_runs,How many runs did Royal Challengers Bangalore ...,2012-04-07,548310_summary,2,548311_summary,0,1
294,548310_innings2_runs,How many runs did Delhi Daredevils score in in...,2012-04-07,548310_summary,2,548311_summary,0,1
363,1473472_innings1_runs,How many runs did Delhi Capitals score in inni...,2025-04-19,1473472_summary,2,1473473_summary,0,1
364,1473472_innings2_runs,How many runs did Gujarat Titans score in inni...,2025-04-19,1473472_summary,2,1473473_summary,0,1
389,598047_innings1_runs,How many runs did Pune Warriors score in innin...,2013-05-05,598047_summary,2,598046_summary,0,1
390,598047_innings2_runs,How many runs did Rajasthan Royals score in in...,2013-05-05,598047_summary,2,598046_summary,0,1


Saved: /content/ipl_bm25_date_filtered_diagnostic.csv


In [31]:
# STEP 27: TEST DATE-AND-TEAM-FILTERED RETRIEVAL
# Purpose:
# 1. Filter test-summary passages by the question date and batting team.
# 2. Rank the remaining candidates with BM25.
# 3. Measure exact-passage Recall@1 and Recall@3.
# 4. Count questions where filtering finds no candidate.
# 5. Save results separately from the original BM25 baseline.

# Step 1: Prepare the summary passages and extract their dates.
summary_store = bm25_passage_stores[("test", "match_summary")].copy()
summary_store["passage_date"] = summary_store["text"].apply(
    extract_passage_date
)

# Step 2: Extract batting-team names from the innings-run questions.
def extract_batting_team(question):
    # Step 2a: Match the team name between "did" and "score".
    match = re.search(r"did (.+?) score in innings", question, flags=re.I)
    return match.group(1).strip() if match else None

# Step 3: Evaluate the filter on test innings-run questions.
team_filter_results = []
innings_test_df = test_questions_df[
    test_questions_df["question_type"] == "innings_runs"
]

for _, row in innings_test_df.iterrows():
    # Step 3a: Extract the date and batting team from the question.
    question_date = extract_question_date(row["question"])
    batting_team = extract_batting_team(row["question"])

    # Step 3b: Filter by date and the team's presence in the passage.
    candidates = summary_store[
        (summary_store["passage_date"] == question_date) &
        (summary_store["text"].str.contains(
            re.escape(str(batting_team)),
            case=False,
            regex=True,
            na=False
        ))
    ].copy() if batting_team else summary_store.iloc[0:0].copy()

    # Step 3c: Rank candidates with BM25.
    if len(candidates) > 0:
        candidate_index = BM25Okapi([
            tokenize_text(text) for text in candidates["text"]
        ])
        scores = candidate_index.get_scores(
            tokenize_text(row["question"])
        )
        top_indices = scores.argsort()[::-1][:3]
        retrieved_ids = (
            candidates.iloc[top_indices]["passage_id"]
            .astype(str)
            .tolist()
        )
    else:
        retrieved_ids = []

    # Step 3d: Compare the retrieved passages with the gold passage.
    target_id = str(row["target_passage_id"])

    team_filter_results.append({
        "question_id": row["question_id"],
        "question": row["question"],
        "question_date": question_date,
        "batting_team": batting_team,
        "target_passage_id": target_id,
        "candidate_count": len(candidates),
        "top1_passage_id": (
            retrieved_ids[0] if retrieved_ids else None
        ),
        "exact_recall_at_1": int(
            bool(retrieved_ids) and retrieved_ids[0] == target_id
        ),
        "exact_recall_at_3": int(target_id in retrieved_ids)
    })

# Step 4: Summarize the results.
team_filter_df = pd.DataFrame(team_filter_results)

print("Innings-run test questions:", len(team_filter_df))
print(
    "Questions with matching candidates:",
    int((team_filter_df["candidate_count"] > 0).sum())
)
print(
    "Questions with no matching candidates:",
    int((team_filter_df["candidate_count"] == 0).sum())
)
print(
    "Date-and-team-filtered Exact Recall@1:",
    round(team_filter_df["exact_recall_at_1"].mean(), 4)
)
print(
    "Date-and-team-filtered Exact Recall@3:",
    round(team_filter_df["exact_recall_at_3"].mean(), 4)
)

# Step 5: Inspect any failures.
display(
    team_filter_df[
        (team_filter_df["exact_recall_at_1"] == 0) |
        (team_filter_df["candidate_count"] == 0)
    ].head(10)
)

# Step 6: Save the diagnostic results separately.
team_filter_df.to_csv(
    "/content/ipl_bm25_date_team_filtered_diagnostic.csv",
    index=False
)

print("Saved: /content/ipl_bm25_date_team_filtered_diagnostic.csv")

Innings-run test questions: 505
Questions with matching candidates: 505
Questions with no matching candidates: 0
Date-and-team-filtered Exact Recall@1: 1.0
Date-and-team-filtered Exact Recall@3: 1.0


,question_id,question,question_date,batting_team,target_passage_id,candidate_count,top1_passage_id,exact_recall_at_1,exact_recall_at_3


Saved: /content/ipl_bm25_date_team_filtered_diagnostic.csv


In [32]:
# STEP 28: INSPECT SUMMARY PASSAGE STRUCTURE
# Purpose:
# 1. Inspect actual match-summary passages.
# 2. Check how batting teams and innings totals are represented.
# 3. Prepare to validate the date-and-team filter correctly.

# Step 1: Print three complete summary passages.
for i, passage in enumerate(match_summary_df["text"].head(3), start=1):
    print(f"\nExample {i}")
    print("-" * 60)
    print(passage)

# Step 2: Display the innings data columns.
print("\nInnings data columns:")
print(innings_df.columns.tolist())

# Step 3: Preview three innings-level records.
display(
    innings_df[
        ["match_id", "date", "innings_number", "batting_team", "innings_runs"]
    ].head(3)
)


Example 1
------------------------------------------------------------
IPL match ID: 419118. Date: 2010-03-20. Season: 2009/10. Teams: Rajasthan Royals, Kolkata Knight Riders. Venue: Sardar Patel Stadium, Motera. City: Ahmedabad. Winner: Rajasthan Royals. Toss winner: Rajasthan Royals. Toss decision: bat. Player of the match: AA Jhunjhunwala. Rajasthan Royals scored 168 runs and lost 7 wickets in innings 1. Kolkata Knight Riders scored 134 runs and lost 5 wickets in innings 2.

Example 2
------------------------------------------------------------
IPL match ID: 336012. Date: 2008-05-28. Season: 2007/08. Teams: Royal Challengers Bangalore, Mumbai Indians. Venue: M Chinnaswamy Stadium. City: Bangalore. Winner: Mumbai Indians. Toss winner: Mumbai Indians. Toss decision: field. Player of the match: CRD Fernando. Royal Challengers Bangalore scored 122 runs and lost 9 wickets in innings 1. Mumbai Indians scored 126 runs and lost 1 wickets in innings 2.

Example 3
---------------------------

,match_id,date,innings_number,batting_team,innings_runs
0,419118,2010-03-20,1,Rajasthan Royals,168
1,419118,2010-03-20,2,Kolkata Knight Riders,134
2,336012,2008-05-28,1,Royal Challengers Bangalore,122


In [33]:
# STEP 29: VALIDATE MATCH IDENTITY IN DATE-AND-TEAM FILTERING
# Purpose:
# 1. Verify that the retrieved passage belongs to the correct match.
# 2. Check all 505 innings-run test questions.
# 3. Distinguish genuine match identification from team-name overlap.

# Step 1: Create a lookup of passage IDs to match IDs.
summary_match_lookup = {
    str(row["passage_id"]): str(row["match_id"])
    for _, row in match_summary_df.iterrows()
}

# Step 2: Copy the date-and-team diagnostic results.
identity_check_df = team_filter_df.copy()

# Step 3: Map target and retrieved passages to their match IDs.
identity_check_df["target_match_id"] = (
    identity_check_df["target_passage_id"]
    .astype(str)
    .map(summary_match_lookup)
)

identity_check_df["retrieved_match_id"] = (
    identity_check_df["top1_passage_id"]
    .astype(str)
    .map(summary_match_lookup)
)

# Step 4: Check whether the retrieved match is the intended match.
identity_check_df["correct_match"] = (
    identity_check_df["target_match_id"]
    == identity_check_df["retrieved_match_id"]
)

# Step 5: Report match-identity accuracy.
print("Questions checked:", len(identity_check_df))
print(
    "Top-1 passages from the correct match:",
    int(identity_check_df["correct_match"].sum())
)
print(
    "Top-1 passages from the wrong match:",
    int((~identity_check_df["correct_match"]).sum())
)

# Step 6: Display any remaining identity errors.
display(
    identity_check_df[~identity_check_df["correct_match"]][
        [
            "question",
            "question_date",
            "batting_team",
            "target_passage_id",
            "top1_passage_id",
            "target_match_id",
            "retrieved_match_id"
        ]
    ].head(10)
)

Questions checked: 505
Top-1 passages from the correct match: 505
Top-1 passages from the wrong match: 0


,question,question_date,batting_team,target_passage_id,top1_passage_id,target_match_id,retrieved_match_id


In [34]:
# STEP 30: SAVE THE CURRENT RETRIEVAL EXPERIMENT SUMMARY
# Purpose:
# 1. Record the original BM25 baseline.
# 2. Record date-filtered and date-plus-team-filtered results.
# 3. Keep the metrics separate for reproducible reporting.

# Step 1: Create a summary of the retrieval experiments.
experiment_summary_df = pd.DataFrame([
    {
        "experiment": "Original BM25",
        "question_type": "innings_runs",
        "test_questions": 505,
        "exact_recall_at_1": 0.956436,
        "exact_recall_at_3": 0.992079,
        "match_identity_accuracy": None,
        "notes": "Unfiltered BM25 over test-split summary passages"
    },
    {
        "experiment": "Date-filtered BM25",
        "question_type": "innings_runs",
        "test_questions": 505,
        "exact_recall_at_1": 0.9723,
        "exact_recall_at_3": 1.0,
        "match_identity_accuracy": None,
        "notes": "Candidates filtered using the question date"
    },
    {
        "experiment": "Date-and-team-filtered BM25",
        "question_type": "innings_runs",
        "test_questions": 505,
        "exact_recall_at_1": 1.0,
        "exact_recall_at_3": 1.0,
        "match_identity_accuracy": 1.0,
        "notes": "Metadata-assisted diagnostic; all 505 top-1 passages matched the target match"
    }
])

# Step 2: Display the summary.
display(experiment_summary_df)

# Step 3: Save it to a CSV file.
summary_path = "/content/ipl_retrieval_experiment_summary.csv"
experiment_summary_df.to_csv(summary_path, index=False)

print("Saved experiment summary to:", summary_path)

,experiment,question_type,test_questions,exact_recall_at_1,exact_recall_at_3,match_identity_accuracy,notes
0,Original BM25,innings_runs,505,0.956436,0.992079,NaN,Unfiltered BM25 over test-split summary passages
1,Date-filtered BM25,innings_runs,505,0.972300,1.000000,NaN,Candidates filtered using the question date
2,Date-and-team-filtered BM25,innings_runs,505,1.000000,1.000000,1.0,Metadata-assisted diagnostic; all 505 top-1 pa...


Saved experiment summary to: /content/ipl_retrieval_experiment_summary.csv


In [35]:
# STEP 31: CHECK TEST QUESTIONS FOR ANSWER-GENERATION EVALUATION
# Purpose:
# 1. Count test questions by type.
# 2. Check for missing questions or gold answers.
# 3. Inspect example questions and answers before model evaluation.
# 4. Keep the evaluation dataset fixed across all retrieval methods.

# Step 1: Select the held-out test questions.
test_questions_df = (
    qa_df[qa_df["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

# Step 2: Summarize test-question counts and answer completeness.
test_data_summary = (
    test_questions_df
    .groupby("question_type")
    .agg(
        question_count=("question_id", "count"),
        missing_questions=("question", lambda s: int(s.isna().sum())),
        missing_gold_answers=("gold_answer", lambda s: int(s.isna().sum())),
        unique_matches=("match_id", "nunique")
    )
    .reset_index()
)

print("Total held-out test questions:", len(test_questions_df))
display(test_data_summary)

# Step 3: Check for duplicate question IDs and missing values.
print(
    "\nDuplicate test question IDs:",
    int(test_questions_df["question_id"].duplicated().sum())
)
print(
    "Missing target passage IDs:",
    int(test_questions_df["target_passage_id"].isna().sum())
)
print(
    "Empty gold answers:",
    int(
        test_questions_df["gold_answer"]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
)

# Step 4: Inspect a few examples from every question type.
for question_type in sorted(test_questions_df["question_type"].unique()):
    print(f"\nExamples: {question_type}")
    display(
        test_questions_df[
            test_questions_df["question_type"] == question_type
        ][
            ["question", "gold_answer", "target_passage_id"]
        ].head(3)
    )

Total held-out test questions: 4545


,question_type,question_count,missing_questions,missing_gold_answers,unique_matches
0,batter_runs,3797,0,0,249
1,innings_runs,505,0,0,249
2,match_winner,243,0,0,243



Duplicate test question IDs: 0
Missing target passage IDs: 0
Empty gold answers: 0

Examples: batter_runs


,question,gold_answer,target_passage_id
748,How many runs did N Saini score for Kings XI P...,14,548331_innings1_N Saini
749,How many runs did PC Valthaty score for Kings ...,6,548331_innings1_PC Valthaty
750,How many runs did SE Marsh score for Kings XI ...,26,548331_innings1_SE Marsh



Examples: innings_runs


,question,gold_answer,target_passage_id
243,How many runs did Kings XI Punjab score in inn...,163,548331_summary
244,How many runs did Royal Challengers Bangalore ...,166,548331_summary
245,How many runs did Kolkata Knight Riders score ...,148,501230_summary



Examples: match_winner


,question,gold_answer,target_passage_id
0,Which team won the IPL match between Kings XI ...,Royal Challengers Bangalore,548331_summary
1,Which team won the IPL match between Delhi Dar...,Kolkata Knight Riders,501230_summary
2,Which team won the IPL match between Punjab Ki...,Punjab Kings,1473442_summary


In [36]:
# STEP 32: INSTALL AND LOAD A LIGHTWEIGHT QUESTION-ANSWERING MODEL
# Purpose:
# 1. Install the Hugging Face libraries needed for text generation.
# 2. Load FLAN-T5-small using the available Colab device.
# 3. Confirm that the model can generate a short answer.
# 4. Test one question before running a full experiment.

# Step 1: Install the required libraries.
%pip -q install transformers sentencepiece

# Step 2: Import the model and tokenizer tools.
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Step 3: Select a lightweight, publicly available model.
MODEL_NAME = "google/flan-t5-small"

# Step 4: Load the tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Step 5: Use a GPU if Colab provides one; otherwise use the CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model:", MODEL_NAME)
print("Device:", device)

# Step 6: Test the model with one simple question.
test_prompt = (
    "Answer the question using only the supplied evidence. "
    "If the evidence does not contain the answer, say 'Not stated'. "
    "Evidence: Rajasthan Royals scored 168 runs in innings 1. "
    "Question: How many runs did Rajasthan Royals score in innings 1? "
    "Answer:"
)

# Step 7: Generate a short response.
inputs = tokenizer(
    test_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False
    )

test_answer = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("Test answer:", test_answer)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model: google/flan-t5-small
Device: cpu
Test answer: 168


In [37]:
# STEP 33: RUN A FIVE-QUESTION ANSWER-GENERATION PILOT
# Purpose:
# 1. Select five questions from the held-out test set.
# 2. Retrieve each question's verified target evidence passage.
# 3. Ask FLAN-T5-small to answer using that evidence only.
# 4. Compare the generated answers with the gold answers.
# 5. Do not change or overwrite the retrieval baseline.

# Step 1: Create a lookup for the evidence passages.
summary_text_lookup = {
    str(row["passage_id"]): str(row["text"])
    for _, row in match_summary_df.iterrows()
}

player_text_lookup = {
    str(row["passage_id"]): str(row["text"])
    for _, row in player_stat_df.iterrows()
}

# Step 2: Select one question from each category, plus two more.
pilot_questions = (
    test_questions_df.groupby("question_type", group_keys=False)
    .head(2)
    .head(5)
    .copy()
)

# Step 3: Retrieve the verified target evidence for each question.
def get_target_evidence(row):
    target_id = str(row["target_passage_id"])

    if row["question_type"] == "batter_runs":
        return player_text_lookup.get(target_id, "")

    return summary_text_lookup.get(target_id, "")

pilot_questions["evidence"] = pilot_questions.apply(
    get_target_evidence, axis=1
)

# Step 4: Generate answers from the evidence.
pilot_records = []

for _, row in pilot_questions.iterrows():
    prompt = (
        "Answer the question using only the evidence. "
        "Return a short answer. Do not guess. "
        "If the answer is not stated, say 'Not stated'.\n\n"
        f"Evidence: {row['evidence']}\n\n"
        f"Question: {row['question']}\n"
        "Answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False
        )

    generated_answer = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

    pilot_records.append({
        "question_type": row["question_type"],
        "question": row["question"],
        "gold_answer": str(row["gold_answer"]),
        "generated_answer": generated_answer,
        "target_passage_id": row["target_passage_id"],
        "evidence_available": bool(row["evidence"])
    })

# Step 5: Display the pilot results.
pilot_results_df = pd.DataFrame(pilot_records)

print("Pilot questions:", len(pilot_results_df))
print(
    "Questions with target evidence:",
    int(pilot_results_df["evidence_available"].sum())
)

display(pilot_results_df)

Pilot questions: 5
Questions with target evidence: 5


,question_type,question,gold_answer,generated_answer,target_passage_id,evidence_available
0,match_winner,Which team won the IPL match between Kings XI ...,Royal Challengers Bangalore,Royal Challengers Bangalore,548331_summary,True
1,match_winner,Which team won the IPL match between Delhi Dar...,Kolkata Knight Riders,Delhi Daredevils,501230_summary,True
2,innings_runs,How many runs did Kings XI Punjab score in inn...,163,163,548331_summary,True
3,innings_runs,How many runs did Royal Challengers Bangalore ...,166,166,548331_summary,True
4,batter_runs,How many runs did N Saini score for Kings XI P...,14,14,548331_innings1_N Saini,True


In [38]:
# STEP 34: INSPECT THE INCORRECT MATCH-WINNER ANSWER
# Purpose:
# 1. Identify the pilot question answered incorrectly.
# 2. Display its gold answer and generated answer.
# 3. Inspect the complete target evidence passage.
# 4. Verify that the evidence contains the expected winner.

# Step 1: Select pilot answers that differ from the gold answer.
incorrect_pilot_df = pilot_results_df[
    pilot_results_df["generated_answer"].str.strip().str.lower()
    != pilot_results_df["gold_answer"].str.strip().str.lower()
].copy()

print("Incorrect pilot answers:", len(incorrect_pilot_df))

# Step 2: Display the mismatched question and answers.
display(
    incorrect_pilot_df[
        [
            "question",
            "gold_answer",
            "generated_answer",
            "target_passage_id"
        ]
    ]
)

# Step 3: Print the full evidence passage for each mismatch.
for _, row in incorrect_pilot_df.iterrows():
    target_id = str(row["target_passage_id"])
    evidence_text = summary_text_lookup.get(target_id, "")

    print("\nTarget passage:", target_id)
    print("-" * 60)
    print(evidence_text)

Incorrect pilot answers: 1


,question,gold_answer,generated_answer,target_passage_id
1,Which team won the IPL match between Delhi Dar...,Kolkata Knight Riders,Delhi Daredevils,501230_summary



Target passage: 501230_summary
------------------------------------------------------------
IPL match ID: 501230. Date: 2011-04-28. Season: 2011. Teams: Delhi Daredevils, Kolkata Knight Riders. Venue: Feroz Shah Kotla. City: Delhi. Winner: Kolkata Knight Riders. Toss winner: Delhi Daredevils. Toss decision: field. Player of the match: MK Tiwary. Kolkata Knight Riders scored 148 runs and lost 7 wickets in innings 1. Delhi Daredevils scored 131 runs and lost 9 wickets in innings 2.


In [39]:
# STEP 35: TEST EXPLICIT WINNER DISAMBIGUATION
# Purpose:
# 1. Reuse the verified evidence passage from the pilot error.
# 2. Explicitly distinguish match winner from toss winner.
# 3. Check whether clearer instructions change the answer.
# 4. Preserve the original answer for comparison.

# Step 1: Select the previously incorrect pilot example.
error_row = incorrect_pilot_df.iloc[0]
target_id = str(error_row["target_passage_id"])
evidence_text = summary_text_lookup[target_id]
question_text = str(error_row["question"])

# Step 2: Build a prompt that explicitly distinguishes winner fields.
clarified_prompt = (
    "Answer the question using only the evidence below.\n"
    "Important: 'Winner' means the team that won the match. "
    "'Toss winner' means the team that won the coin toss. "
    "These are different fields. Do not confuse them.\n\n"
    f"Evidence: {evidence_text}\n\n"
    f"Question: {question_text}\n"
    "Answer with only the winning team's name:"
)

# Step 3: Generate a new answer with deterministic decoding.
clarified_inputs = tokenizer(
    clarified_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    clarified_output_ids = model.generate(
        **clarified_inputs,
        max_new_tokens=32,
        do_sample=False
    )

clarified_answer = tokenizer.decode(
    clarified_output_ids[0],
    skip_special_tokens=True
).strip()

# Step 4: Compare the original and clarified answers.
print("Question:", question_text)
print("Gold answer:", error_row["gold_answer"])
print("Original answer:", error_row["generated_answer"])
print("Clarified-prompt answer:", clarified_answer)

Question: Which team won the IPL match between Delhi Daredevils and Kolkata Knight Riders played on 2011-04-28?
Gold answer: Kolkata Knight Riders
Original answer: Delhi Daredevils
Clarified-prompt answer: Delhi Daredevils


In [40]:
# STEP 36: VERIFY EVIDENCE AVAILABILITY FOR THE TEST SET
# Purpose:
# 1. Confirm that each test question has its intended evidence passage.
# 2. Check that the evidence is non-empty.
# 3. Summarize evidence availability by question type.
# 4. Establish a consistent starting point for the controlled comparison.

# Step 1: Define the existing passage stores.
summary_text_lookup = {
    str(row["passage_id"]): str(row["text"])
    for _, row in match_summary_df.iterrows()
}

player_text_lookup = {
    str(row["passage_id"]): str(row["text"])
    for _, row in player_stat_df.iterrows()
}

# Step 2: Select the held-out test questions.
test_questions_df = (
    qa_df[qa_df["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

# Step 3: Retrieve the intended evidence for each question.
def get_target_evidence(row):
    target_id = str(row["target_passage_id"])

    if row["question_type"] == "batter_runs":
        return player_text_lookup.get(target_id, "")

    return summary_text_lookup.get(target_id, "")

test_questions_df["target_evidence"] = test_questions_df.apply(
    get_target_evidence, axis=1
)

test_questions_df["evidence_available"] = (
    test_questions_df["target_evidence"]
    .fillna("")
    .str.strip()
    .ne("")
)

# Step 4: Summarize evidence availability by question type.
evidence_check_df = (
    test_questions_df
    .groupby("question_type")
    .agg(
        questions=("question_id", "count"),
        evidence_available=("evidence_available", "sum")
    )
    .reset_index()
)

evidence_check_df["missing_evidence"] = (
    evidence_check_df["questions"]
    - evidence_check_df["evidence_available"]
)

print("Total test questions:", len(test_questions_df))
display(evidence_check_df)

# Step 5: Confirm that the test set has no missing target evidence.
print(
    "Total questions with missing evidence:",
    int((~test_questions_df["evidence_available"]).sum())
)

Total test questions: 4545


,question_type,questions,evidence_available,missing_evidence
0,batter_runs,3797,3797,0
1,innings_runs,505,505,0
2,match_winner,243,243,0


Total questions with missing evidence: 0


In [41]:
# STEP 37: COMPARE NO-RETRIEVAL, FIXED BM25, AND ORACLE EVIDENCE
# Purpose:
# 1. Use the same five held-out questions for all three conditions.
# 2. Keep the model and answer instructions consistent.
# 3. Compare answers with the gold labels.
# 4. Treat oracle evidence as a diagnostic upper bound, not a retrieval method.

# Step 1: Select five pilot questions from the held-out test set.
pilot_compare_df = (
    test_questions_df.groupby("question_type", group_keys=False)
    .head(2)
    .head(5)
    .copy()
    .reset_index(drop=True)
)

# Step 2: Define a function to retrieve the top BM25 passage.
def retrieve_top_bm25(row):
    # Step 2a: Select the collection appropriate for this question type.
    collection_name = (
        "player_stat"
        if row["question_type"] == "batter_runs"
        else "match_summary"
    )

    index = bm25_indexes[("test", collection_name)]
    passage_store = bm25_passage_stores[("test", collection_name)]

    # Step 2b: Rank the passages using the question.
    scores = index.get_scores(tokenize_text(row["question"]))
    top_index = int(scores.argmax())

    return str(passage_store.iloc[top_index]["text"])

# Step 3: Generate an answer under a specified evidence condition.
def generate_answer(question, evidence=None):
    # Step 3a: Use the same instruction style across conditions.
    if evidence is None:
        prompt = (
            "Answer the question as accurately as possible. "
            "If you do not know, say 'Not sure'.\n\n"
            f"Question: {question}\nAnswer:"
        )
    else:
        prompt = (
            "Answer the question using only the supplied evidence. "
            "Return a short answer. Do not guess. "
            "If the evidence does not contain the answer, say 'Not stated'.\n\n"
            f"Evidence: {evidence}\n\n"
            f"Question: {question}\nAnswer:"
        )

    # Step 3b: Tokenize the prompt and generate deterministically.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()

# Step 4: Evaluate each question under all three conditions.
comparison_records = []

for _, row in pilot_compare_df.iterrows():
    # Step 4a: Generate the three answers.
    fixed_evidence = retrieve_top_bm25(row)
    oracle_evidence = row["target_evidence"]

    no_retrieval_answer = generate_answer(row["question"])
    fixed_answer = generate_answer(row["question"], fixed_evidence)
    oracle_answer = generate_answer(row["question"], oracle_evidence)

    # Step 4b: Store answers without overwriting earlier experiments.
    comparison_records.append({
        "question_type": row["question_type"],
        "question": row["question"],
        "gold_answer": str(row["gold_answer"]),
        "no_retrieval_answer": no_retrieval_answer,
        "fixed_bm25_answer": fixed_answer,
        "oracle_evidence_answer": oracle_answer,
        "fixed_passage_matches_target": (
            str(bm25_passage_stores[
                (
                    "test",
                    "player_stat" if row["question_type"] == "batter_runs"
                    else "match_summary"
                )
            ].iloc[
                int(
                    bm25_indexes[
                        (
                            "test",
                            "player_stat" if row["question_type"] == "batter_runs"
                            else "match_summary"
                        )
                    ].get_scores(
                        tokenize_text(row["question"])
                    ).argmax()
                )]["passage_id"]) == str(row["target_passage_id"])
        )
    })

# Step 5: Display the pilot comparison.
comparison_pilot_df = pd.DataFrame(comparison_records)
display(comparison_pilot_df)

,question_type,question,gold_answer,no_retrieval_answer,fixed_bm25_answer,oracle_evidence_answer,fixed_passage_matches_target
0,match_winner,Which team won the IPL match between Kings XI ...,Royal Challengers Bangalore,Not sure,Royal Challengers Bangalore,Royal Challengers Bangalore,True
1,match_winner,Which team won the IPL match between Delhi Dar...,Kolkata Knight Riders,Kolkata Knight Riders,Delhi Daredevils,Delhi Daredevils,True
2,innings_runs,How many runs did Kings XI Punjab score in inn...,163,2,163,163,True
3,innings_runs,How many runs did Royal Challengers Bangalore ...,166,0,166,166,True
4,batter_runs,How many runs did N Saini score for Kings XI P...,14,2,14,14,True


In [42]:
# STEP 38: DEFINE ANSWER NORMALIZATION AND CHECK THE PILOT
# Purpose:
# 1. Normalize generated answers for fair comparison.
# 2. Compare numeric answers by their numeric value.
# 3. Compare team names using normalized exact matching.
# 4. Keep answer accuracy separate from retrieval accuracy.

# Step 1: Import numeric parsing utilities.
import re

# Step 2: Normalize text by lowercasing and removing punctuation.
def normalize_answer_text(answer):
    # Step 2a: Standardize whitespace and punctuation.
    answer = str(answer).lower().strip()
    answer = re.sub(r"[^a-z0-9\s]", " ", answer)
    answer = re.sub(r"\s+", " ", answer)
    return answer

# Step 3: Compare a generated answer with its gold answer.
def answer_is_correct(question_type, generated_answer, gold_answer):
    generated = str(generated_answer).strip()
    gold = str(gold_answer).strip()

    # Step 3a: For run totals, compare the first integer found.
    if question_type in {"innings_runs", "batter_runs"}:
        generated_numbers = re.findall(r"\b\d+\b", generated)
        gold_numbers = re.findall(r"\b\d+\b", gold)

        if not generated_numbers or not gold_numbers:
            return False

        return int(generated_numbers[0]) == int(gold_numbers[0])

    # Step 3b: For match winners, require normalized exact team-name matching.
    return normalize_answer_text(generated) == normalize_answer_text(gold)

# Step 4: Evaluate the pilot answers under all three conditions.
pilot_accuracy_records = []

for _, row in comparison_pilot_df.iterrows():
    for condition, answer_column in [
        ("no_retrieval", "no_retrieval_answer"),
        ("fixed_bm25", "fixed_bm25_answer"),
        ("oracle_evidence", "oracle_evidence_answer")
    ]:
        generated = row[answer_column]

        pilot_accuracy_records.append({
            "question_type": row["question_type"],
            "condition": condition,
            "gold_answer": row["gold_answer"],
            "generated_answer": generated,
            "correct": int(
                answer_is_correct(
                    row["question_type"],
                    generated,
                    row["gold_answer"]
                )
            )
        })

# Step 5: Summarize pilot accuracy.
pilot_accuracy_df = pd.DataFrame(pilot_accuracy_records)

pilot_accuracy_summary = (
    pilot_accuracy_df
    .groupby("condition")
    .agg(
        questions=("correct", "count"),
        correct_answers=("correct", "sum"),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

display(pilot_accuracy_summary)

,condition,questions,correct_answers,accuracy
0,fixed_bm25,5,4,0.8
1,no_retrieval,5,1,0.2
2,oracle_evidence,5,4,0.8


In [43]:
# STEP 39: CHECK QUESTION ENTITY EXTRACTION
# Purpose:
# 1. Extract dates and team/player names from test questions.
# 2. Check whether those entities appear in the target evidence.
# 3. Establish the fields needed for an interpretable adaptive rule.

# Step 1: Extract a date from a question.
def extract_question_date(question):
    match = re.search(r"\b\d{4}-\d{2}-\d{2}\b", str(question))
    return match.group(0) if match else None

# Step 2: Extract the team from an innings-run question.
def extract_batting_team(question):
    match = re.search(
        r"did (.+?) score in innings",
        str(question),
        flags=re.IGNORECASE
    )
    return match.group(1).strip() if match else None

# Step 3: Extract the player from a batter-run question.
def extract_batter_name(question):
    match = re.search(
        r"How many runs did (.+?) score for ",
        str(question),
        flags=re.IGNORECASE
    )
    return match.group(1).strip() if match else None

# Step 4: Preview extracted entities from each question type.
for question_type in ["match_winner", "innings_runs", "batter_runs"]:
    examples = test_questions_df[
        test_questions_df["question_type"] == question_type
    ].head(2)

    print(f"\nQuestion type: {question_type}")

    for _, row in examples.iterrows():
        print("Question:", row["question"])
        print("Date:", extract_question_date(row["question"]))
        print("Batting team:", extract_batting_team(row["question"]))
        print("Batter:", extract_batter_name(row["question"]))


Question type: match_winner
Question: Which team won the IPL match between Kings XI Punjab and Royal Challengers Bangalore played on 2012-04-20?
Date: 2012-04-20
Batting team: None
Batter: None
Question: Which team won the IPL match between Delhi Daredevils and Kolkata Knight Riders played on 2011-04-28?
Date: 2011-04-28
Batting team: None
Batter: None

Question type: innings_runs
Question: How many runs did Kings XI Punjab score in innings 1 of the IPL match played on 2012-04-20?
Date: 2012-04-20
Batting team: Kings XI Punjab
Batter: None
Question: How many runs did Royal Challengers Bangalore score in innings 2 of the IPL match played on 2012-04-20?
Date: 2012-04-20
Batting team: Royal Challengers Bangalore
Batter: None

Question type: batter_runs
Question: How many runs did N Saini score for Kings XI Punjab in innings 1 of the IPL match played on 2012-04-20?
Date: 2012-04-20
Batting team: None
Batter: N Saini
Question: How many runs did PC Valthaty score for Kings XI Punjab in inni

In [44]:
# STEP 40: TEST THE ADAPTIVE RETRIEVAL DECISION RULE
# Purpose:
# 1. Retrieve the top BM25 passage for each pilot question.
# 2. Check whether it contains the date and relevant entities.
# 3. Use top-1 evidence when the checks pass.
# 4. Fall back to top-3 evidence when the checks fail.
# 5. Record decisions without generating answers yet.

# Step 1: Select five held-out pilot questions.
adaptive_pilot_df = (
    test_questions_df.groupby("question_type", group_keys=False)
    .head(2)
    .head(5)
    .copy()
    .reset_index(drop=True)
)

# Step 2: Define which evidence collection applies to each question type.
def get_collection_name(question_type):
    return "player_stat" if question_type == "batter_runs" else "match_summary"

# Step 3: Retrieve and rank the top three passages.
def retrieve_top_three(row):
    collection_name = get_collection_name(row["question_type"])
    index = bm25_indexes[("test", collection_name)]
    passage_store = bm25_passage_stores[("test", collection_name)]

    scores = index.get_scores(tokenize_text(row["question"]))
    top_indices = scores.argsort()[::-1][:3]

    return passage_store.iloc[top_indices].copy()

# Step 4: Check whether the top passage contains the identifying entities.
def check_passage_entities(row, passage_text):
    text_lower = str(passage_text).lower()
    question_date = extract_question_date(row["question"])

    # Step 4a: Check the date.
    date_ok = bool(question_date and question_date in text_lower)

    # Step 4b: Check the relevant team or player.
    if row["question_type"] == "innings_runs":
        entity = extract_batting_team(row["question"])
        entity_ok = bool(entity and entity.lower() in text_lower)

    elif row["question_type"] == "batter_runs":
        entity = extract_batter_name(row["question"])
        entity_ok = bool(entity and entity.lower() in text_lower)

    else:
        # Step 4c: For match-winner questions, both teams must appear.
        teams_match = re.search(
            r"between (.+?) and (.+?) played on",
            str(row["question"]),
            flags=re.IGNORECASE
        )

        if teams_match:
            team_a = teams_match.group(1).strip().lower()
            team_b = teams_match.group(2).strip().lower()
            entity_ok = team_a in text_lower and team_b in text_lower
        else:
            entity_ok = False

    return date_ok, entity_ok

# Step 5: Apply the adaptive decision rule.
adaptive_decisions = []

for _, row in adaptive_pilot_df.iterrows():
    ranked_passages = retrieve_top_three(row)
    top1 = ranked_passages.iloc[0]
    top1_text = str(top1["text"])

    date_ok, entity_ok = check_passage_entities(row, top1_text)

    # Step 5a: Fall back to top-3 if either check fails.
    use_top3 = not (date_ok and entity_ok)

    selected_passages = ranked_passages if use_top3 else ranked_passages.head(1)

    adaptive_decisions.append({
        "question_type": row["question_type"],
        "question": row["question"],
        "target_passage_id": str(row["target_passage_id"]),
        "top1_passage_id": str(top1["passage_id"]),
        "date_check_passed": date_ok,
        "entity_check_passed": entity_ok,
        "adaptive_action": "use_top3" if use_top3 else "use_top1",
        "selected_passage_ids": " | ".join(
            selected_passages["passage_id"].astype(str).tolist()
        ),
        "target_in_selected_evidence": (
            str(row["target_passage_id"])
            in selected_passages["passage_id"].astype(str).tolist()
        )
    })

# Step 6: Display the decisions.
adaptive_decisions_df = pd.DataFrame(adaptive_decisions)
display(adaptive_decisions_df)

print("\nAdaptive actions:")
print(adaptive_decisions_df["adaptive_action"].value_counts())

print(
    "\nTarget evidence included:",
    int(adaptive_decisions_df["target_in_selected_evidence"].sum()),
    "of",
    len(adaptive_decisions_df)
)

,question_type,question,target_passage_id,top1_passage_id,date_check_passed,entity_check_passed,adaptive_action,selected_passage_ids,target_in_selected_evidence
0,match_winner,Which team won the IPL match between Kings XI ...,548331_summary,548331_summary,True,True,use_top1,548331_summary,True
1,match_winner,Which team won the IPL match between Delhi Dar...,501230_summary,501230_summary,True,True,use_top1,501230_summary,True
2,innings_runs,How many runs did Kings XI Punjab score in inn...,548331_summary,548331_summary,True,True,use_top1,548331_summary,True
3,innings_runs,How many runs did Royal Challengers Bangalore ...,548331_summary,548331_summary,True,True,use_top1,548331_summary,True
4,batter_runs,How many runs did N Saini score for Kings XI P...,548331_innings1_N Saini,548331_innings1_N Saini,True,True,use_top1,548331_innings1_N Saini,True



Adaptive actions:
adaptive_action
use_top1    5
Name: count, dtype: int64

Target evidence included: 5 of 5


In [45]:
# STEP 41: EVALUATE ADAPTIVE RETRIEVAL DECISIONS ON THE FULL TEST SET
# Purpose:
# 1. Apply the same entity-checking rule to all held-out test questions.
# 2. Measure how often the rule chooses top-1 versus top-3.
# 3. Measure whether selected passages contain the target passage.
# 4. Save question-level decisions for reproducibility.

# Step 1: Prepare a list for all test-set decisions.
adaptive_full_records = []

# Step 2: Evaluate every held-out question.
for _, row in test_questions_df.iterrows():
    ranked_passages = retrieve_top_three(row)
    top1 = ranked_passages.iloc[0]

    # Step 2a: Check identifying details in the top-ranked passage.
    date_ok, entity_ok = check_passage_entities(row, str(top1["text"]))

    # Step 2b: Select top-1 when both checks pass; otherwise use top-3.
    use_top3 = not (date_ok and entity_ok)

    selected_passages = (
        ranked_passages if use_top3 else ranked_passages.head(1)
    )

    selected_ids = selected_passages["passage_id"].astype(str).tolist()
    target_id = str(row["target_passage_id"])

    # Step 2c: Record the decision and retrieval outcome.
    adaptive_full_records.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "target_passage_id": target_id,
        "top1_passage_id": str(top1["passage_id"]),
        "date_check_passed": date_ok,
        "entity_check_passed": entity_ok,
        "adaptive_action": "use_top3" if use_top3 else "use_top1",
        "selected_passage_ids": " | ".join(selected_ids),
        "target_in_selected_evidence": int(target_id in selected_ids)
    })

# Step 3: Store the results in a DataFrame.
adaptive_full_df = pd.DataFrame(adaptive_full_records)

# Step 4: Summarize decisions and target-evidence coverage.
print("Test questions evaluated:", len(adaptive_full_df))

print("\nAdaptive actions:")
display(
    adaptive_full_df["adaptive_action"]
    .value_counts()
    .rename_axis("adaptive_action")
    .reset_index(name="question_count")
)

print(
    "\nTarget evidence included:",
    int(adaptive_full_df["target_in_selected_evidence"].sum()),
    "of",
    len(adaptive_full_df)
)

# Step 5: Summarize outcomes by question type.
adaptive_type_summary = (
    adaptive_full_df
    .groupby("question_type")
    .agg(
        questions=("question_id", "count"),
        fallback_count=(
            "adaptive_action",
            lambda s: int((s == "use_top3").sum())
        ),
        target_evidence_count=("target_in_selected_evidence", "sum")
    )
    .reset_index()
)

display(adaptive_type_summary)

# Step 6: Save the complete decision log.
adaptive_full_df.to_csv(
    "/content/ipl_adaptive_retrieval_decisions.csv",
    index=False
)

print("Saved: /content/ipl_adaptive_retrieval_decisions.csv")

Test questions evaluated: 4545

Adaptive actions:


,adaptive_action,question_count
0,use_top1,4497
1,use_top3,48



Target evidence included: 4540 of 4545


,question_type,questions,fallback_count,target_evidence_count
0,batter_runs,3797,26,3796
1,innings_runs,505,22,501
2,match_winner,243,0,243


Saved: /content/ipl_adaptive_retrieval_decisions.csv


In [46]:
# STEP 42: INSPECT QUESTIONS WHERE ADAPTIVE RETRIEVAL MISSED THE TARGET
# Purpose:
# 1. Identify all five questions where the target passage was not selected.
# 2. Check whether the top-1 passage passed the entity checks.
# 3. Compare target and retrieved passage IDs.
# 4. Diagnose the failure before changing the adaptive rule.

# Step 1: Select the questions where the target passage was not included.
adaptive_misses_df = adaptive_full_df[
    adaptive_full_df["target_in_selected_evidence"] == 0
].copy()

print("Target passages missed:", len(adaptive_misses_df))

# Step 2: Display the retrieval decisions for these questions.
display(
    adaptive_misses_df[
        [
            "question_id",
            "question_type",
            "target_passage_id",
            "top1_passage_id",
            "date_check_passed",
            "entity_check_passed",
            "adaptive_action",
            "selected_passage_ids"
        ]
    ]
)

# Step 3: Join the failed question IDs to the original question text.
miss_details_df = adaptive_misses_df.merge(
    test_questions_df[
        ["question_id", "question", "gold_answer"]
    ],
    on="question_id",
    how="left"
)

# Step 4: Display the questions and gold answers for diagnosis.
display(
    miss_details_df[
        [
            "question_type",
            "question",
            "gold_answer",
            "target_passage_id",
            "top1_passage_id",
            "adaptive_action"
        ]
    ]
)

Target passages missed: 5


,question_id,question_type,target_passage_id,top1_passage_id,date_check_passed,entity_check_passed,adaptive_action,selected_passage_ids
286,1359482_innings2_runs,innings_runs,1359482_summary,1359526_summary,False,True,use_top3,1359526_summary | 1359530_summary | 1359540_su...
452,548352_innings2_runs,innings_runs,548352_summary,548341_summary,False,True,use_top3,548341_summary | 548326_summary | 548311_summary
608,1527681_innings1_runs,innings_runs,1527681_summary,1529312_summary,False,True,use_top3,1529312_summary | 1529273_summary | 1529267_su...
654,1422136_innings1_runs,innings_runs,1422136_summary,1422125_summary,False,True,use_top3,1422125_summary | 1426277_summary | 1422119_su...
1623,1175365_innings1_KD Karthik_runs,batter_runs,1175365_innings1_KD Karthik,1175365_innings4_KD Karthik,True,True,use_top1,1175365_innings4_KD Karthik


,question_type,question,gold_answer,target_passage_id,top1_passage_id,adaptive_action
0,innings_runs,How many runs did Rajasthan Royals score in in...,192,1359482_summary,1359526_summary,use_top3
1,innings_runs,How many runs did Deccan Chargers score in inn...,150,548352_summary,548341_summary,use_top3
2,innings_runs,How many runs did Mumbai Indians score in inni...,162,1527681_summary,1529312_summary,use_top3
3,innings_runs,How many runs did Chennai Super Kings score in...,165,1422136_summary,1422125_summary,use_top3
4,batter_runs,How many runs did KD Karthik score for Kolkata...,50,1175365_innings1_KD Karthik,1175365_innings4_KD Karthik,use_top1


In [47]:
# STEP 43: ADD AN INNINGS-NUMBER CHECK FOR BATTER QUESTIONS
# Purpose:
# 1. Extract the requested innings number from each batter-run question.
# 2. Extract the innings number from the retrieved passage ID.
# 3. Check whether the retrieved passage matches the requested innings.
# 4. Identify failures the previous entity rule could not detect.

# Step 1: Extract the innings number from a question.
def extract_innings_number(question):
    match = re.search(
        r"\binnings\s+(\d+)\b",
        str(question),
        flags=re.IGNORECASE
    )
    return int(match.group(1)) if match else None

# Step 2: Extract the innings number from a player passage ID.
def extract_passage_innings(passage_id):
    match = re.search(
        r"_innings(\d+)_",
        str(passage_id)
    )
    return int(match.group(1)) if match else None

# Step 3: Apply the check to the full adaptive evaluation.
adaptive_full_df["requested_innings"] = (
    adaptive_full_df["question_id"]
    .map(
        test_questions_df.set_index("question_id")["question"]
    )
    .apply(extract_innings_number)
)

adaptive_full_df["retrieved_innings"] = (
    adaptive_full_df["top1_passage_id"]
    .apply(extract_passage_innings)
)

# Step 4: Check innings consistency for batter-run questions.
adaptive_full_df["innings_check_passed"] = True

batter_mask = adaptive_full_df["question_type"] == "batter_runs"

adaptive_full_df.loc[batter_mask, "innings_check_passed"] = (
    adaptive_full_df.loc[batter_mask, "requested_innings"]
    == adaptive_full_df.loc[batter_mask, "retrieved_innings"]
)

# Step 5: Report the number of batter questions with mismatched innings.
innings_mismatches_df = adaptive_full_df[
    batter_mask &
    (~adaptive_full_df["innings_check_passed"])
].copy()

print(
    "Batter-run questions with a retrieved innings mismatch:",
    len(innings_mismatches_df)
)

# Step 6: Display the mismatches.
display(
    innings_mismatches_df[
        [
            "question_id",
            "requested_innings",
            "retrieved_innings",
            "target_passage_id",
            "top1_passage_id",
            "adaptive_action"
        ]
    ].head(10)
)

Batter-run questions with a retrieved innings mismatch: 1


,question_id,requested_innings,retrieved_innings,target_passage_id,top1_passage_id,adaptive_action
1623,1175365_innings1_KD Karthik_runs,1.0,4.0,1175365_innings1_KD Karthik,1175365_innings4_KD Karthik,use_top1


In [48]:
# STEP 44: MEASURE THE EFFECT OF THE NEW INNINGS CHECK
# Purpose:
# 1. Preserve the original adaptive decisions.
# 2. Identify questions that fail the new innings check.
# 3. Measure how many decisions would need reconsideration.

# Step 1: Count questions flagged by the innings check.
print(
    "Questions failing the innings check:",
    int((~adaptive_full_df["innings_check_passed"]).sum())
)

# Step 2: Count flagged questions by question type.
print("\nFlagged questions by type:")

display(
    adaptive_full_df[
        ~adaptive_full_df["innings_check_passed"]
    ]["question_type"]
    .value_counts()
    .rename_axis("question_type")
    .reset_index(name="count")
)

# Step 3: Display the original and revised-check information.
display(
    adaptive_full_df[
        ~adaptive_full_df["innings_check_passed"]
    ][
        [
            "question_id",
            "question_type",
            "requested_innings",
            "retrieved_innings",
            "adaptive_action"
        ]
    ]
)

Questions failing the innings check: 1

Flagged questions by type:


,question_type,count
0,batter_runs,1


,question_id,question_type,requested_innings,retrieved_innings,adaptive_action
1623,1175365_innings1_KD Karthik_runs,batter_runs,1.0,4.0,use_top1


In [49]:
# STEP 45: INSPECT TOP-3 CANDIDATES FOR THE INNINGS MISMATCH
# Purpose:
# 1. Inspect the question flagged by our new check.
# 2. Review the top three BM25 candidates.
# 3. Determine whether a candidate matches the requested innings.
# 4. Avoid changing the adaptive rule before examining the evidence.

# Step 1: Get the flagged question.
flagged_question_id = "1175365_innings1_KD Karthik_runs"

flagged_row = test_questions_df[
    test_questions_df["question_id"] == flagged_question_id
].iloc[0]

print("Question:", flagged_row["question"])
print("Gold answer (evaluation only):", flagged_row["gold_answer"])
print("Target passage ID (evaluation only):", flagged_row["target_passage_id"])

# Step 2: Retrieve the top three candidates using the existing BM25 function.
candidate_passages_df = retrieve_top_three(flagged_row)

# Step 3: Display the candidates and check their innings numbers.
candidate_passages_df = candidate_passages_df.copy()

candidate_passages_df["candidate_innings"] = (
    candidate_passages_df["passage_id"]
    .apply(extract_passage_innings)
)

candidate_passages_df["requested_innings"] = extract_innings_number(
    flagged_row["question"]
)

candidate_passages_df["innings_matches"] = (
    candidate_passages_df["candidate_innings"]
    == candidate_passages_df["requested_innings"]
)

display(
    candidate_passages_df[
        [
            "passage_id",
            "candidate_innings",
            "requested_innings",
            "innings_matches",
            "text"
        ]
    ]
)

Question: How many runs did KD Karthik score for Kolkata Knight Riders in innings 1 of the IPL match played on 2019-03-30?
Gold answer (evaluation only): 50
Target passage ID (evaluation only): 1175365_innings1_KD Karthik


,passage_id,candidate_innings,requested_innings,innings_matches,text
890,1175365_innings4_KD Karthik,4,1,False,IPL match ID: 1175365. Date: 2019-03-30. Seaso...
875,1175365_innings1_KD Karthik,1,1,True,IPL match ID: 1175365. Date: 2019-03-30. Seaso...
434,1178427_innings2_KD Karthik,2,1,False,IPL match ID: 1178427. Date: 2019-05-03. Seaso...


In [50]:
# STEP 46: TEST AN INNINGS-AWARE RETRIEVAL FALLBACK
# Purpose:
# 1. Examine candidates in BM25 ranking order.
# 2. Select the first candidate matching the requested innings.
# 3. Also require the requested date and player to match.
# 4. Keep the gold answer and target passage ID out of selection.

# Step 1: Extract the requested details from the question.
requested_date = extract_question_date(flagged_row["question"])
requested_innings = extract_innings_number(flagged_row["question"])
requested_batter = extract_batter_name(flagged_row["question"])

# Step 2: Check candidates in their existing BM25 order.
eligible_candidates = []

for _, candidate in candidate_passages_df.iterrows():
    passage_text = str(candidate["text"])
    passage_id = str(candidate["passage_id"])

    date_matches = requested_date in passage_text
    batter_matches = (
        requested_batter is not None
        and requested_batter.lower() in passage_text.lower()
    )
    innings_matches = (
        extract_passage_innings(passage_id) == requested_innings
    )

    if date_matches and batter_matches and innings_matches:
        eligible_candidates.append({
            "passage_id": passage_id,
            "date_matches": date_matches,
            "batter_matches": batter_matches,
            "innings_matches": innings_matches,
            "text": passage_text
        })

# Step 3: Report the first eligible candidate, if one exists.
if eligible_candidates:
    selected_candidate = eligible_candidates[0]
    print("Selected passage:", selected_candidate["passage_id"])
    print("Date matches:", selected_candidate["date_matches"])
    print("Batter matches:", selected_candidate["batter_matches"])
    print("Innings matches:", selected_candidate["innings_matches"])
    print("Selected evidence:", selected_candidate["text"])
else:
    print("No eligible candidate found in the top three.")

Selected passage: 1175365_innings1_KD Karthik
Date matches: True
Batter matches: True
Innings matches: True
Selected evidence: IPL match ID: 1175365. Date: 2019-03-30. Season: 2019. Innings: 1. Batting team: Kolkata Knight Riders. Player: KD Karthik. Runs scored by KD Karthik: 50.


In [51]:
# STEP 47: EVALUATE THE INNINGS-AWARE FALLBACK ON THE TEST SET
# Purpose:
# 1. Reuse the existing top-three BM25 candidates.
# 2. Apply date, player, and innings checks to batter questions.
# 3. Keep the original adaptive results unchanged.
# 4. Measure target-passage retrieval using gold labels only after selection.

# Step 1: Create a separate results list.
revised_records = []

# Step 2: Process every held-out test question.
for _, row in test_questions_df.iterrows():
    candidates = retrieve_top_three(row)
    question_type = row["question_type"]
    question = str(row["question"])

    requested_date = extract_question_date(question)
    requested_innings = extract_innings_number(question)
    requested_batter = extract_batter_name(question)

    selected_id = None
    selection_reason = "no_candidate_passed_checks"

    # Step 3: Examine candidates in BM25 ranking order.
    for _, candidate in candidates.iterrows():
        passage_id = str(candidate["passage_id"])
        passage_text = str(candidate["text"])

        if question_type == "batter_runs":
            valid = (
                requested_date is not None
                and requested_date in passage_text
                and requested_batter is not None
                and requested_batter.lower() in passage_text.lower()
                and extract_passage_innings(passage_id) == requested_innings
            )
        else:
            # Preserve the existing top-1 choice for other question types.
            valid = passage_id == str(candidates.iloc[0]["passage_id"])

        if valid:
            selected_id = passage_id
            selection_reason = (
                "innings_aware_candidate"
                if question_type == "batter_runs"
                else "original_top1"
            )
            break

    # Step 4: Record the selected passage without using the gold target
    # to influence retrieval.
    target_id = str(row["target_passage_id"])

    revised_records.append({
        "question_id": row["question_id"],
        "question_type": question_type,
        "selected_passage_id": selected_id,
        "selection_reason": selection_reason,
        "candidate_found": selected_id is not None,
        "target_passage_retrieved": (
            selected_id == target_id if selected_id is not None else False
        )
    })

# Step 5: Summarise results after retrieval decisions are complete.
revised_retrieval_df = pd.DataFrame(revised_records)

print("Test questions:", len(revised_retrieval_df))
print(
    "Questions with a selected passage:",
    int(revised_retrieval_df["candidate_found"].sum())
)
print(
    "Questions whose exact target passage was retrieved:",
    int(revised_retrieval_df["target_passage_retrieved"].sum())
)

# Step 6: Report results by question type.
revised_summary_df = (
    revised_retrieval_df
    .groupby("question_type")
    .agg(
        questions=("question_id", "count"),
        selected=("candidate_found", "sum"),
        exact_target_retrieved=("target_passage_retrieved", "sum")
    )
    .reset_index()
)

revised_summary_df["exact_target_recall"] = (
    revised_summary_df["exact_target_retrieved"]
    / revised_summary_df["questions"]
)

display(revised_summary_df)

Test questions: 4545
Questions with a selected passage: 4545
Questions whose exact target passage was retrieved: 4523


,question_type,questions,selected,exact_target_retrieved,exact_target_recall
0,batter_runs,3797,3797,3797,1.000000
1,innings_runs,505,505,483,0.956436
2,match_winner,243,243,243,1.000000


In [52]:
# STEP 48: INSPECT REMAINING INNINGS-TOTAL RETRIEVAL FAILURES
# Purpose:
# 1. Find innings-run questions where the selected passage was incorrect.
# 2. Check whether the correct passage appears in BM25's top three.
# 3. Keep target IDs for evaluation only, not for retrieval decisions.

# Step 1: Identify innings-run questions that missed the exact target.
innings_misses_df = revised_retrieval_df[
    (revised_retrieval_df["question_type"] == "innings_runs")
    & (~revised_retrieval_df["target_passage_retrieved"])
].copy()

print("Innings-run questions missed:", len(innings_misses_df))

# Step 2: Inspect the top three candidates for each missed question.
miss_inspection_records = []

for question_id in innings_misses_df["question_id"]:
    row = test_questions_df[
        test_questions_df["question_id"] == question_id
    ].iloc[0]

    candidates = retrieve_top_three(row)
    target_id = str(row["target_passage_id"])
    candidate_ids = candidates["passage_id"].astype(str).tolist()

    miss_inspection_records.append({
        "question_id": question_id,
        "question": row["question"],
        "gold_answer": row["gold_answer"],
        "target_passage_id": target_id,
        "top1_passage_id": candidate_ids[0] if len(candidate_ids) > 0 else None,
        "top2_passage_id": candidate_ids[1] if len(candidate_ids) > 1 else None,
        "top3_passage_id": candidate_ids[2] if len(candidate_ids) > 2 else None,
        "target_in_top3": target_id in candidate_ids
    })

miss_inspection_df = pd.DataFrame(miss_inspection_records)

# Step 3: Summarise how many targets are present in the top three.
print(
    "Missed questions where the target is in top three:",
    int(miss_inspection_df["target_in_top3"].sum()),
    "out of",
    len(miss_inspection_df)
)

# Step 4: Display the cases for diagnosis.
display(miss_inspection_df)

Innings-run questions missed: 22
Missed questions where the target is in top three: 18 out of 22


,question_id,question,gold_answer,target_passage_id,top1_passage_id,top2_passage_id,top3_passage_id,target_in_top3
0,392236_innings2_runs,How many runs did Deccan Chargers score in inn...,158,392236_summary,392237_summary,392212_summary,392236_summary,True
1,1473455_innings1_runs,How many runs did Rajasthan Royals score in in...,205,1473455_summary,1473487_summary,1473455_summary,1473465_summary,True
2,1473455_innings2_runs,How many runs did Punjab Kings score in inning...,155,1473455_summary,1473481_summary,1473510_summary,1473455_summary,True
3,1359482_innings2_runs,How many runs did Rajasthan Royals score in in...,192,1359482_summary,1359526_summary,1359530_summary,1359540_summary,False
4,501242_innings1_runs,How many runs did Kochi Tuskers Kerala score i...,156,501242_summary,501258_summary,501215_summary,501242_summary,True
5,980919_innings1_runs,How many runs did Rising Pune Supergiants scor...,152,980919_summary,980997_summary,980919_summary,980943_summary,True
6,1359484_innings1_runs,How many runs did Sunrisers Hyderabad score in...,121,1359484_summary,1359526_summary,1359484_summary,1359478_summary,True
7,1178431_innings1_runs,How many runs did Kolkata Knight Riders score ...,133,1178431_summary,1175372_summary,1178431_summary,1178427_summary,True
8,1216523_innings2_runs,How many runs did Kings XI Punjab score in inn...,162,1216523_summary,1216531_summary,1216546_summary,1216523_summary,True
9,598046_innings2_runs,How many runs did Chennai Super Kings score in...,79,598046_summary,598070_summary,598068_summary,598046_summary,True


In [53]:
# STEP 49: TEST DATE-AND-TEAM FILTERING FOR INNINGS TOTALS
# Purpose:
# 1. Retrieve the existing top-three BM25 candidates.
# 2. Extract the requested date and batting team from each question.
# 3. Select a candidate whose passage contains both details.
# 4. Measure exact target retrieval after selection.

# Step 1: Create a separate results list.
date_team_records = []

# Step 2: Evaluate all innings-run questions in the test set.
innings_test_df = test_questions_df[
    test_questions_df["question_type"] == "innings_runs"
]

for _, row in innings_test_df.iterrows():
    candidates = retrieve_top_three(row)
    question = str(row["question"])

    requested_date = extract_question_date(question)
    requested_team = extract_batting_team(question)

    selected_id = None
    selection_reason = "no_date_team_match"

    # Step 3: Search candidates in BM25 ranking order.
    for _, candidate in candidates.iterrows():
        passage_id = str(candidate["passage_id"])
        passage_text = str(candidate["text"])

        date_matches = (
            requested_date is not None
            and requested_date in passage_text
        )

        team_matches = (
            requested_team is not None
            and requested_team.lower() in passage_text.lower()
        )

        if date_matches and team_matches:
            selected_id = passage_id
            selection_reason = "date_and_team_match"
            break

    # Step 4: Evaluate against the target only after selection.
    target_id = str(row["target_passage_id"])

    date_team_records.append({
        "question_id": row["question_id"],
        "selected_passage_id": selected_id,
        "selection_reason": selection_reason,
        "candidate_found": selected_id is not None,
        "exact_target_retrieved": (
            selected_id == target_id if selected_id is not None else False
        )
    })

# Step 5: Summarise the diagnostic.
date_team_df = pd.DataFrame(date_team_records)

print("Innings-run test questions:", len(date_team_df))
print(
    "Questions with a matching candidate:",
    int(date_team_df["candidate_found"].sum())
)
print(
    "Exact target passages retrieved:",
    int(date_team_df["exact_target_retrieved"].sum())
)

print(
    "Exact target recall:",
    round(date_team_df["exact_target_retrieved"].mean(), 4)
)

# Step 6: Show questions for which no candidate matched both constraints.
display(
    date_team_df[~date_team_df["candidate_found"]]
)

Innings-run test questions: 505
Questions with a matching candidate: 501
Exact target passages retrieved: 501
Exact target recall: 0.9921


,question_id,selected_passage_id,selection_reason,candidate_found,exact_target_retrieved
43,1359482_innings2_runs,None,no_date_team_match,False,False
209,548352_innings2_runs,None,no_date_team_match,False,False
365,1527681_innings1_runs,None,no_date_team_match,False,False
411,1422136_innings1_runs,None,no_date_team_match,False,False


In [54]:
# STEP 50: INSPECT THE FOUR UNRESOLVED INNINGS-TOTAL QUESTIONS
# Purpose:
# 1. Retrieve the four questions with no date-and-team match.
# 2. Display the requested date and team.
# 3. Inspect the text of all three BM25 candidates.
# 4. Diagnose the failure without changing any experiment results.

# Step 1: Get the unresolved question IDs.
unresolved_ids = date_team_df[
    ~date_team_df["candidate_found"]
]["question_id"].tolist()

# Step 2: Inspect each unresolved question.
for question_id in unresolved_ids:
    row = test_questions_df[
        test_questions_df["question_id"] == question_id
    ].iloc[0]

    candidates = retrieve_top_three(row)
    question = str(row["question"])

    print("\n" + "=" * 80)
    print("Question:", question)
    print("Requested date:", extract_question_date(question))
    print("Requested team:", extract_batting_team(question))
    print("Gold answer (evaluation only):", row["gold_answer"])

    # Step 3: Display each candidate and its passage text.
    for rank, (_, candidate) in enumerate(candidates.iterrows(), start=1):
        print(f"\nRank {rank}: {candidate['passage_id']}")
        print(candidate["text"])


Question: How many runs did Rajasthan Royals score in innings 2 of the IPL match played on 2023-04-05?
Requested date: 2023-04-05
Requested team: Rajasthan Royals
Gold answer (evaluation only): 192

Rank 1: 1359526_summary
IPL match ID: 1359526. Date: 2023-05-07. Season: 2023. Teams: Rajasthan Royals, Sunrisers Hyderabad. Venue: Sawai Mansingh Stadium, Jaipur. City: Jaipur. Winner: Sunrisers Hyderabad. Toss winner: Rajasthan Royals. Toss decision: bat. Player of the match: GD Phillips. Rajasthan Royals scored 214 runs and lost 2 wickets in innings 1. Sunrisers Hyderabad scored 217 runs and lost 6 wickets in innings 2.

Rank 2: 1359530_summary
IPL match ID: 1359530. Date: 2023-05-11. Season: 2023. Teams: Kolkata Knight Riders, Rajasthan Royals. Venue: Eden Gardens, Kolkata. City: Kolkata. Winner: Rajasthan Royals. Toss winner: Rajasthan Royals. Toss decision: field. Player of the match: YBK Jaiswal. Kolkata Knight Riders scored 149 runs and lost 8 wickets in innings 1. Rajasthan Royals

In [55]:
# STEP 51: VERIFY QUESTION DATES AGAINST TARGET PASSAGE DATES
# Purpose:
# 1. Inspect the four unresolved innings-run questions.
# 2. Extract the date from each question.
# 3. Extract the date from its target passage.
# 4. Check whether the question and target passage agree.

# Step 1: Look up the four unresolved questions.
for question_id in unresolved_ids:
    row = test_questions_df[
        test_questions_df["question_id"] == question_id
    ].iloc[0]

    target_id = str(row["target_passage_id"])

    # Step 2: Retrieve the target passage for diagnosis only.
    target_passage = match_summary_df[
        match_summary_df["passage_id"].astype(str) == target_id
    ]

    if target_passage.empty:
        print("\nQuestion ID:", question_id)
        print("Target passage not found:", target_id)
        continue

    target_text = str(target_passage.iloc[0]["text"])

    # Step 3: Extract the question date and target passage date.
    question_date = extract_question_date(row["question"])
    target_date_match = re.search(
        r"\bDate:\s*(\d{4}-\d{2}-\d{2})\b",
        target_text
    )
    target_date = (
        target_date_match.group(1)
        if target_date_match
        else None
    )

    # Step 4: Report the comparison.
    print("\nQuestion ID:", question_id)
    print("Question date:", question_date)
    print("Target passage date:", target_date)
    print("Dates match:", question_date == target_date)
    print("Target passage ID:", target_id)


Question ID: 1359482_innings2_runs
Question date: 2023-04-05
Target passage date: 2023-04-05
Dates match: True
Target passage ID: 1359482_summary

Question ID: 548352_innings2_runs
Question date: 2012-05-04
Target passage date: 2012-05-04
Dates match: True
Target passage ID: 548352_summary

Question ID: 1527681_innings1_runs
Question date: 2026-04-04
Target passage date: 2026-04-04
Dates match: True
Target passage ID: 1527681_summary

Question ID: 1422136_innings1_runs
Question date: 2024-04-05
Target passage date: 2024-04-05
Dates match: True
Target passage ID: 1422136_summary


In [56]:
# STEP 52: CHECK TOP-10 BM25 RANKINGS FOR THE FOUR MISSES
# Purpose:
# 1. Re-run BM25 for the four unresolved innings-run questions.
# 2. Expand the candidate pool from top 3 to top 10.
# 3. Record the rank of the correct passage for evaluation.
# 4. Keep the target passage ID out of the retrieval ranking itself.

import numpy as np

# Step 1: Inspect each unresolved question.
for question_id in unresolved_ids:
    row = test_questions_df[
        test_questions_df["question_id"] == question_id
    ].iloc[0]

    question = str(row["question"])
    requested_date = extract_question_date(question)
    requested_team = extract_batting_team(question)
    target_id = str(row["target_passage_id"])

    # Step 2: Get the existing BM25 index and passage store.
    collection_name = get_collection_name(row["question_type"])
    index = bm25_indexes[("test", collection_name)]
    store = bm25_passage_stores[("test", collection_name)]

    # Step 3: Rank the full test collection and inspect the top 10.
    scores = index.get_scores(tokenize_text(question))
    ranked_positions = scores.argsort()[::-1][:10]
    top10 = store.iloc[ranked_positions].copy()

    # Step 4: Find the target's rank only after ranking is complete.
    target_positions = np.where(
        store["passage_id"].astype(str).to_numpy() == target_id
    )[0]

    target_rank = None
    if len(target_positions) > 0:
        target_position = target_positions[0]
        target_rank = int(
            np.where(scores.argsort()[::-1] == target_position)[0][0] + 1
        )

    print("\n" + "=" * 80)
    print("Question:", question)
    print("Requested date:", requested_date)
    print("Requested team:", requested_team)
    print("Correct target rank in full BM25 ranking:", target_rank)

    # Step 5: Display whether each top-10 candidate matches date and team.
    for rank, (_, candidate) in enumerate(top10.iterrows(), start=1):
        text = str(candidate["text"])
        date_matches = requested_date in text
        team_matches = (
            requested_team is not None
            and requested_team.lower() in text.lower()
        )

        print(
            f"\nRank {rank}: {candidate['passage_id']} | "
            f"date_match={date_matches} | team_match={team_matches}"
        )
        print(text)


Question: How many runs did Rajasthan Royals score in innings 2 of the IPL match played on 2023-04-05?
Requested date: 2023-04-05
Requested team: Rajasthan Royals
Correct target rank in full BM25 ranking: 4

Rank 1: 1359526_summary | date_match=False | team_match=True
IPL match ID: 1359526. Date: 2023-05-07. Season: 2023. Teams: Rajasthan Royals, Sunrisers Hyderabad. Venue: Sawai Mansingh Stadium, Jaipur. City: Jaipur. Winner: Sunrisers Hyderabad. Toss winner: Rajasthan Royals. Toss decision: bat. Player of the match: GD Phillips. Rajasthan Royals scored 214 runs and lost 2 wickets in innings 1. Sunrisers Hyderabad scored 217 runs and lost 6 wickets in innings 2.

Rank 2: 1359530_summary | date_match=False | team_match=True
IPL match ID: 1359530. Date: 2023-05-11. Season: 2023. Teams: Kolkata Knight Riders, Rajasthan Royals. Venue: Eden Gardens, Kolkata. City: Kolkata. Winner: Rajasthan Royals. Toss winner: Rajasthan Royals. Toss decision: field. Player of the match: YBK Jaiswal. Kolk

In [57]:
# STEP 52A: IMPORT NUMPY
# Purpose: Make NumPy available for the retrieval diagnostic.

import numpy as np

print("NumPy imported successfully.")

NumPy imported successfully.


In [58]:
# STEP 52: CHECK TOP-10 BM25 RANKINGS FOR THE FOUR MISSES
# Purpose:
# 1. Re-run BM25 for the four unresolved innings-run questions.
# 2. Expand the candidate pool from top 3 to top 10.
# 3. Record the rank of the correct passage for evaluation.
# 4. Keep the target passage ID out of the retrieval ranking itself.

# Step 1: Inspect each unresolved question.
for question_id in unresolved_ids:
    row = test_questions_df[
        test_questions_df["question_id"] == question_id
    ].iloc[0]

    question = str(row["question"])
    requested_date = extract_question_date(question)
    requested_team = extract_batting_team(question)
    target_id = str(row["target_passage_id"])

    # Step 2: Get the existing BM25 index and passage store.
    collection_name = get_collection_name(row["question_type"])
    index = bm25_indexes[("test", collection_name)]
    store = bm25_passage_stores[("test", collection_name)]

    # Step 3: Rank the full test collection and inspect the top 10.
    scores = index.get_scores(tokenize_text(question))
    ranked_positions = scores.argsort()[::-1][:10]
    top10 = store.iloc[ranked_positions].copy()

    # Step 4: Find the target's rank only after ranking is complete.
    target_positions = np.where(
        store["passage_id"].astype(str).to_numpy() == target_id
    )[0]

    target_rank = None
    if len(target_positions) > 0:
        target_position = target_positions[0]
        target_rank = int(
            np.where(scores.argsort()[::-1] == target_position)[0][0] + 1
        )

    print("\n" + "=" * 80)
    print("Question:", question)
    print("Requested date:", requested_date)
    print("Requested team:", requested_team)
    print("Correct target rank in full BM25 ranking:", target_rank)

    # Step 5: Display whether each top-10 candidate matches date and team.
    for rank, (_, candidate) in enumerate(top10.iterrows(), start=1):
        text = str(candidate["text"])
        date_matches = requested_date in text
        team_matches = (
            requested_team is not None
            and requested_team.lower() in text.lower()
        )

        print(
            f"\nRank {rank}: {candidate['passage_id']} | "
            f"date_match={date_matches} | team_match={team_matches}"
        )
        print(text)


Question: How many runs did Rajasthan Royals score in innings 2 of the IPL match played on 2023-04-05?
Requested date: 2023-04-05
Requested team: Rajasthan Royals
Correct target rank in full BM25 ranking: 4

Rank 1: 1359526_summary | date_match=False | team_match=True
IPL match ID: 1359526. Date: 2023-05-07. Season: 2023. Teams: Rajasthan Royals, Sunrisers Hyderabad. Venue: Sawai Mansingh Stadium, Jaipur. City: Jaipur. Winner: Sunrisers Hyderabad. Toss winner: Rajasthan Royals. Toss decision: bat. Player of the match: GD Phillips. Rajasthan Royals scored 214 runs and lost 2 wickets in innings 1. Sunrisers Hyderabad scored 217 runs and lost 6 wickets in innings 2.

Rank 2: 1359530_summary | date_match=False | team_match=True
IPL match ID: 1359530. Date: 2023-05-11. Season: 2023. Teams: Kolkata Knight Riders, Rajasthan Royals. Venue: Eden Gardens, Kolkata. City: Kolkata. Winner: Rajasthan Royals. Toss winner: Rajasthan Royals. Toss decision: field. Player of the match: YBK Jaiswal. Kolk

In [59]:
# STEP 53: EVALUATE TOP-10 DATE-AND-TEAM FILTERING
# Purpose:
# 1. Retrieve up to 10 candidates for each innings-run question.
# 2. Select the highest-ranked passage matching date and team.
# 3. Evaluate exact target retrieval after selection.
# 4. Compare with the earlier top-3 diagnostic.

# Step 1: Prepare a separate results list.
top10_date_team_records = []

# Step 2: Process each innings-run test question.
for _, row in innings_test_df.iterrows():
    question = str(row["question"])
    requested_date = extract_question_date(question)
    requested_team = extract_batting_team(question)

    collection_name = get_collection_name(row["question_type"])
    index = bm25_indexes[("test", collection_name)]
    store = bm25_passage_stores[("test", collection_name)]

    # Step 3: Rank candidates using BM25.
    scores = index.get_scores(tokenize_text(question))
    ranked_positions = scores.argsort()[::-1][:10]
    candidates = store.iloc[ranked_positions]

    selected_id = None
    selected_rank = None

    # Step 4: Select the first candidate matching date and team.
    for rank, (_, candidate) in enumerate(candidates.iterrows(), start=1):
        passage_text = str(candidate["text"])
        passage_id = str(candidate["passage_id"])

        date_matches = (
            requested_date is not None
            and requested_date in passage_text
        )
        team_matches = (
            requested_team is not None
            and requested_team.lower() in passage_text.lower()
        )

        if date_matches and team_matches:
            selected_id = passage_id
            selected_rank = rank
            break

    # Step 5: Evaluate against the target after selection.
    target_id = str(row["target_passage_id"])

    top10_date_team_records.append({
        "question_id": row["question_id"],
        "selected_passage_id": selected_id,
        "selected_rank": selected_rank,
        "candidate_found": selected_id is not None,
        "exact_target_retrieved": selected_id == target_id
    })

# Step 6: Summarise the results.
top10_date_team_df = pd.DataFrame(top10_date_team_records)

print("Questions:", len(top10_date_team_df))
print(
    "Questions with a matching candidate:",
    int(top10_date_team_df["candidate_found"].sum())
)
print(
    "Exact target passages retrieved:",
    int(top10_date_team_df["exact_target_retrieved"].sum())
)
print(
    "Exact target recall:",
    round(top10_date_team_df["exact_target_retrieved"].mean(), 4)
)

# Step 7: Show how far down the ranking the selected candidates appeared.
display(
    top10_date_team_df["selected_rank"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("selected_rank")
    .reset_index(name="question_count")
)

Questions: 505
Questions with a matching candidate: 505
Exact target passages retrieved: 505
Exact target recall: 1.0


,selected_rank,question_count
0,1,483
1,2,10
2,3,8
3,4,4


In [60]:
# STEP 54: SAVE THE TOP-10 DATE-AND-TEAM DIAGNOSTIC
# Purpose:
# 1. Save question-level retrieval decisions.
# 2. Save a summary of exact-passage recall.
# 3. Preserve the earlier baseline and top-3 diagnostic.
# 4. Make the results available for the final research report.

# Step 1: Save the question-level results.
top10_date_team_df.to_csv(
    "/content/ipl_top10_date_team_diagnostic.csv",
    index=False
)

# Step 2: Calculate the summary metrics.
top10_diagnostic_summary = pd.DataFrame([{
    "experiment": "BM25 top-10 with date-and-team filtering",
    "question_type": "innings_runs",
    "test_questions": len(top10_date_team_df),
    "candidate_found": int(top10_date_team_df["candidate_found"].sum()),
    "exact_target_retrieved": int(
        top10_date_team_df["exact_target_retrieved"].sum()
    ),
    "exact_target_recall": float(
        top10_date_team_df["exact_target_retrieved"].mean()
    )
}])

# Step 3: Save the summary.
top10_diagnostic_summary.to_csv(
    "/content/ipl_top10_date_team_summary.csv",
    index=False
)

# Step 4: Display the summary for verification.
display(top10_diagnostic_summary)

print("\nSaved files:")
print("/content/ipl_top10_date_team_diagnostic.csv")
print("/content/ipl_top10_date_team_summary.csv")

,experiment,question_type,test_questions,candidate_found,exact_target_retrieved,exact_target_recall
0,BM25 top-10 with date-and-team filtering,innings_runs,505,505,505,1.0



Saved files:
/content/ipl_top10_date_team_diagnostic.csv
/content/ipl_top10_date_team_summary.csv


In [61]:
# STEP 55: CREATE A STRATIFIED 60-QUESTION GENERATION PILOT
# Purpose:
# 1. Sample questions from all three question types.
# 2. Keep the test set and random seed fixed.
# 3. Create a manageable pilot for CPU-based answer generation.
# 4. Save the selected questions separately from retrieval diagnostics.

# Step 1: Define the sample size for each question type.
pilot_counts = {
    "batter_runs": 20,
    "innings_runs": 20,
    "match_winner": 20
}

# Step 2: Sample deterministically from the held-out test set.
pilot_parts = []

for question_type, sample_count in pilot_counts.items():
    type_questions = test_questions_df[
        test_questions_df["question_type"] == question_type
    ]

    sample = type_questions.sample(
        n=min(sample_count, len(type_questions)),
        random_state=42
    )

    pilot_parts.append(sample)

# Step 3: Combine and shuffle the selected questions.
generation_pilot_df = (
    pd.concat(pilot_parts, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Step 4: Verify the sample composition.
print("Total pilot questions:", len(generation_pilot_df))
display(
    generation_pilot_df["question_type"]
    .value_counts()
    .rename_axis("question_type")
    .reset_index(name="sample_count")
)

# Step 5: Save the pilot question set.
generation_pilot_df.to_csv(
    "/content/ipl_generation_pilot_60_questions.csv",
    index=False
)

print("\nSaved: /content/ipl_generation_pilot_60_questions.csv")

Total pilot questions: 60


,question_type,sample_count
0,batter_runs,20
1,innings_runs,20
2,match_winner,20



Saved: /content/ipl_generation_pilot_60_questions.csv


In [62]:
# STEP 56: PREPARE EVIDENCE FOR THE THREE GENERATION CONDITIONS
# Purpose:
# 1. Use the same 60 questions for every condition.
# 2. Prepare no-retrieval and fixed-BM25 evidence.
# 3. Prepare metadata-assisted evidence for each question type.
# 4. Save the inputs without generating answers yet.

# Step 1: Define a helper to retrieve a passage by its ID.
def get_passage_text(passage_id, question_type):
    if passage_id is None:
        return None

    collection_name = get_collection_name(question_type)

    if collection_name == "player_stat":
        lookup = player_text_lookup
    else:
        lookup = summary_text_lookup

    return lookup.get(str(passage_id))

# Step 2: Create a separate evidence-preparation table.
generation_input_records = []

for _, row in generation_pilot_df.iterrows():
    question = str(row["question"])
    question_type = row["question_type"]

    # Step 3: Retrieve the original BM25 top-1 passage.
    candidates = retrieve_top_three(row)
    fixed_top1 = candidates.iloc[0]
    fixed_passage_id = str(fixed_top1["passage_id"])
    fixed_evidence = str(fixed_top1["text"])

    # Step 4: Prepare metadata-assisted evidence.
    # Batter questions use date, player, and innings.
    # Innings-total questions search up to top 10 using date and team.
    # Match-winner questions retain the original top-1 candidate for now.
    adaptive_passage_id = None
    adaptive_evidence = None

    requested_date = extract_question_date(question)

    if question_type == "batter_runs":
        requested_innings = extract_innings_number(question)
        requested_batter = extract_batter_name(question)

        # Search the available top-three candidates in BM25 order.
        for _, candidate in candidates.iterrows():
            passage_id = str(candidate["passage_id"])
            passage_text = str(candidate["text"])

            if (
                requested_date is not None
                and requested_date in passage_text
                and requested_batter is not None
                and requested_batter.lower() in passage_text.lower()
                and extract_passage_innings(passage_id) == requested_innings
            ):
                adaptive_passage_id = passage_id
                adaptive_evidence = passage_text
                break

    elif question_type == "innings_runs":
        requested_team = extract_batting_team(question)

        collection_name = get_collection_name(question_type)
        index = bm25_indexes[("test", collection_name)]
        store = bm25_passage_stores[("test", collection_name)]

        scores = index.get_scores(tokenize_text(question))
        ranked_positions = scores.argsort()[::-1][:10]
        top10_candidates = store.iloc[ranked_positions]

        for _, candidate in top10_candidates.iterrows():
            passage_id = str(candidate["passage_id"])
            passage_text = str(candidate["text"])

            if (
                requested_date is not None
                and requested_date in passage_text
                and requested_team is not None
                and requested_team.lower() in passage_text.lower()
            ):
                adaptive_passage_id = passage_id
                adaptive_evidence = passage_text
                break

    else:
        # Match-winner questions retain BM25 top-1 in this pilot.
        adaptive_passage_id = fixed_passage_id
        adaptive_evidence = fixed_evidence

    generation_input_records.append({
        "question_id": row["question_id"],
        "question_type": question_type,
        "question": question,
        "gold_answer": row["gold_answer"],
        "no_retrieval_evidence": "",
        "fixed_passage_id": fixed_passage_id,
        "fixed_evidence": fixed_evidence,
        "adaptive_passage_id": adaptive_passage_id,
        "adaptive_evidence": adaptive_evidence
    })

# Step 5: Save the prepared inputs.
generation_inputs_df = pd.DataFrame(generation_input_records)

generation_inputs_df.to_csv(
    "/content/ipl_generation_pilot_evidence_inputs.csv",
    index=False
)

# Step 6: Check whether all conditions have evidence where expected.
print("Prepared questions:", len(generation_inputs_df))
print(
    "Fixed-RAG evidence available:",
    int(generation_inputs_df["fixed_evidence"].notna().sum())
)
print(
    "Metadata-assisted evidence available:",
    int(generation_inputs_df["adaptive_evidence"].notna().sum())
)

display(
    generation_inputs_df[
        [
            "question_type",
            "question",
            "fixed_passage_id",
            "adaptive_passage_id"
        ]
    ].head(10)
)

print("\nSaved: /content/ipl_generation_pilot_evidence_inputs.csv")

Prepared questions: 60
Fixed-RAG evidence available: 60
Metadata-assisted evidence available: 60


,question_type,question,fixed_passage_id,adaptive_passage_id
0,batter_runs,How many runs did R Tewatia score for Gujarat ...,1359531_innings2_R Tewatia,1359531_innings2_R Tewatia
1,batter_runs,How many runs did DT Christian score for Risin...,1082650_innings2_DT Christian,1082650_innings2_DT Christian
2,innings_runs,How many runs did Sunrisers Hyderabad score in...,1216512_summary,1216512_summary
3,match_winner,Which team won the IPL match between Mumbai In...,1527681_summary,1527681_summary
4,batter_runs,How many runs did KL Rahul score for Royal Cha...,598017_innings2_KL Rahul,598017_innings2_KL Rahul
5,match_winner,Which team won the IPL match between Sunrisers...,980943_summary,980943_summary
6,innings_runs,How many runs did Chennai Super Kings score in...,1304092_summary,1304092_summary
7,match_winner,Which team won the IPL match between Royal Cha...,392236_summary,392236_summary
8,batter_runs,How many runs did JC Buttler score for Rajasth...,1216522_innings1_JC Buttler,1216522_innings1_JC Buttler
9,match_winner,Which team won the IPL match between Kings XI ...,598007_summary,598007_summary



Saved: /content/ipl_generation_pilot_evidence_inputs.csv


In [63]:
# STEP 57: CHECK EVIDENCE DIFFERENCES BEFORE GENERATION
# Purpose:
# 1. Count how often metadata-assisted retrieval changes the passage.
# 2. Identify questions where the two methods select different passages.
# 3. Check for missing evidence.
# 4. Avoid running the model before validating the experiment inputs.

# Step 1: Count questions where the selected passage differs.
different_passage_mask = (
    generation_inputs_df["fixed_passage_id"].astype(str)
    != generation_inputs_df["adaptive_passage_id"].astype(str)
)

print(
    "Questions with different fixed vs metadata-assisted passages:",
    int(different_passage_mask.sum()),
    "out of",
    len(generation_inputs_df)
)

# Step 2: Check for missing evidence in either condition.
print(
    "Missing fixed-RAG evidence:",
    int(generation_inputs_df["fixed_evidence"].isna().sum())
)
print(
    "Missing metadata-assisted evidence:",
    int(generation_inputs_df["adaptive_evidence"].isna().sum())
)

# Step 3: Display examples where the selected passages differ.
display(
    generation_inputs_df.loc[
        different_passage_mask,
        [
            "question_type",
            "question",
            "fixed_passage_id",
            "adaptive_passage_id"
        ]
    ].head(10)
)

Questions with different fixed vs metadata-assisted passages: 1 out of 60
Missing fixed-RAG evidence: 0
Missing metadata-assisted evidence: 0


,question_type,question,fixed_passage_id,adaptive_passage_id
42,innings_runs,How many runs did Deccan Chargers score in inn...,392237_summary,392188_summary


In [64]:
# STEP 58: RUN THE 60-QUESTION ANSWER-GENERATION PILOT
# Purpose:
# 1. Compare no retrieval, fixed RAG, and metadata-assisted retrieval.
# 2. Use the same FLAN-T5-small model and decoding settings.
# 3. Save every generated answer for later evaluation.
# 4. Keep this pilot separate from previous retrieval experiments.

# Step 1: Define a consistent answer-generation function.
def generate_answer(question, evidence=""):
    if evidence:
        prompt = (
            "Answer the question using the evidence below. "
            "Give only the answer. Do not confuse the match winner "
            "with the toss winner.\n\n"
            f"Evidence: {evidence}\n\n"
            f"Question: {question}\n"
            "Answer:"
        )
    else:
        prompt = (
            "Answer the following question as accurately as possible. "
            "Give only the answer.\n\n"
            f"Question: {question}\n"
            "Answer:"
        )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            num_beams=1
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

# Step 2: Generate answers under all three conditions.
generation_results = []

for index, row in generation_inputs_df.iterrows():
    question = str(row["question"])

    # No retrieval: question only.
    answer_no_retrieval = generate_answer(question)

    # Fixed RAG: use the original BM25 top-1 passage.
    answer_fixed_rag = generate_answer(
        question,
        str(row["fixed_evidence"])
    )

    # Metadata-assisted retrieval: use the selected passage.
    answer_metadata_assisted = generate_answer(
        question,
        str(row["adaptive_evidence"])
    )

    generation_results.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "question": question,
        "gold_answer": row["gold_answer"],
        "fixed_passage_id": row["fixed_passage_id"],
        "adaptive_passage_id": row["adaptive_passage_id"],
        "answer_no_retrieval": answer_no_retrieval,
        "answer_fixed_rag": answer_fixed_rag,
        "answer_metadata_assisted": answer_metadata_assisted
    })

    # Step 3: Print progress every 10 questions.
    if (index + 1) % 10 == 0:
        print(f"Completed {index + 1} of {len(generation_inputs_df)} questions.")

# Step 4: Save the generated answers.
generation_results_df = pd.DataFrame(generation_results)

generation_results_df.to_csv(
    "/content/ipl_generation_pilot_answers.csv",
    index=False
)

print("\nGeneration complete.")
print("Total questions:", len(generation_results_df))
print("Total answer generations:", len(generation_results_df) * 3)
print("Saved: /content/ipl_generation_pilot_answers.csv")

# Step 5: Inspect a few answers before scoring.
display(
    generation_results_df[
        [
            "question_type",
            "question",
            "gold_answer",
            "answer_no_retrieval",
            "answer_fixed_rag",
            "answer_metadata_assisted"
        ]
    ].head(10)
)

Completed 10 of 60 questions.
Completed 20 of 60 questions.
Completed 30 of 60 questions.
Completed 40 of 60 questions.
Completed 50 of 60 questions.
Completed 60 of 60 questions.

Generation complete.
Total questions: 60
Total answer generations: 180
Saved: /content/ipl_generation_pilot_answers.csv


,question_type,question,gold_answer,answer_no_retrieval,answer_fixed_rag,answer_metadata_assisted
0,batter_runs,How many runs did R Tewatia score for Gujarat ...,14,69,0,0
1,batter_runs,How many runs did DT Christian score for Risin...,4,0,0,0
2,innings_runs,How many runs did Sunrisers Hyderabad score in...,2,0,0,0
3,match_winner,Which team won the IPL match between Mumbai In...,Delhi Capitals,Mumbai Indians,Mumbai Indians,Mumbai Indians
4,batter_runs,How many runs did KL Rahul score for Royal Cha...,12,0,598017,598017
5,match_winner,Which team won the IPL match between Sunrisers...,Rising Pune Supergiants,san juan,Sunrisers Hyderabad,Sunrisers Hyderabad
6,innings_runs,How many runs did Chennai Super Kings score in...,202,0,202,202
7,match_winner,Which team won the IPL match between Royal Cha...,Royal Challengers Bangalore,san juan,Royal Challengers Bangalore,Royal Challengers Bangalore
8,batter_runs,How many runs did JC Buttler score for Rajasth...,24,0,24,24
9,match_winner,Which team won the IPL match between Kings XI ...,Chennai Super Kings,san juan,Chennai Super Kings,Chennai Super Kings


In [65]:
# STEP 59: INSPECT INCORRECT MODEL ANSWERS
# Purpose:
# 1. Find examples where fixed RAG generated an incorrect answer.
# 2. Compare the question, gold answer, and generated answer.
# 3. Inspect the evidence provided to the model.
# 4. Diagnose errors before modifying the experiment.

# Step 1: Select a small sample of fixed-RAG errors.
incorrect_examples_df = generation_results_df[
    generation_results_df["answer_fixed_rag"].astype(str).str.strip()
    != generation_results_df["gold_answer"].astype(str).str.strip()
].head(8)

print("Incorrect fixed-RAG examples inspected:", len(incorrect_examples_df))

# Step 2: Display the question, expected answer, and model output.
for _, row in incorrect_examples_df.iterrows():
    source_row = generation_inputs_df[
        generation_inputs_df["question_id"] == row["question_id"]
    ].iloc[0]

    print("\n" + "=" * 80)
    print("Question type:", row["question_type"])
    print("Question:", row["question"])
    print("Gold answer:", row["gold_answer"])
    print("Fixed-RAG answer:", row["answer_fixed_rag"])
    print("Passage ID:", row["fixed_passage_id"])
    print("Evidence supplied to model:")
    print(source_row["fixed_evidence"])

Incorrect fixed-RAG examples inspected: 8

Question type: batter_runs
Question: How many runs did R Tewatia score for Gujarat Titans in innings 2 of the IPL match played on 2023-05-12?
Gold answer: 14
Fixed-RAG answer: 0
Passage ID: 1359531_innings2_R Tewatia
Evidence supplied to model:
IPL match ID: 1359531. Date: 2023-05-12. Season: 2023. Innings: 2. Batting team: Gujarat Titans. Player: R Tewatia. Runs scored by R Tewatia: 14.

Question type: batter_runs
Question: How many runs did DT Christian score for Rising Pune Supergiant in innings 2 of the IPL match played on 2017-05-21?
Gold answer: 4
Fixed-RAG answer: 0
Passage ID: 1082650_innings2_DT Christian
Evidence supplied to model:
IPL match ID: 1082650. Date: 2017-05-21. Season: 2017. Innings: 2. Batting team: Rising Pune Supergiant. Player: DT Christian. Runs scored by DT Christian: 4.

Question type: innings_runs
Question: How many runs did Sunrisers Hyderabad score in innings 3 of the IPL match played on 2020-10-18?
Gold answer: 

In [66]:
# STEP 60: TEST STRUCTURED PROMPTS ON EIGHT DIAGNOSTIC EXAMPLES
# Purpose:
# 1. Use question-specific instructions for each answer type.
# 2. Ask the model to extract only the requested field.
# 3. Keep the evidence unchanged for a fair prompt comparison.
# 4. Save diagnostic outputs separately from the original baseline.

# Step 1: Define a structured prompt function.
def generate_structured_answer(question, question_type, evidence):
    if question_type == "batter_runs":
        instruction = (
            "Find the exact player's runs-scored value in the evidence. "
            "Return only that number. Do not return the match ID, "
            "innings number, or another player's runs."
        )
    elif question_type == "innings_runs":
        instruction = (
            "Find the runs scored by the named team in the requested innings. "
            "Return only the number of runs for that team and innings."
        )
    else:
        instruction = (
            "Find the value of the Winner field in the evidence. "
            "Return only the winning team. Do not use the Toss winner field."
        )

    prompt = (
        f"Instruction: {instruction}\n\n"
        f"Evidence:\n{evidence}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

    # Step 2: Generate deterministically using the same model.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            num_beams=1
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

# Step 3: Select the eight diagnostic examples.
diagnostic_ids = incorrect_examples_df["question_id"].tolist()
structured_records = []

for question_id in diagnostic_ids:
    row = generation_inputs_df[
        generation_inputs_df["question_id"] == question_id
    ].iloc[0]

    # Step 4: Generate a structured-prompt answer using the same evidence.
    structured_answer = generate_structured_answer(
        str(row["question"]),
        str(row["question_type"]),
        str(row["fixed_evidence"])
    )

    structured_records.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "question": row["question"],
        "gold_answer": row["gold_answer"],
        "original_fixed_rag_answer": generation_results_df.loc[
            generation_results_df["question_id"] == question_id,
            "answer_fixed_rag"
        ].iloc[0],
        "structured_prompt_answer": structured_answer
    })

# Step 5: Display the comparison without overwriting original results.
structured_diagnostic_df = pd.DataFrame(structured_records)

display(
    structured_diagnostic_df[
        [
            "question_type",
            "gold_answer",
            "original_fixed_rag_answer",
            "structured_prompt_answer"
        ]
    ]
)

# Step 6: Save the diagnostic.
structured_diagnostic_df.to_csv(
    "/content/ipl_structured_prompt_diagnostic.csv",
    index=False
)

print("Saved: /content/ipl_structured_prompt_diagnostic.csv")

,question_type,gold_answer,original_fixed_rag_answer,structured_prompt_answer
0,batter_runs,14,0,14
1,batter_runs,4,0,4
2,innings_runs,2,0,0
3,match_winner,Delhi Capitals,Mumbai Indians,Mumbai Indians
4,batter_runs,12,598017,598017
5,match_winner,Rising Pune Supergiants,Sunrisers Hyderabad,Sunrisers Hyderabad
6,batter_runs,19,0,0
7,match_winner,Gujarat Titans,Kolkata Knight Riders,Kolkata Knight Riders


Saved: /content/ipl_structured_prompt_diagnostic.csv


In [67]:
# STEP 61: TEST RULE-BASED EXTRACTION ON THE EIGHT DIAGNOSTIC EXAMPLES
# Purpose:
# 1. Extract the requested value directly from structured evidence.
# 2. Use the question type to choose the appropriate field.
# 3. Compare the extracted value with the gold answer.
# 4. Keep these results separate from the LLM experiments.

# Step 1: Define an evidence-field extractor.
def extract_answer_from_evidence(question_type, evidence, question):
    evidence = str(evidence)

    if question_type == "batter_runs":
        match = re.search(
            r"Runs scored by .+?:\s*(\d+)",
            evidence
        )
        return match.group(1) if match else None

    if question_type == "innings_runs":
        team = extract_batting_team(question)
        innings = extract_innings_number(question)

        if team is None or innings is None:
            return None

        # Find the sentence for the requested innings and team.
        pattern = (
            re.escape(team)
            + r"\s+scored\s+(\d+)\s+runs.*?"
            + r"\bin\s+innings\s+"
            + str(innings)
            + r"\b"
        )

        match = re.search(pattern, evidence, flags=re.IGNORECASE)
        return match.group(1) if match else None

    if question_type == "match_winner":
        match = re.search(
            r"\bWinner:\s*([^.]*)",
            evidence,
            flags=re.IGNORECASE
        )
        if match:
            winner = match.group(1).strip()
            return winner if winner and winner.lower() != "none" else None

    return None

# Step 2: Apply the extractor to the same eight diagnostic examples.
rule_records = []

for question_id in diagnostic_ids:
    row = generation_inputs_df[
        generation_inputs_df["question_id"] == question_id
    ].iloc[0]

    extracted = extract_answer_from_evidence(
        str(row["question_type"]),
        str(row["fixed_evidence"]),
        str(row["question"])
    )

    rule_records.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "gold_answer": str(row["gold_answer"]),
        "rule_extracted_answer": extracted,
        "exact_match": (
            normalize_answer_text(extracted)
            == normalize_answer_text(row["gold_answer"])
            if extracted is not None
            else False
        )
    })

# Step 3: Display and save the diagnostic results.
rule_diagnostic_df = pd.DataFrame(rule_records)

display(rule_diagnostic_df)

print(
    "Exact matches:",
    int(rule_diagnostic_df["exact_match"].sum()),
    "out of",
    len(rule_diagnostic_df)
)

rule_diagnostic_df.to_csv(
    "/content/ipl_rule_extraction_diagnostic.csv",
    index=False
)

print("Saved: /content/ipl_rule_extraction_diagnostic.csv")

,question_id,question_type,gold_answer,rule_extracted_answer,exact_match
0,1359531_innings2_R Tewatia_runs,batter_runs,14,14,True
1,1082650_innings2_DT Christian_runs,batter_runs,4,4,True
2,1216512_innings3_runs,innings_runs,2,163,False
3,1527681_winner,match_winner,Delhi Capitals,Delhi Capitals,True
4,598017_innings2_KL Rahul_runs,batter_runs,12,12,True
5,980943_winner,match_winner,Rising Pune Supergiants,Rising Pune Supergiants,True
6,1216522_innings2_Gurkeerat Singh_runs,batter_runs,19,19,True
7,1359513_winner,match_winner,Gujarat Titans,Gujarat Titans,True


Exact matches: 7 out of 8
Saved: /content/ipl_rule_extraction_diagnostic.csv


In [68]:
# STEP 62: FIX INNINGS-SPECIFIC RUN EXTRACTION
# Purpose:
# 1. Parse each innings statement separately.
# 2. Match the requested team and innings number.
# 3. Avoid confusing regular innings with Super Over innings.
# 4. Recheck the eight diagnostic examples.

# Step 1: Define a corrected extractor.
def extract_answer_from_evidence_v2(question_type, evidence, question):
    evidence = str(evidence)

    if question_type == "batter_runs":
        match = re.search(
            r"Runs scored by .+?:\s*(\d+)",
            evidence
        )
        return match.group(1) if match else None

    if question_type == "innings_runs":
        team = extract_batting_team(question)
        innings = extract_innings_number(question)

        if team is None or innings is None:
            return None

        # Parse each team innings statement independently.
        pattern = (
            r"([^.]+?)\s+scored\s+(\d+)\s+runs"
            r"\s+and\s+lost\s+\d+\s+wickets\s+in\s+innings\s+(\d+)"
        )

        for match in re.finditer(pattern, evidence, flags=re.IGNORECASE):
            passage_team = match.group(1).strip()
            runs = match.group(2)
            passage_innings = int(match.group(3))

            if (
                passage_team.lower() == team.lower()
                and passage_innings == innings
            ):
                return runs

        return None

    if question_type == "match_winner":
        match = re.search(
            r"\bWinner:\s*([^.]*)",
            evidence,
            flags=re.IGNORECASE
        )
        if match:
            winner = match.group(1).strip()
            return winner if winner and winner.lower() != "none" else None

    return None

# Step 2: Re-evaluate the same eight examples.
corrected_records = []

for question_id in diagnostic_ids:
    row = generation_inputs_df[
        generation_inputs_df["question_id"] == question_id
    ].iloc[0]

    extracted = extract_answer_from_evidence_v2(
        str(row["question_type"]),
        str(row["fixed_evidence"]),
        str(row["question"])
    )

    corrected_records.append({
        "question_id": question_id,
        "question_type": row["question_type"],
        "gold_answer": str(row["gold_answer"]),
        "corrected_extraction": extracted,
        "exact_match": (
            normalize_answer_text(extracted)
            == normalize_answer_text(row["gold_answer"])
            if extracted is not None
            else False
        )
    })

# Step 3: Display the corrected diagnostic.
corrected_diagnostic_df = pd.DataFrame(corrected_records)
display(corrected_diagnostic_df)

print(
    "\nExact matches:",
    int(corrected_diagnostic_df["exact_match"].sum()),
    "out of",
    len(corrected_diagnostic_df)
)

,question_id,question_type,gold_answer,corrected_extraction,exact_match
0,1359531_innings2_R Tewatia_runs,batter_runs,14,14,True
1,1082650_innings2_DT Christian_runs,batter_runs,4,4,True
2,1216512_innings3_runs,innings_runs,2,2,True
3,1527681_winner,match_winner,Delhi Capitals,Delhi Capitals,True
4,598017_innings2_KL Rahul_runs,batter_runs,12,12,True
5,980943_winner,match_winner,Rising Pune Supergiants,Rising Pune Supergiants,True
6,1216522_innings2_Gurkeerat Singh_runs,batter_runs,19,19,True
7,1359513_winner,match_winner,Gujarat Titans,Gujarat Titans,True



Exact matches: 8 out of 8


In [69]:
# STEP 63: EVALUATE CORRECTED RULE-BASED EXTRACTION ON ALL 60 QUESTIONS
# Purpose:
# 1. Apply the corrected extractor to every pilot question.
# 2. Use the fixed BM25 evidence for a consistent comparison.
# 3. Score extracted answers against gold answers.
# 4. Save results separately from the LLM-generation experiment.

# Step 1: Create the rule-extraction results.
rule_pilot_records = []

for _, row in generation_inputs_df.iterrows():
    extracted_answer = extract_answer_from_evidence_v2(
        str(row["question_type"]),
        str(row["fixed_evidence"]),
        str(row["question"])
    )

    gold_answer = str(row["gold_answer"])

    is_correct = (
        normalize_answer_text(extracted_answer)
        == normalize_answer_text(gold_answer)
        if extracted_answer is not None
        else False
    )

    rule_pilot_records.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "gold_answer": gold_answer,
        "extracted_answer": extracted_answer,
        "answer_found": extracted_answer is not None,
        "correct": is_correct
    })

# Step 2: Store the question-level results.
rule_pilot_df = pd.DataFrame(rule_pilot_records)

# Step 3: Summarise overall performance.
print("Total questions:", len(rule_pilot_df))
print("Answers extracted:", int(rule_pilot_df["answer_found"].sum()))
print("Correct answers:", int(rule_pilot_df["correct"].sum()))
print("Overall accuracy:", round(rule_pilot_df["correct"].mean(), 4))

# Step 4: Summarise performance by question type.
rule_pilot_summary_df = (
    rule_pilot_df
    .groupby("question_type")
    .agg(
        questions=("question_id", "count"),
        answers_extracted=("answer_found", "sum"),
        correct_answers=("correct", "sum")
    )
    .reset_index()
)

rule_pilot_summary_df["accuracy"] = (
    rule_pilot_summary_df["correct_answers"]
    / rule_pilot_summary_df["questions"]
)

display(rule_pilot_summary_df)

# Step 5: Save the detailed results and summary.
rule_pilot_df.to_csv(
    "/content/ipl_rule_extraction_pilot_60.csv",
    index=False
)

rule_pilot_summary_df.to_csv(
    "/content/ipl_rule_extraction_pilot_summary.csv",
    index=False
)

print("\nSaved rule-extraction results and summary.")

Total questions: 60
Answers extracted: 59
Correct answers: 59
Overall accuracy: 0.9833


,question_type,questions,answers_extracted,correct_answers,accuracy
0,batter_runs,20,20,20,1.00
1,innings_runs,20,19,19,0.95
2,match_winner,20,20,20,1.00



Saved rule-extraction results and summary.


In [70]:
# STEP 64: COMPARE ALL FOUR ANSWERING METHODS ON THE SAME 60 QUESTIONS
# Purpose:
# 1. Score all methods against the same gold answers.
# 2. Use consistent scoring for numeric and winner questions.
# 3. Report overall and question-type accuracy.
# 4. Save the comparison for the research paper.

import re
import pandas as pd

# Step 1: Define a stricter, consistent answer scorer.
def score_answer(question_type, answer, gold_answer):
    if answer is None or pd.isna(answer):
        return False

    answer = str(answer).strip()
    gold_answer = str(gold_answer).strip()

    if not answer:
        return False

    if question_type in {"innings_runs", "batter_runs"}:
        answer_numbers = re.findall(r"\b\d+\b", answer)
        gold_numbers = re.findall(r"\b\d+\b", gold_answer)

        return (
            len(answer_numbers) == 1
            and len(gold_numbers) == 1
            and answer_numbers[0] == gold_numbers[0]
        )

    return (
        normalize_answer_text(answer)
        == normalize_answer_text(gold_answer)
    )

# Step 2: Join the generated answers with the rule-extraction results.
comparison_df = generation_results_df.merge(
    rule_pilot_df[
        ["question_id", "extracted_answer"]
    ],
    on="question_id",
    how="left",
    validate="one_to_one"
)

# Step 3: Define the four methods and their answer columns.
method_columns = {
    "No retrieval": "answer_no_retrieval",
    "Fixed BM25 RAG": "answer_fixed_rag",
    "Metadata-assisted RAG": "answer_metadata_assisted",
    "Rule-based extraction": "extracted_answer"
}

# Step 4: Score every method using the same function.
for method_name, answer_column in method_columns.items():
    comparison_df[method_name] = comparison_df.apply(
        lambda row: score_answer(
            row["question_type"],
            row[answer_column],
            row["gold_answer"]
        ),
        axis=1
    )

# Step 5: Create the overall comparison.
overall_records = []

for method_name in method_columns:
    correct = int(comparison_df[method_name].sum())
    total = len(comparison_df)

    overall_records.append({
        "method": method_name,
        "correct_answers": correct,
        "total_questions": total,
        "accuracy": correct / total
    })

overall_comparison_df = pd.DataFrame(overall_records)
overall_comparison_df["accuracy_percent"] = (
    overall_comparison_df["accuracy"] * 100
).round(2)

display(overall_comparison_df)

# Step 6: Compare accuracy by question type.
type_records = []

for question_type, group in comparison_df.groupby("question_type"):
    for method_name in method_columns:
        type_records.append({
            "question_type": question_type,
            "method": method_name,
            "correct_answers": int(group[method_name].sum()),
            "total_questions": len(group),
            "accuracy": group[method_name].mean()
        })

type_comparison_df = pd.DataFrame(type_records)
type_comparison_df["accuracy_percent"] = (
    type_comparison_df["accuracy"] * 100
).round(2)

display(
    type_comparison_df.sort_values(
        ["question_type", "method"]
    ).reset_index(drop=True)
)

# Step 7: Save both comparison tables.
overall_comparison_df.to_csv(
    "/content/ipl_generation_method_comparison.csv",
    index=False
)

type_comparison_df.to_csv(
    "/content/ipl_generation_method_comparison_by_type.csv",
    index=False
)

print("Saved both comparison tables.")

,method,correct_answers,total_questions,accuracy,accuracy_percent
0,No retrieval,5,60,0.083333,8.33
1,Fixed BM25 RAG,33,60,0.550000,55.00
2,Metadata-assisted RAG,34,60,0.566667,56.67
3,Rule-based extraction,59,60,0.983333,98.33


,question_type,method,correct_answers,total_questions,accuracy,accuracy_percent
0,batter_runs,Fixed BM25 RAG,9,20,0.45,45.0
1,batter_runs,Metadata-assisted RAG,9,20,0.45,45.0
2,batter_runs,No retrieval,2,20,0.10,10.0
3,batter_runs,Rule-based extraction,20,20,1.00,100.0
4,innings_runs,Fixed BM25 RAG,10,20,0.50,50.0
5,innings_runs,Metadata-assisted RAG,11,20,0.55,55.0
6,innings_runs,No retrieval,0,20,0.00,0.0
7,innings_runs,Rule-based extraction,19,20,0.95,95.0
8,match_winner,Fixed BM25 RAG,14,20,0.70,70.0
9,match_winner,Metadata-assisted RAG,14,20,0.70,70.0


Saved both comparison tables.


In [73]:

# STEP 65: DIAGNOSE FIXED-RAG ANSWER ERRORS

# 1. Keep questions where fixed BM25 RAG was incorrect.
fixed_rag_errors_df = comparison_df[
    ~comparison_df["Fixed BM25 RAG"].astype(bool)
].copy()

# 2. Add evidence text without duplicating passage ID columns.
evidence_columns = [
    c for c in ["question_id", "fixed_evidence", "adaptive_evidence"]
    if c in generation_inputs_df.columns
]

evidence_lookup_df = generation_inputs_df[
    evidence_columns
].drop_duplicates(subset=["question_id"])

fixed_rag_errors_df = fixed_rag_errors_df.merge(
    evidence_lookup_df,
    on="question_id",
    how="left",
    validate="one_to_one"
)

# 3. Check whether rule-based extraction was correct.
fixed_rag_errors_df["rule_extractor_correct"] = (
    fixed_rag_errors_df["Rule-based extraction"].fillna(False).astype(bool)
)

# 4. Compare selected passages, handling possible merge suffixes.
def get_passage_column(df, base_name):
    for candidate in [
        base_name,
        f"{base_name}_x",
        f"{base_name}_y"
    ]:
        if candidate in df.columns:
            return df[candidate]
    return pd.Series(index=df.index, dtype="object")

fixed_ids = get_passage_column(
    fixed_rag_errors_df, "fixed_passage_id"
)
adaptive_ids = get_passage_column(
    fixed_rag_errors_df, "adaptive_passage_id"
)

if fixed_ids.notna().any() and adaptive_ids.notna().any():
    fixed_rag_errors_df["evidence_changed"] = (
        fixed_ids.astype(str) != adaptive_ids.astype(str)
    )
else:
    fixed_rag_errors_df["evidence_changed"] = pd.NA
    print("Passage IDs unavailable; evidence comparison skipped.")

# 5. Display available diagnostic columns.
display(fixed_rag_errors_df.head(20))

print("Fixed-RAG errors:", len(fixed_rag_errors_df))
print(
    "Errors also answered correctly by extraction:",
    int(fixed_rag_errors_df["rule_extractor_correct"].sum())
)

# 6. Save diagnostic output.
fixed_rag_errors_df.to_csv(
    "/content/ipl_fixed_rag_error_analysis.csv",
    index=False
)
print("Saved diagnostic CSV.")


,question_id,question_type,question,gold_answer,fixed_passage_id,adaptive_passage_id,answer_no_retrieval,answer_fixed_rag,answer_metadata_assisted,extracted_answer,No retrieval,Fixed BM25 RAG,Metadata-assisted RAG,Rule-based extraction,fixed_evidence,adaptive_evidence,rule_extractor_correct,evidence_changed
0,1359531_innings2_R Tewatia_runs,batter_runs,How many runs did R Tewatia score for Gujarat ...,14,1359531_innings2_R Tewatia,1359531_innings2_R Tewatia,69,0,0,14,False,False,False,True,IPL match ID: 1359531. Date: 2023-05-12. Seaso...,IPL match ID: 1359531. Date: 2023-05-12. Seaso...,True,False
1,1082650_innings2_DT Christian_runs,batter_runs,How many runs did DT Christian score for Risin...,4,1082650_innings2_DT Christian,1082650_innings2_DT Christian,0,0,0,4,False,False,False,True,IPL match ID: 1082650. Date: 2017-05-21. Seaso...,IPL match ID: 1082650. Date: 2017-05-21. Seaso...,True,False
2,1216512_innings3_runs,innings_runs,How many runs did Sunrisers Hyderabad score in...,2,1216512_summary,1216512_summary,0,0,0,2,False,False,False,True,IPL match ID: 1216512. Date: 2020-10-18. Seaso...,IPL match ID: 1216512. Date: 2020-10-18. Seaso...,True,False
3,1527681_winner,match_winner,Which team won the IPL match between Mumbai In...,Delhi Capitals,1527681_summary,1527681_summary,Mumbai Indians,Mumbai Indians,Mumbai Indians,Delhi Capitals,False,False,False,True,IPL match ID: 1527681. Date: 2026-04-04. Seaso...,IPL match ID: 1527681. Date: 2026-04-04. Seaso...,True,False
4,598017_innings2_KL Rahul_runs,batter_runs,How many runs did KL Rahul score for Royal Cha...,12,598017_innings2_KL Rahul,598017_innings2_KL Rahul,0,598017,598017,12,False,False,False,True,IPL match ID: 598017. Date: 2013-04-16. Season...,IPL match ID: 598017. Date: 2013-04-16. Season...,True,False
5,980943_winner,match_winner,Which team won the IPL match between Sunrisers...,Rising Pune Supergiants,980943_summary,980943_summary,san juan,Sunrisers Hyderabad,Sunrisers Hyderabad,Rising Pune Supergiants,False,False,False,True,IPL match ID: 980943. Date: 2016-04-26. Season...,IPL match ID: 980943. Date: 2016-04-26. Season...,True,False
6,1216522_innings2_Gurkeerat Singh_runs,batter_runs,How many runs did Gurkeerat Singh score for Ro...,19,1216522_innings2_Gurkeerat Singh,1216522_innings2_Gurkeerat Singh,69,0,0,19,False,False,False,True,IPL match ID: 1216522. Date: 2020-10-17. Seaso...,IPL match ID: 1216522. Date: 2020-10-17. Seaso...,True,False
7,1359513_winner,match_winner,Which team won the IPL match between Kolkata K...,Gujarat Titans,1359513_summary,1359513_summary,Kolkata Knight Riders,Kolkata Knight Riders,Kolkata Knight Riders,Gujarat Titans,False,False,False,True,IPL match ID: 1359513. Date: 2023-04-29. Seaso...,IPL match ID: 1359513. Date: 2023-04-29. Seaso...,True,False
8,1529272_innings2_A Badoni_runs,batter_runs,How many runs did A Badoni score for Lucknow S...,35,1529272_innings2_A Badoni,1529272_innings2_A Badoni,0,69,69,35,False,False,False,True,IPL match ID: 1529272. Date: 2026-04-19. Seaso...,IPL match ID: 1529272. Date: 2026-04-19. Seaso...,True,False
9,392226_winner,match_winner,Which team won the IPL match between Delhi Dar...,Kings XI Punjab,392226_summary,392226_summary,shanghai,Delhi Daredevils,Delhi Daredevils,Kings XI Punjab,False,False,False,True,IPL match ID: 392226. Date: 2009-05-15. Season...,IPL match ID: 392226. Date: 2009-05-15. Season...,True,False


Fixed-RAG errors: 27
Errors also answered correctly by extraction: 26
Saved diagnostic CSV.


In [74]:
# STEP 65A: INSPECT COLUMNS AFTER THE MERGE
# Purpose:
# 1. Check which passage ID columns already exist.
# 2. Identify any merge suffixes.
# 3. Avoid overwriting or duplicating existing results.

print("comparison_df columns:")
print(comparison_df.columns.tolist())

print("\ngeneration_inputs_df columns:")
print(generation_inputs_df.columns.tolist())

comparison_df columns:
['question_id', 'question_type', 'question', 'gold_answer', 'fixed_passage_id', 'adaptive_passage_id', 'answer_no_retrieval', 'answer_fixed_rag', 'answer_metadata_assisted', 'extracted_answer', 'No retrieval', 'Fixed BM25 RAG', 'Metadata-assisted RAG', 'Rule-based extraction']

generation_inputs_df columns:
['question_id', 'question_type', 'question', 'gold_answer', 'no_retrieval_evidence', 'fixed_passage_id', 'fixed_evidence', 'adaptive_passage_id', 'adaptive_evidence']


In [75]:
# STEP 65B: DIAGNOSE FIXED-RAG ANSWER ERRORS
# Purpose:
# 1. Reuse the existing comparison results.
# 2. Add evidence text without duplicating passage ID columns.
# 3. Identify LLM errors that rule-based extraction can recover.

# Step 1: Keep only questions where fixed BM25 RAG was incorrect.
fixed_rag_errors_df = comparison_df[
    ~comparison_df["Fixed BM25 RAG"]
].copy()

# Step 2: Add evidence text only, avoiding duplicate passage ID columns.
evidence_lookup_df = generation_inputs_df[
    ["question_id", "fixed_evidence", "adaptive_evidence"]
].drop_duplicates(subset=["question_id"])

fixed_rag_errors_df = fixed_rag_errors_df.merge(
    evidence_lookup_df,
    on="question_id",
    how="left",
    validate="one_to_one"
)

# Step 3: Check whether rule-based extraction got these answers right.
fixed_rag_errors_df["rule_extractor_correct"] = (
    fixed_rag_errors_df["Rule-based extraction"]
)

# Step 4: Check whether fixed and metadata-assisted retrieval selected
# different passages. The IDs already exist in comparison_df.
fixed_rag_errors_df["evidence_changed"] = (
    fixed_rag_errors_df["fixed_passage_id"].astype(str)
    != fixed_rag_errors_df["adaptive_passage_id"].astype(str)
)

# Step 5: Display the errors for inspection.
display(
    fixed_rag_errors_df[
        [
            "question_id",
            "question_type",
            "question",
            "gold_answer",
            "answer_fixed_rag",
            "extracted_answer",
            "rule_extractor_correct",
            "fixed_passage_id",
            "adaptive_passage_id",
            "evidence_changed"
        ]
    ].reset_index(drop=True)
)

# Step 6: Print diagnostic counts.
print("Fixed-RAG errors:", len(fixed_rag_errors_df))
print(
    "Errors recoverable by rule extraction:",
    int(fixed_rag_errors_df["rule_extractor_correct"].sum())
)
print(
    "Errors where the selected evidence changed:",
    int(fixed_rag_errors_df["evidence_changed"].sum())
)

# Step 7: Save the diagnostic table.
fixed_rag_errors_df.to_csv(
    "/content/ipl_fixed_rag_error_analysis.csv",
    index=False
)

print("Saved: /content/ipl_fixed_rag_error_analysis.csv")

,question_id,question_type,question,gold_answer,answer_fixed_rag,extracted_answer,rule_extractor_correct,fixed_passage_id,adaptive_passage_id,evidence_changed
0,1359531_innings2_R Tewatia_runs,batter_runs,How many runs did R Tewatia score for Gujarat ...,14,0,14,True,1359531_innings2_R Tewatia,1359531_innings2_R Tewatia,False
1,1082650_innings2_DT Christian_runs,batter_runs,How many runs did DT Christian score for Risin...,4,0,4,True,1082650_innings2_DT Christian,1082650_innings2_DT Christian,False
2,1216512_innings3_runs,innings_runs,How many runs did Sunrisers Hyderabad score in...,2,0,2,True,1216512_summary,1216512_summary,False
3,1527681_winner,match_winner,Which team won the IPL match between Mumbai In...,Delhi Capitals,Mumbai Indians,Delhi Capitals,True,1527681_summary,1527681_summary,False
4,598017_innings2_KL Rahul_runs,batter_runs,How many runs did KL Rahul score for Royal Cha...,12,598017,12,True,598017_innings2_KL Rahul,598017_innings2_KL Rahul,False
5,980943_winner,match_winner,Which team won the IPL match between Sunrisers...,Rising Pune Supergiants,Sunrisers Hyderabad,Rising Pune Supergiants,True,980943_summary,980943_summary,False
6,1216522_innings2_Gurkeerat Singh_runs,batter_runs,How many runs did Gurkeerat Singh score for Ro...,19,0,19,True,1216522_innings2_Gurkeerat Singh,1216522_innings2_Gurkeerat Singh,False
7,1359513_winner,match_winner,Which team won the IPL match between Kolkata K...,Gujarat Titans,Kolkata Knight Riders,Gujarat Titans,True,1359513_summary,1359513_summary,False
8,1529272_innings2_A Badoni_runs,batter_runs,How many runs did A Badoni score for Lucknow S...,35,69,35,True,1529272_innings2_A Badoni,1529272_innings2_A Badoni,False
9,392226_winner,match_winner,Which team won the IPL match between Delhi Dar...,Kings XI Punjab,Delhi Daredevils,Kings XI Punjab,True,392226_summary,392226_summary,False


Fixed-RAG errors: 27
Errors recoverable by rule extraction: 26
Errors where the selected evidence changed: 1
Saved: /content/ipl_fixed_rag_error_analysis.csv


In [76]:
# STEP 66: INSPECT THE REMAINING EXTRACTION AND RETRIEVAL ISSUES
# Purpose:
# 1. Find the fixed-RAG errors not recovered by rule extraction.
# 2. Inspect their questions, answers, and evidence passages.
# 3. Determine whether the problem is retrieval or extraction.

# Step 1: Select errors that rule extraction did not recover.
unresolved_df = fixed_rag_errors_df[
    ~fixed_rag_errors_df["rule_extractor_correct"]
].copy()

print("Unresolved cases:", len(unresolved_df))

# Step 2: Display the relevant details.
for _, row in unresolved_df.iterrows():
    print("\n" + "=" * 80)
    print("Question ID:", row["question_id"])
    print("Question type:", row["question_type"])
    print("Question:", row["question"])
    print("Gold answer:", row["gold_answer"])
    print("Fixed-RAG answer:", row["answer_fixed_rag"])
    print("Rule-extracted answer:", row["extracted_answer"])
    print("Fixed passage ID:", row["fixed_passage_id"])
    print("Adaptive passage ID:", row["adaptive_passage_id"])
    print("\nFixed evidence:")
    print(row["fixed_evidence"])
    print("\nMetadata-assisted evidence:")
    print(row["adaptive_evidence"])

Unresolved cases: 1

Question ID: 392188_innings1_runs
Question type: innings_runs
Question: How many runs did Deccan Chargers score in innings 1 of the IPL match played on 2009-04-22?
Gold answer: 184
Fixed-RAG answer: 63
Rule-extracted answer: None
Fixed passage ID: 392237_summary
Adaptive passage ID: 392188_summary

Fixed evidence:
IPL match ID: 392237. Date: 2009-05-22. Season: 2009. Teams: Delhi Daredevils, Deccan Chargers. Venue: SuperSport Park. City: Centurion. Winner: Deccan Chargers. Toss winner: Deccan Chargers. Toss decision: field. Player of the match: AC Gilchrist. Delhi Daredevils scored 153 runs and lost 8 wickets in innings 1. Deccan Chargers scored 154 runs and lost 4 wickets in innings 2.

Metadata-assisted evidence:
IPL match ID: 392188. Date: 2009-04-22. Season: 2009. Teams: Royal Challengers Bangalore, Deccan Chargers. Venue: Newlands. City: Cape Town. Winner: Deccan Chargers. Toss winner: Deccan Chargers. Toss decision: bat. Player of the match: AC Gilchrist. Dec

In [77]:
# STEP 67: EVALUATE HYBRID RETRIEVAL PLUS RULE-BASED EXTRACTION
# Purpose:
# 1. Extract answers from metadata-assisted evidence.
# 2. Compare those answers with the same 60 gold answers.
# 3. Measure answer coverage and accuracy by question type.
# 4. Save the results separately from the existing experiments.

hybrid_records = []

for _, row in generation_inputs_df.iterrows():
    extracted_answer = extract_answer_from_evidence_v2(
        str(row["question_type"]),
        str(row["adaptive_evidence"]),
        str(row["question"])
    )

    gold_answer = str(row["gold_answer"])

    is_correct = (
        score_answer(
            str(row["question_type"]),
            extracted_answer,
            gold_answer
        )
        if extracted_answer is not None
        else False
    )

    hybrid_records.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "gold_answer": gold_answer,
        "adaptive_passage_id": row["adaptive_passage_id"],
        "hybrid_answer": extracted_answer,
        "answer_found": extracted_answer is not None,
        "correct": is_correct
    })

hybrid_pilot_df = pd.DataFrame(hybrid_records)

print("Total questions:", len(hybrid_pilot_df))
print("Answers extracted:", int(hybrid_pilot_df["answer_found"].sum()))
print("Correct answers:", int(hybrid_pilot_df["correct"].sum()))
print("Overall accuracy:", round(hybrid_pilot_df["correct"].mean(), 4))

hybrid_summary_df = (
    hybrid_pilot_df
    .groupby("question_type")
    .agg(
        questions=("question_id", "count"),
        answers_extracted=("answer_found", "sum"),
        correct_answers=("correct", "sum")
    )
    .reset_index()
)

hybrid_summary_df["accuracy"] = (
    hybrid_summary_df["correct_answers"]
    / hybrid_summary_df["questions"]
)

display(hybrid_summary_df)

hybrid_pilot_df.to_csv(
    "/content/ipl_hybrid_extraction_pilot_60.csv",
    index=False
)

hybrid_summary_df.to_csv(
    "/content/ipl_hybrid_extraction_pilot_summary.csv",
    index=False
)

print("Saved hybrid extraction results.")

Total questions: 60
Answers extracted: 60
Correct answers: 60
Overall accuracy: 1.0


,question_type,questions,answers_extracted,correct_answers,accuracy
0,batter_runs,20,20,20,1.0
1,innings_runs,20,20,20,1.0
2,match_winner,20,20,20,1.0


Saved hybrid extraction results.


In [78]:
# STEP 68A: CONSOLIDATE THE 60-QUESTION PILOT RESULTS
# Purpose:
# 1. Put the five evaluated methods into one table.
# 2. Record correct answers, total questions, and accuracy.
# 3. Save a compact experiment summary for reproducibility.

pilot_methods = [
    {
        "method": "No retrieval + FLAN-T5-small",
        "correct_answers": 5,
        "total_questions": 60
    },
    {
        "method": "Fixed BM25 RAG + FLAN-T5-small",
        "correct_answers": 33,
        "total_questions": 60
    },
    {
        "method": "Metadata-assisted RAG + FLAN-T5-small",
        "correct_answers": 34,
        "total_questions": 60
    },
    {
        "method": "Fixed BM25 + rule-based extraction",
        "correct_answers": int(rule_pilot_df["correct"].sum()),
        "total_questions": len(rule_pilot_df)
    },
    {
        "method": "Metadata-assisted retrieval + rule-based extraction",
        "correct_answers": int(hybrid_pilot_df["correct"].sum()),
        "total_questions": len(hybrid_pilot_df)
    }
]

pilot_results_summary_df = pd.DataFrame(pilot_methods)
pilot_results_summary_df["accuracy"] = (
    pilot_results_summary_df["correct_answers"]
    / pilot_results_summary_df["total_questions"]
)
pilot_results_summary_df["accuracy_percent"] = (
    pilot_results_summary_df["accuracy"] * 100
).round(2)

display(pilot_results_summary_df)

pilot_results_summary_df.to_csv(
    "/content/ipl_pilot_results_consolidated.csv",
    index=False
)

print("Saved: /content/ipl_pilot_results_consolidated.csv")

,method,correct_answers,total_questions,accuracy,accuracy_percent
0,No retrieval + FLAN-T5-small,5,60,0.083333,8.33
1,Fixed BM25 RAG + FLAN-T5-small,33,60,0.550000,55.00
2,Metadata-assisted RAG + FLAN-T5-small,34,60,0.566667,56.67
3,Fixed BM25 + rule-based extraction,59,60,0.983333,98.33
4,Metadata-assisted retrieval + rule-based extra...,60,60,1.000000,100.00


Saved: /content/ipl_pilot_results_consolidated.csv


In [79]:
# STEP 70: FREEZE THE METHOD AND SELECT 180 FRESH EVALUATION QUESTIONS
# Purpose:
# 1. Exclude the 60 questions used in the development pilot.
# 2. Select 60 questions from each of the three question types.
# 3. Use a fixed random seed for reproducibility.
# 4. Save the evaluation IDs before running any new experiments.

import pandas as pd

# Step 1: Identify the pilot questions that must be excluded.
pilot_question_ids = set(generation_pilot_df["question_id"].astype(str))

# Step 2: Prepare the eligible pool from the existing held-out test split.
eligible_questions_df = test_questions_df[
    ~test_questions_df["question_id"].astype(str).isin(pilot_question_ids)
].copy()

print("Original held-out test questions:", len(test_questions_df))
print("Pilot questions excluded:", len(pilot_question_ids))
print("Eligible questions remaining:", len(eligible_questions_df))

# Step 3: Confirm that every question type has enough eligible questions.
required_types = ["batter_runs", "innings_runs", "match_winner"]

available_counts = eligible_questions_df["question_type"].value_counts()
print("\nAvailable questions by type:")
print(available_counts.reindex(required_types, fill_value=0))

assert all(
    available_counts.get(question_type, 0) >= 60
    for question_type in required_types
), "Not enough eligible questions in at least one category."

# Step 4: Sample 60 questions per category using a fixed seed.
fresh_eval_df = (
    eligible_questions_df[
        eligible_questions_df["question_type"].isin(required_types)
    ]
    .groupby("question_type", group_keys=False)
    .sample(n=60, random_state=20261028)
    .sort_values(["question_type", "question_id"])
    .reset_index(drop=True)
)

# Step 5: Validate the sample.
print("\nFresh evaluation sample size:", len(fresh_eval_df))
print("\nSample counts:")
print(fresh_eval_df["question_type"].value_counts().sort_index())

assert len(fresh_eval_df) == 180
assert fresh_eval_df["question_id"].is_unique
assert not set(fresh_eval_df["question_id"].astype(str)) & pilot_question_ids

# Step 6: Save the frozen evaluation questions and their IDs.
fresh_eval_df.to_csv(
    "/content/ipl_fresh_evaluation_180_questions.csv",
    index=False
)

print("\nSaved frozen evaluation questions.")
print("Random seed: 20261028")

Original held-out test questions: 4545
Pilot questions excluded: 60
Eligible questions remaining: 4485

Available questions by type:
question_type
batter_runs     3777
innings_runs     485
match_winner     223
Name: count, dtype: int64

Fresh evaluation sample size: 180

Sample counts:
question_type
batter_runs     60
innings_runs    60
match_winner    60
Name: count, dtype: int64

Saved frozen evaluation questions.
Random seed: 20261028


In [80]:
# STEP 71: FREEZE AND VERIFY THE EXPERIMENT CONFIGURATION
# Purpose:
# 1. Record the fixed model and retrieval settings.
# 2. Verify the evaluation questions have the required fields.
# 3. Confirm the BM25 indexes and passage stores exist for each split.
# 4. Save the configuration for reproducibility.

import json
import os

# Step 1: Record the experiment configuration.
frozen_config = {
    "experiment": "IPL adaptive RAG fresh evaluation",
    "evaluation_sample_size": 180,
    "questions_per_type": 60,
    "random_seed": 20261028,
    "excluded_development_pilot_size": 60,
    "language_model": "google/flan-t5-small",
    "retrieval_method": "BM25",
    "retrieval_top_k": 3,
    "adaptive_innings_runs_candidate_pool": 10,
    "adaptive_batter_runs_candidate_pool": 3,
    "winner_retrieval_rule": "fixed BM25 top-1",
    "extraction_method": "extract_answer_from_evidence_v2",
    "answer_generation": {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 32
    }
}

# Step 2: Verify required evaluation columns.
required_columns = [
    "question_id",
    "question_type",
    "question",
    "gold_answer",
    "match_id",
    "split"
]

missing_columns = [
    column for column in required_columns
    if column not in fresh_eval_df.columns
]

assert not missing_columns, f"Missing evaluation columns: {missing_columns}"
assert len(fresh_eval_df) == 180
assert fresh_eval_df["question_id"].is_unique

# Step 3: Verify each retrieval index and passage store exists.
required_index_keys = [
    ("test", "match_summary"),
    ("test", "player_stat")
]

for key in required_index_keys:
    assert key in bm25_indexes, f"Missing BM25 index: {key}"
    assert key in bm25_passage_stores, f"Missing passage store: {key}"

# Step 4: Save the frozen configuration and evaluation IDs.
config_path = "/content/ipl_fresh_evaluation_frozen_config.json"

with open(config_path, "w") as file:
    json.dump(frozen_config, file, indent=2)

fresh_eval_df[["question_id"]].to_csv(
    "/content/ipl_fresh_evaluation_frozen_ids.csv",
    index=False
)

print("Configuration checks passed.")
print("Evaluation questions:", len(fresh_eval_df))
print("Question types:", fresh_eval_df["question_type"].value_counts().to_dict())
print("BM25 indexes and passage stores: available")
print("Configuration saved:", config_path)
print("Frozen IDs saved: /content/ipl_fresh_evaluation_frozen_ids.csv")

Configuration checks passed.
Evaluation questions: 180
Question types: {'batter_runs': 60, 'innings_runs': 60, 'match_winner': 60}
BM25 indexes and passage stores: available
Configuration saved: /content/ipl_fresh_evaluation_frozen_config.json
Frozen IDs saved: /content/ipl_fresh_evaluation_frozen_ids.csv


In [81]:
# STEP 72: RETRIEVE FIXED AND METADATA-ASSISTED EVIDENCE
# Purpose:
# 1. Retrieve fixed BM25 top-1 evidence for all 180 questions.
# 2. Apply the frozen metadata-assisted selection rules.
# 3. Record evidence IDs and texts for both conditions.
# 4. Evaluate target-passage inclusion without using gold IDs for selection.

def retrieve_fresh_fixed_and_adaptive(row):
    question_type = str(row["question_type"])
    question = str(row["question"])

    # Step 1: Retrieve the top-three candidates as a DataFrame.
    candidates = retrieve_top_three(row)

    if candidates is None or candidates.empty:
        return {
            "fixed_passage_id": None,
            "fixed_evidence": None,
            "adaptive_passage_id": None,
            "adaptive_evidence": None,
            "adaptive_action": "no_candidates"
        }

    # Step 2: Read the top-ranked passage.
    fixed_row = candidates.iloc[0]
    fixed_id = str(fixed_row["passage_id"])
    fixed_text = str(fixed_row["text"])
    selected = fixed_row
    action = "use_top1"

    # Step 3: For batter runs, require date, player, and innings to match.
    if question_type == "batter_runs":
        requested_date = extract_question_date(question)
        requested_batter = extract_batter_name(question)
        requested_innings = extract_innings_number(question)

        for _, candidate in candidates.head(3).iterrows():
            candidate_id = str(candidate["passage_id"])
            candidate_text = str(candidate["text"])

            date_ok = (
                requested_date is not None
                and requested_date in candidate_text
            )
            batter_ok = (
                requested_batter is not None
                and requested_batter.lower() in candidate_text.lower()
            )
            innings_ok = (
                requested_innings is not None
                and extract_passage_innings(candidate_id) == requested_innings
            )

            if date_ok and batter_ok and innings_ok:
                selected = candidate
                action = (
                    "use_top1"
                    if candidate_id == fixed_id
                    else "fallback_top3"
                )
                break

    # Step 4: For innings totals, search the top 10 BM25 candidates.
    elif question_type == "innings_runs":
        index = bm25_indexes[("test", "match_summary")]
        store = bm25_passage_stores[("test", "match_summary")]

        scores = index.get_scores(tokenize_text(question))
        ranked_positions = sorted(
            range(len(scores)),
            key=lambda position: scores[position],
            reverse=True
        )[:10]

        requested_date = extract_question_date(question)
        requested_team = extract_batting_team(question)
        requested_innings = extract_innings_number(question)

        selected = fixed_row
        found_match = False

        innings_pattern = (
            r"([^.]+?)\s+scored\s+(\d+)\s+runs"
            r"\s+and\s+lost\s+\d+\s+wickets\s+in\s+innings\s+(\d+)"
        )

        for position in ranked_positions:
            candidate = store.iloc[position]
            candidate_text = str(candidate["text"])

            if (
                requested_date is None
                or requested_team is None
                or requested_innings is None
                or requested_date not in candidate_text
            ):
                continue

            for match in re.finditer(
                innings_pattern,
                candidate_text,
                flags=re.IGNORECASE
            ):
                passage_team = match.group(1).strip()
                passage_innings = int(match.group(3))

                if (
                    passage_team.lower() == requested_team.lower()
                    and passage_innings == requested_innings
                ):
                    selected = candidate
                    found_match = True
                    break

            if found_match:
                break

        if found_match:
            action = (
                "use_top1"
                if str(selected["passage_id"]) == fixed_id
                else "metadata_fallback_top10"
            )
        else:
            selected = fixed_row
            action = "fallback_top1_no_metadata_match"

    # Step 5: Winner questions retain fixed BM25 top-1.
    return {
        "fixed_passage_id": fixed_id,
        "fixed_evidence": fixed_text,
        "adaptive_passage_id": str(selected["passage_id"]),
        "adaptive_evidence": str(selected["text"]),
        "adaptive_action": action
    }


# Step 6: Test the corrected function on one question only.
single_test_result = retrieve_fresh_fixed_and_adaptive(
    fresh_eval_df.iloc[0]
)

print("Single-question retrieval test:")
print("Question:", fresh_eval_df.iloc[0]["question"])
print("Fixed passage:", single_test_result["fixed_passage_id"])
print("Adaptive passage:", single_test_result["adaptive_passage_id"])
print("Adaptive action:", single_test_result["adaptive_action"])
print("Evidence returned:", single_test_result["adaptive_evidence"] is not None)

Single-question retrieval test:
Question: How many runs did V Shankar score for Sunrisers Hyderabad in innings 1 of the IPL match played on 2017-05-17?
Fixed passage: 1082648_innings1_V Shankar
Adaptive passage: 1082648_innings1_V Shankar
Adaptive action: use_top1
Evidence returned: True


In [82]:
# STEP 72D: INSPECT RETRIEVAL OUTPUT FOR ONE QUESTION
# Purpose:
# 1. Test retrieval on a single frozen evaluation question.
# 2. Identify whether retrieve_top_three returns a DataFrame or list.
# 3. Avoid running the full 180-question experiment until verified.

# Step 1: Select the first frozen evaluation question.
test_row = fresh_eval_df.iloc[0]

print("Question:", test_row["question"])
print("Question type:", test_row["question_type"])

# Step 2: Retrieve candidates for this question.
test_candidates = retrieve_top_three(test_row)

# Step 3: Inspect the result safely.
print("\nReturned type:", type(test_candidates))
print("Is None:", test_candidates is None)

if test_candidates is None:
    print("No candidates returned.")
elif isinstance(test_candidates, pd.DataFrame):
    print("Is empty:", test_candidates.empty)
    print("Shape:", test_candidates.shape)
    display(test_candidates.head(3))
elif isinstance(test_candidates, list):
    print("Number of candidates:", len(test_candidates))
    print(test_candidates[:3])
else:
    print("Returned object:")
    print(test_candidates)

Question: How many runs did V Shankar score for Sunrisers Hyderabad in innings 1 of the IPL match played on 2017-05-17?
Question type: batter_runs

Returned type: <class 'pandas.core.frame.DataFrame'>
Is None: False
Is empty: False
Shape: (3, 7)


,passage_id,match_id,innings_number,batter,passage_type,text,split
1555,1082648_innings1_V Shankar,1082648,1,V Shankar,player_stat,IPL match ID: 1082648. Date: 2017-05-17. Seaso...,test
3342,1082643_innings2_V Shankar,1082643,2,V Shankar,player_stat,IPL match ID: 1082643. Date: 2017-05-13. Seaso...,test
2389,1254066_innings2_V Shankar,1254066,2,V Shankar,player_stat,IPL match ID: 1254066. Date: 2021-04-17. Seaso...,test


In [83]:
# STEP 72F: RUN THE FROZEN RETRIEVAL EXPERIMENT
# Purpose:
# 1. Process all 180 frozen evaluation questions.
# 2. Record fixed and metadata-assisted evidence.
# 3. Evaluate exact target-passage hits after retrieval.
# 4. Save the results for reproducibility.

fresh_retrieval_records = []

# Step 1: Retrieve evidence for each question.
for _, row in fresh_eval_df.iterrows():
    result = retrieve_fresh_fixed_and_adaptive(row)

    fresh_retrieval_records.append({
        "question_id": row["question_id"],
        **result
    })

# Step 2: Build the retrieval results table.
fresh_retrieval_df = pd.DataFrame(fresh_retrieval_records)

# Step 3: Attach gold target IDs for evaluation only.
fresh_retrieval_df = fresh_retrieval_df.merge(
    fresh_eval_df[
        ["question_id", "target_passage_id", "question_type"]
    ],
    on="question_id",
    how="left",
    validate="one_to_one"
)

# Step 4: Calculate exact target-passage hits.
fresh_retrieval_df["fixed_target_hit"] = (
    fresh_retrieval_df["fixed_passage_id"].astype(str)
    == fresh_retrieval_df["target_passage_id"].astype(str)
)

fresh_retrieval_df["adaptive_target_hit"] = (
    fresh_retrieval_df["adaptive_passage_id"].astype(str)
    == fresh_retrieval_df["target_passage_id"].astype(str)
)

# Step 5: Report the overall results.
print("Questions processed:", len(fresh_retrieval_df))
print(
    "Fixed BM25 top-1 target hits:",
    int(fresh_retrieval_df["fixed_target_hit"].sum()),
    "/",
    len(fresh_retrieval_df)
)
print(
    "Metadata-assisted target hits:",
    int(fresh_retrieval_df["adaptive_target_hit"].sum()),
    "/",
    len(fresh_retrieval_df)
)

# Step 6: Summarise results by question type.
retrieval_summary_df = (
    fresh_retrieval_df
    .groupby("question_type")
    .agg(
        questions=("question_id", "count"),
        fixed_target_hits=("fixed_target_hit", "sum"),
        adaptive_target_hits=("adaptive_target_hit", "sum")
    )
    .reset_index()
)

retrieval_summary_df["fixed_hit_rate"] = (
    retrieval_summary_df["fixed_target_hits"]
    / retrieval_summary_df["questions"]
)

retrieval_summary_df["adaptive_hit_rate"] = (
    retrieval_summary_df["adaptive_target_hits"]
    / retrieval_summary_df["questions"]
)

display(retrieval_summary_df)

# Step 7: Show how often each adaptive decision was used.
print("\nAdaptive actions:")
print(fresh_retrieval_df["adaptive_action"].value_counts(dropna=False))

# Step 8: Save detailed and summary results.
fresh_retrieval_df.to_csv(
    "/content/ipl_fresh_evaluation_retrieval_results.csv",
    index=False
)

retrieval_summary_df.to_csv(
    "/content/ipl_fresh_evaluation_retrieval_summary.csv",
    index=False
)

print("\nSaved retrieval results and summary.")

Questions processed: 180
Fixed BM25 top-1 target hits: 179 / 180
Metadata-assisted target hits: 180 / 180


,question_type,questions,fixed_target_hits,adaptive_target_hits,fixed_hit_rate,adaptive_hit_rate
0,batter_runs,60,60,60,1.000000,1.0
1,innings_runs,60,59,60,0.983333,1.0
2,match_winner,60,60,60,1.000000,1.0



Adaptive actions:
adaptive_action
use_top1                   179
metadata_fallback_top10      1
Name: count, dtype: int64

Saved retrieval results and summary.


In [84]:
# STEP 73: VALIDATE FRESH EVALUATION EVIDENCE
# Purpose:
# 1. Confirm that both retrieval conditions have evidence for every question.
# 2. Inspect the question where metadata-assisted retrieval changed the passage.
# 3. Verify that the selected adaptive passage matches the target passage.

# Step 1: Check evidence coverage.
fresh_retrieval_df["fixed_evidence_available"] = (
    fresh_retrieval_df["fixed_evidence"].notna()
    & fresh_retrieval_df["fixed_evidence"].astype(str).str.strip().ne("")
)

fresh_retrieval_df["adaptive_evidence_available"] = (
    fresh_retrieval_df["adaptive_evidence"].notna()
    & fresh_retrieval_df["adaptive_evidence"].astype(str).str.strip().ne("")
)

print("Total questions:", len(fresh_retrieval_df))
print(
    "Fixed evidence available:",
    int(fresh_retrieval_df["fixed_evidence_available"].sum())
)
print(
    "Adaptive evidence available:",
    int(fresh_retrieval_df["adaptive_evidence_available"].sum())
)

# Step 2: Identify changed retrieval decisions.
changed_evidence_df = fresh_retrieval_df[
    fresh_retrieval_df["fixed_passage_id"].astype(str)
    != fresh_retrieval_df["adaptive_passage_id"].astype(str)
].copy()

print("\nQuestions with changed evidence:", len(changed_evidence_df))

# Step 3: Inspect the changed case, if one exists.
if len(changed_evidence_df) > 0:
    display(
        changed_evidence_df[
            [
                "question_id",
                "question_type",
                "fixed_passage_id",
                "adaptive_passage_id",
                "target_passage_id",
                "fixed_target_hit",
                "adaptive_target_hit",
                "fixed_evidence",
                "adaptive_evidence"
            ]
        ]
    )
else:
    print("No changed retrieval decisions.")

# Step 4: Assert that the evidence is available for every question.
assert fresh_retrieval_df["fixed_evidence_available"].all()
assert fresh_retrieval_df["adaptive_evidence_available"].all()

print("\nEvidence validation passed.")

Total questions: 180
Fixed evidence available: 180
Adaptive evidence available: 180

Questions with changed evidence: 1


,question_id,question_type,fixed_passage_id,adaptive_passage_id,target_passage_id,fixed_target_hit,adaptive_target_hit,fixed_evidence,adaptive_evidence
67,1254062_innings2_runs,innings_runs,1254116_summary,1254062_summary,1254062_summary,False,True,IPL match ID: 1254116. Date: 2021-10-13. Seaso...,IPL match ID: 1254062. Date: 2021-04-13. Seaso...



Evidence validation passed.


In [85]:
# STEP 74: GENERATE ANSWERS FOR THE FRESH 180-QUESTION EVALUATION
# Purpose:
# 1. Generate no-retrieval, fixed-RAG, and metadata-assisted answers.
# 2. Use the same model and deterministic generation settings.
# 3. Save all answers before calculating accuracy.
# 4. Avoid modifying the frozen retrieval rules.

import torch
import pandas as pd

# Step 1: Define the frozen generation function.
def generate_fresh_answer(question, evidence=None):
    if evidence is None:
        prompt = (
            "Answer the following IPL cricket question as briefly as possible. "
            "If you do not know the answer, say unknown.\n"
            f"Question: {question}\nAnswer:"
        )
    else:
        prompt = (
            "Answer the IPL question using only the evidence provided. "
            "Return only the requested answer. "
            "For a match winner, return the value after Winner:. "
            "Do not confuse the winner with the toss winner.\n"
            f"Evidence: {evidence}\n"
            f"Question: {question}\n"
            "Answer:"
        )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=False,
            num_beams=1,
            max_new_tokens=32
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()


# Step 2: Generate answers for each frozen question.
fresh_generation_records = []

for i, (_, row) in enumerate(fresh_eval_df.iterrows(), start=1):
    evidence_row = fresh_retrieval_df[
        fresh_retrieval_df["question_id"] == row["question_id"]
    ].iloc[0]

    question = str(row["question"])

    answer_no_retrieval = generate_fresh_answer(question)
    answer_fixed_rag = generate_fresh_answer(
        question,
        str(evidence_row["fixed_evidence"])
    )
    answer_metadata_assisted = generate_fresh_answer(
        question,
        str(evidence_row["adaptive_evidence"])
    )

    fresh_generation_records.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "question": question,
        "gold_answer": str(row["gold_answer"]),
        "fixed_passage_id": evidence_row["fixed_passage_id"],
        "adaptive_passage_id": evidence_row["adaptive_passage_id"],
        "answer_no_retrieval": answer_no_retrieval,
        "answer_fixed_rag": answer_fixed_rag,
        "answer_metadata_assisted": answer_metadata_assisted
    })

    if i % 15 == 0:
        print(f"Generated answers for {i}/180 questions.")

# Step 3: Store and save the generated answers.
fresh_generation_df = pd.DataFrame(fresh_generation_records)

fresh_generation_df.to_csv(
    "/content/ipl_fresh_evaluation_generated_answers.csv",
    index=False
)

print("\nGeneration complete.")
print("Questions evaluated:", len(fresh_generation_df))
print("Saved: /content/ipl_fresh_evaluation_generated_answers.csv")

# Step 4: Display a small sample for a sanity check.
display(
    fresh_generation_df[
        [
            "question_type",
            "question",
            "gold_answer",
            "answer_no_retrieval",
            "answer_fixed_rag",
            "answer_metadata_assisted"
        ]
    ].head(10)
)

Generated answers for 15/180 questions.
Generated answers for 30/180 questions.
Generated answers for 45/180 questions.
Generated answers for 60/180 questions.
Generated answers for 75/180 questions.
Generated answers for 90/180 questions.
Generated answers for 105/180 questions.
Generated answers for 120/180 questions.
Generated answers for 135/180 questions.
Generated answers for 150/180 questions.
Generated answers for 165/180 questions.
Generated answers for 180/180 questions.

Generation complete.
Questions evaluated: 180
Saved: /content/ipl_fresh_evaluation_generated_answers.csv


,question_type,question,gold_answer,answer_no_retrieval,answer_fixed_rag,answer_metadata_assisted
0,batter_runs,How many runs did V Shankar score for Sunriser...,22,0,22,22
1,batter_runs,How many runs did KA Pollard score for Mumbai ...,28,0,28,28
2,batter_runs,How many runs did P Chopra score for Rajasthan...,8,0,8,8
3,batter_runs,How many runs did RR Pant score for Delhi Capi...,25,0,25,25
4,batter_runs,How many runs did CA Ingram score for Delhi Ca...,1,0,0,0
5,batter_runs,How many runs did N Rana score for Kolkata Kni...,26,0,26,26
6,batter_runs,How many runs did R Tewatia score for Rajastha...,5,0,5,5
7,batter_runs,How many runs did SM Curran score for Chennai ...,34,0,34,34
8,batter_runs,How many runs did RA Tripathi score for Kolkat...,5,0,5,5
9,batter_runs,How many runs did EJG Morgan score for Kolkata...,47,0,47,47


In [86]:
# STEP 75: SCORE THE FRESH 180-QUESTION GENERATION RESULTS
# Purpose:
# 1. Score all three LLM conditions consistently.
# 2. Calculate overall and per-question-type accuracy.
# 3. Save the results without changing the frozen methods.

# Step 1: Define the three answer conditions.
fresh_method_columns = {
    "No retrieval + FLAN-T5-small": "answer_no_retrieval",
    "Fixed BM25 RAG + FLAN-T5-small": "answer_fixed_rag",
    "Metadata-assisted RAG + FLAN-T5-small": "answer_metadata_assisted"
}

# Step 2: Score every answer against its gold answer.
for method_name, answer_column in fresh_method_columns.items():
    fresh_generation_df[method_name] = fresh_generation_df.apply(
        lambda row: score_answer(
            str(row["question_type"]),
            row[answer_column],
            row["gold_answer"]
        ),
        axis=1
    )

# Step 3: Create the overall summary.
fresh_overall_records = []

for method_name in fresh_method_columns:
    correct = int(fresh_generation_df[method_name].sum())
    total = len(fresh_generation_df)

    fresh_overall_records.append({
        "method": method_name,
        "correct_answers": correct,
        "total_questions": total,
        "accuracy": correct / total,
        "accuracy_percent": round(100 * correct / total, 2)
    })

fresh_overall_summary_df = pd.DataFrame(fresh_overall_records)

print("OVERALL RESULTS")
display(fresh_overall_summary_df)

# Step 4: Calculate accuracy by question type.
fresh_type_records = []

for question_type, group in fresh_generation_df.groupby("question_type"):
    for method_name in fresh_method_columns:
        correct = int(group[method_name].sum())
        total = len(group)

        fresh_type_records.append({
            "question_type": question_type,
            "method": method_name,
            "correct_answers": correct,
            "total_questions": total,
            "accuracy": correct / total,
            "accuracy_percent": round(100 * correct / total, 2)
        })

fresh_type_summary_df = pd.DataFrame(fresh_type_records)

print("\nRESULTS BY QUESTION TYPE")
display(
    fresh_type_summary_df.sort_values(
        ["question_type", "method"]
    ).reset_index(drop=True)
)

# Step 5: Save the scoring results.
fresh_overall_summary_df.to_csv(
    "/content/ipl_fresh_evaluation_generation_summary.csv",
    index=False
)

fresh_type_summary_df.to_csv(
    "/content/ipl_fresh_evaluation_generation_by_type.csv",
    index=False
)

fresh_generation_df.to_csv(
    "/content/ipl_fresh_evaluation_scored_answers.csv",
    index=False
)

print("\nSaved overall summary, per-type summary, and scored answers.")

OVERALL RESULTS


,method,correct_answers,total_questions,accuracy,accuracy_percent
0,No retrieval + FLAN-T5-small,3,180,0.016667,1.67
1,Fixed BM25 RAG + FLAN-T5-small,100,180,0.555556,55.56
2,Metadata-assisted RAG + FLAN-T5-small,101,180,0.561111,56.11



RESULTS BY QUESTION TYPE


,question_type,method,correct_answers,total_questions,accuracy,accuracy_percent
0,batter_runs,Fixed BM25 RAG + FLAN-T5-small,49,60,0.816667,81.67
1,batter_runs,Metadata-assisted RAG + FLAN-T5-small,49,60,0.816667,81.67
2,batter_runs,No retrieval + FLAN-T5-small,3,60,0.050000,5.00
3,innings_runs,Fixed BM25 RAG + FLAN-T5-small,39,60,0.650000,65.00
4,innings_runs,Metadata-assisted RAG + FLAN-T5-small,40,60,0.666667,66.67
5,innings_runs,No retrieval + FLAN-T5-small,0,60,0.000000,0.00
6,match_winner,Fixed BM25 RAG + FLAN-T5-small,12,60,0.200000,20.00
7,match_winner,Metadata-assisted RAG + FLAN-T5-small,12,60,0.200000,20.00
8,match_winner,No retrieval + FLAN-T5-small,0,60,0.000000,0.00



Saved overall summary, per-type summary, and scored answers.


In [87]:
# STEP 77: EVALUATE RULE-BASED AND HYBRID EXTRACTION ON FRESH QUESTIONS
# Purpose:
# 1. Extract answers from fixed BM25 evidence.
# 2. Extract answers from metadata-assisted evidence.
# 3. Score both conditions against the same 180 gold answers.
# 4. Save results without modifying the extraction rules.

# Step 1: Join questions with both evidence conditions.
fresh_extraction_inputs_df = fresh_eval_df[
    ["question_id", "question_type", "question", "gold_answer"]
].merge(
    fresh_retrieval_df[
        [
            "question_id",
            "fixed_evidence",
            "adaptive_evidence",
            "fixed_passage_id",
            "adaptive_passage_id"
        ]
    ],
    on="question_id",
    how="left",
    validate="one_to_one"
)

# Step 2: Apply the same frozen extractor to both evidence conditions.
fresh_extraction_records = []

for _, row in fresh_extraction_inputs_df.iterrows():
    fixed_answer = extract_answer_from_evidence_v2(
        str(row["question_type"]),
        str(row["fixed_evidence"]),
        str(row["question"])
    )

    adaptive_answer = extract_answer_from_evidence_v2(
        str(row["question_type"]),
        str(row["adaptive_evidence"]),
        str(row["question"])
    )

    fresh_extraction_records.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "gold_answer": str(row["gold_answer"]),
        "fixed_passage_id": row["fixed_passage_id"],
        "adaptive_passage_id": row["adaptive_passage_id"],
        "fixed_rule_answer": fixed_answer,
        "adaptive_rule_answer": adaptive_answer,
        "fixed_rule_correct": score_answer(
            str(row["question_type"]),
            fixed_answer,
            row["gold_answer"]
        ),
        "adaptive_rule_correct": score_answer(
            str(row["question_type"]),
            adaptive_answer,
            row["gold_answer"]
        )
    })

fresh_extraction_df = pd.DataFrame(fresh_extraction_records)

# Step 3: Summarise overall results.
print("Total questions:", len(fresh_extraction_df))
print(
    "Fixed-evidence rule extraction:",
    int(fresh_extraction_df["fixed_rule_correct"].sum()),
    "/",
    len(fresh_extraction_df)
)
print(
    "Metadata-assisted hybrid extraction:",
    int(fresh_extraction_df["adaptive_rule_correct"].sum()),
    "/",
    len(fresh_extraction_df)
)

# Step 4: Summarise results by question type.
fresh_extraction_summary_df = (
    fresh_extraction_df
    .groupby("question_type")
    .agg(
        questions=("question_id", "count"),
        fixed_correct=("fixed_rule_correct", "sum"),
        adaptive_correct=("adaptive_rule_correct", "sum")
    )
    .reset_index()
)

fresh_extraction_summary_df["fixed_accuracy"] = (
    fresh_extraction_summary_df["fixed_correct"]
    / fresh_extraction_summary_df["questions"]
)

fresh_extraction_summary_df["adaptive_accuracy"] = (
    fresh_extraction_summary_df["adaptive_correct"]
    / fresh_extraction_summary_df["questions"]
)

display(fresh_extraction_summary_df)

# Step 5: Save the detailed results and summary.
fresh_extraction_df.to_csv(
    "/content/ipl_fresh_evaluation_extraction_results.csv",
    index=False
)

fresh_extraction_summary_df.to_csv(
    "/content/ipl_fresh_evaluation_extraction_summary.csv",
    index=False
)

print("\nSaved fresh extraction results and summary.")

Total questions: 180
Fixed-evidence rule extraction: 179 / 180
Metadata-assisted hybrid extraction: 180 / 180


,question_type,questions,fixed_correct,adaptive_correct,fixed_accuracy,adaptive_accuracy
0,batter_runs,60,60,60,1.000000,1.0
1,innings_runs,60,59,60,0.983333,1.0
2,match_winner,60,60,60,1.000000,1.0



Saved fresh extraction results and summary.


In [88]:
# STEP 78: CONSOLIDATE ALL FIVE METHODS ON THE FRESH EVALUATION
# Purpose:
# 1. Compare all five methods on the same 180 questions.
# 2. Calculate overall and per-category accuracy.
# 3. Save reproducible result tables for the paper.

# Step 1: Define the five evaluated methods.
method_results = {
    "No retrieval + FLAN-T5-small": (
        fresh_generation_df["No retrieval + FLAN-T5-small"]
    ),
    "Fixed BM25 RAG + FLAN-T5-small": (
        fresh_generation_df["Fixed BM25 RAG + FLAN-T5-small"]
    ),
    "Metadata-assisted RAG + FLAN-T5-small": (
        fresh_generation_df["Metadata-assisted RAG + FLAN-T5-small"]
    ),
    "Fixed BM25 + rule extraction": (
        fresh_extraction_df["fixed_rule_correct"]
    ),
    "Metadata-assisted retrieval + rule extraction": (
        fresh_extraction_df["adaptive_rule_correct"]
    )
}

# Step 2: Build the overall results table.
overall_records = []

for method_name, correct_series in method_results.items():
    correct_count = int(correct_series.sum())
    total_count = len(correct_series)

    overall_records.append({
        "method": method_name,
        "correct_answers": correct_count,
        "total_questions": total_count,
        "accuracy_percent": round(
            100 * correct_count / total_count, 2
        )
    })

fresh_five_method_summary_df = pd.DataFrame(overall_records)

print("OVERALL COMPARISON")
display(fresh_five_method_summary_df)

# Step 3: Build the per-question-type comparison.
type_records = []

for question_type in ["batter_runs", "innings_runs", "match_winner"]:
    question_mask = (
        fresh_generation_df["question_type"] == question_type
    )
    total_count = int(question_mask.sum())

    for method_name in method_results:
        if method_name == "Fixed BM25 + rule extraction":
            correct_series = fresh_extraction_df.loc[
                question_mask, "fixed_rule_correct"
            ]
        elif method_name == "Metadata-assisted retrieval + rule extraction":
            correct_series = fresh_extraction_df.loc[
                question_mask, "adaptive_rule_correct"
            ]
        else:
            correct_series = fresh_generation_df.loc[
                question_mask, method_name
            ]

        correct_count = int(correct_series.sum())

        type_records.append({
            "question_type": question_type,
            "method": method_name,
            "correct_answers": correct_count,
            "total_questions": total_count,
            "accuracy_percent": round(
                100 * correct_count / total_count, 2
            )
        })

fresh_five_method_by_type_df = pd.DataFrame(type_records)

print("\nCOMPARISON BY QUESTION TYPE")
display(fresh_five_method_by_type_df)

# Step 4: Save the final comparison tables.
fresh_five_method_summary_df.to_csv(
    "/content/ipl_fresh_five_method_summary.csv",
    index=False
)

fresh_five_method_by_type_df.to_csv(
    "/content/ipl_fresh_five_method_by_type.csv",
    index=False
)

print("\nSaved both consolidated comparison tables.")

OVERALL COMPARISON


,method,correct_answers,total_questions,accuracy_percent
0,No retrieval + FLAN-T5-small,3,180,1.67
1,Fixed BM25 RAG + FLAN-T5-small,100,180,55.56
2,Metadata-assisted RAG + FLAN-T5-small,101,180,56.11
3,Fixed BM25 + rule extraction,179,180,99.44
4,Metadata-assisted retrieval + rule extraction,180,180,100.00



COMPARISON BY QUESTION TYPE


,question_type,method,correct_answers,total_questions,accuracy_percent
0,batter_runs,No retrieval + FLAN-T5-small,3,60,5.00
1,batter_runs,Fixed BM25 RAG + FLAN-T5-small,49,60,81.67
2,batter_runs,Metadata-assisted RAG + FLAN-T5-small,49,60,81.67
3,batter_runs,Fixed BM25 + rule extraction,60,60,100.00
4,batter_runs,Metadata-assisted retrieval + rule extraction,60,60,100.00
5,innings_runs,No retrieval + FLAN-T5-small,0,60,0.00
6,innings_runs,Fixed BM25 RAG + FLAN-T5-small,39,60,65.00
7,innings_runs,Metadata-assisted RAG + FLAN-T5-small,40,60,66.67
8,innings_runs,Fixed BM25 + rule extraction,59,60,98.33
9,innings_runs,Metadata-assisted retrieval + rule extraction,60,60,100.00



Saved both consolidated comparison tables.


In [89]:
# STEP 79: CALCULATE CONFIDENCE INTERVALS AND PAIRED TESTS
# Purpose:
# 1. Calculate 95% Wilson confidence intervals for accuracy.
# 2. Align method results by question_id before paired comparisons.
# 3. Run exact McNemar tests using discordant question outcomes.
# 4. Save the statistical results for transparent reporting.

import pandas as pd
from math import sqrt
from scipy.stats import binomtest

# Step 1: Define the Wilson confidence interval.
def wilson_interval(correct, total, z=1.959963984540054):
    if total == 0:
        return float("nan"), float("nan")

    p = correct / total
    denominator = 1 + (z ** 2) / total
    centre = (p + (z ** 2) / (2 * total)) / denominator
    margin = (
        z
        * sqrt((p * (1 - p) / total) + (z ** 2) / (4 * total ** 2))
        / denominator
    )

    return max(0, centre - margin), min(1, centre + margin)


# Step 2: Align the generation and extraction results by question ID.
paired_results_df = fresh_generation_df[
    [
        "question_id",
        "No retrieval + FLAN-T5-small",
        "Fixed BM25 RAG + FLAN-T5-small",
        "Metadata-assisted RAG + FLAN-T5-small"
    ]
].merge(
    fresh_extraction_df[
        [
            "question_id",
            "fixed_rule_correct",
            "adaptive_rule_correct"
        ]
    ],
    on="question_id",
    how="inner",
    validate="one_to_one"
)

assert len(paired_results_df) == 180
assert paired_results_df["question_id"].is_unique

# Step 3: Define the five method result columns.
method_columns = {
    "No retrieval + FLAN-T5-small": "No retrieval + FLAN-T5-small",
    "Fixed BM25 RAG + FLAN-T5-small": "Fixed BM25 RAG + FLAN-T5-small",
    "Metadata-assisted RAG + FLAN-T5-small": "Metadata-assisted RAG + FLAN-T5-small",
    "Fixed BM25 + rule extraction": "fixed_rule_correct",
    "Metadata-assisted retrieval + rule extraction": "adaptive_rule_correct"
}

# Step 4: Calculate accuracy and 95% Wilson confidence intervals.
ci_records = []

for method_name, column in method_columns.items():
    correct = int(paired_results_df[column].sum())
    total = len(paired_results_df)
    lower, upper = wilson_interval(correct, total)

    ci_records.append({
        "method": method_name,
        "correct": correct,
        "total": total,
        "accuracy_percent": round(100 * correct / total, 2),
        "ci_95_lower_percent": round(100 * lower, 2),
        "ci_95_upper_percent": round(100 * upper, 2)
    })

confidence_intervals_df = pd.DataFrame(ci_records)

print("ACCURACY WITH 95% WILSON CONFIDENCE INTERVALS")
display(confidence_intervals_df)

# Step 5: Define paired exact McNemar testing.
def exact_mcnemar(column_a, column_b):
    a = paired_results_df[column_a].astype(bool)
    b = paired_results_df[column_b].astype(bool)

    # A correct, B incorrect.
    a_only = int((a & ~b).sum())

    # B correct, A incorrect.
    b_only = int((~a & b).sum())

    discordant = a_only + b_only

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = binomtest(
            min(a_only, b_only),
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue

    return {
        "method_A": column_a,
        "method_B": column_b,
        "A_only_correct": a_only,
        "B_only_correct": b_only,
        "discordant_pairs": discordant,
        "exact_mcnemar_p": p_value
    }

# Step 6: Test the three preselected comparisons.
paired_tests = [
    (
        "Fixed BM25 RAG + FLAN-T5-small",
        "Metadata-assisted RAG + FLAN-T5-small"
    ),
    (
        "Fixed BM25 + rule extraction",
        "Metadata-assisted retrieval + rule extraction"
    ),
    (
        "Fixed BM25 RAG + FLAN-T5-small",
        "Fixed BM25 + rule extraction"
    )
]

mcnemar_records = []

for method_a, method_b in paired_tests:
    result = exact_mcnemar(
        method_columns[method_a],
        method_columns[method_b]
    )
    result["method_A"] = method_a
    result["method_B"] = method_b
    mcnemar_records.append(result)

mcnemar_results_df = pd.DataFrame(mcnemar_records)

print("\nEXACT PAIRED McNEMAR TESTS")
display(mcnemar_results_df)

# Step 7: Save both statistical tables.
confidence_intervals_df.to_csv(
    "/content/ipl_fresh_evaluation_confidence_intervals.csv",
    index=False
)

mcnemar_results_df.to_csv(
    "/content/ipl_fresh_evaluation_mcnemar_tests.csv",
    index=False
)

print("\nSaved confidence intervals and paired-test results.")

ACCURACY WITH 95% WILSON CONFIDENCE INTERVALS


,method,correct,total,accuracy_percent,ci_95_lower_percent,ci_95_upper_percent
0,No retrieval + FLAN-T5-small,3,180,1.67,0.57,4.78
1,Fixed BM25 RAG + FLAN-T5-small,100,180,55.56,48.26,62.62
2,Metadata-assisted RAG + FLAN-T5-small,101,180,56.11,48.81,63.16
3,Fixed BM25 + rule extraction,179,180,99.44,96.92,99.90
4,Metadata-assisted retrieval + rule extraction,180,180,100.00,97.91,100.00



EXACT PAIRED McNEMAR TESTS


,method_A,method_B,A_only_correct,B_only_correct,discordant_pairs,exact_mcnemar_p
0,Fixed BM25 RAG + FLAN-T5-small,Metadata-assisted RAG + FLAN-T5-small,0,1,1,1.000000e+00
1,Fixed BM25 + rule extraction,Metadata-assisted retrieval + rule extraction,0,1,1,1.000000e+00
2,Fixed BM25 RAG + FLAN-T5-small,Fixed BM25 + rule extraction,0,79,79,3.308722e-24



Saved confidence intervals and paired-test results.


In [90]:

# STEP 80: FIX THE METHOD NAME AND RUN PAIRED McNEMAR TESTS

# Step 1: Define the three paired comparisons using exact dictionary keys.
paired_tests = [
    (
        "Fixed BM25 RAG + FLAN-T5-small",
        "Metadata-assisted RAG + FLAN-T5-small"
    ),
    (
        "Fixed BM25 + rule extraction",
        "Metadata-assisted retrieval + rule extraction"
    ),
    (
        "Fixed BM25 RAG + FLAN-T5-small",
        "Fixed BM25 + rule extraction"
    )
]

# Step 2: Run the exact McNemar tests.
mcnemar_records = []

for method_a, method_b in paired_tests:
    result = exact_mcnemar(
        method_columns[method_a],
        method_columns[method_b]
    )
    result["method_A"] = method_a
    result["method_B"] = method_b
    mcnemar_records.append(result)

# Step 3: Display the results.
mcnemar_results_df = pd.DataFrame(mcnemar_records)
display(mcnemar_results_df)

# Step 4: Save the results.
mcnemar_results_df.to_csv(
    "/content/ipl_fresh_evaluation_mcnemar_tests.csv",
    index=False
)

print("Step 80 complete. Paired test results saved.")


,method_A,method_B,A_only_correct,B_only_correct,discordant_pairs,exact_mcnemar_p
0,Fixed BM25 RAG + FLAN-T5-small,Metadata-assisted RAG + FLAN-T5-small,0,1,1,1.000000e+00
1,Fixed BM25 + rule extraction,Metadata-assisted retrieval + rule extraction,0,1,1,1.000000e+00
2,Fixed BM25 RAG + FLAN-T5-small,Fixed BM25 + rule extraction,0,79,79,3.308722e-24


Step 80 complete. Paired test results saved.


In [91]:
# STEP 81: PREPARE THE MAIN RESULTS TABLE FOR THE RESEARCH PAPER

# Step 1: Select the main performance metrics.
paper_results_df = fresh_five_method_summary_df[
    [
        "method",
        "correct_answers",
        "total_questions",
        "accuracy_percent"
    ]
].copy()

# Step 2: Add the 95% Wilson confidence intervals.
ci_lookup = confidence_intervals_df.set_index("method")

paper_results_df["95% CI"] = paper_results_df["method"].apply(
    lambda method: (
        f'{ci_lookup.loc[method, "ci_95_lower_percent"]:.2f}%–'
        f'{ci_lookup.loc[method, "ci_95_upper_percent"]:.2f}%'
    )
)

# Step 3: Add a short interpretation of each method.
interpretations = {
    "No retrieval + FLAN-T5-small":
        "Parametric baseline without retrieved evidence",
    "Fixed BM25 RAG + FLAN-T5-small":
        "Fixed lexical retrieval followed by generation",
    "Metadata-assisted RAG + FLAN-T5-small":
        "Metadata-assisted retrieval followed by generation",
    "Fixed BM25 + rule extraction":
        "Deterministic extraction from the top-ranked passage",
    "Metadata-assisted retrieval + rule extraction":
        "Metadata-assisted evidence selection and extraction"
}

paper_results_df["method_description"] = (
    paper_results_df["method"].map(interpretations)
)

# Step 4: Display and save the table.
display(paper_results_df)

paper_results_df.to_csv(
    "/content/ipl_paper_main_results_table.csv",
    index=False
)

print("Saved: /content/ipl_paper_main_results_table.csv")

,method,correct_answers,total_questions,accuracy_percent,95% CI,method_description
0,No retrieval + FLAN-T5-small,3,180,1.67,0.57%–4.78%,Parametric baseline without retrieved evidence
1,Fixed BM25 RAG + FLAN-T5-small,100,180,55.56,48.26%–62.62%,Fixed lexical retrieval followed by generation
2,Metadata-assisted RAG + FLAN-T5-small,101,180,56.11,48.81%–63.16%,Metadata-assisted retrieval followed by genera...
3,Fixed BM25 + rule extraction,179,180,99.44,96.92%–99.90%,Deterministic extraction from the top-ranked p...
4,Metadata-assisted retrieval + rule extraction,180,180,100.00,97.91%–100.00%,Metadata-assisted evidence selection and extra...


Saved: /content/ipl_paper_main_results_table.csv


In [92]:

# STEP 82: ANALYSE FLAN-T5-SMALL ERRORS BY QUESTION TYPE

# Step 1: Select the frozen evaluation results.
error_analysis_df = fresh_generation_df[
    [
        "question_id",
        "question_type",
        "question",
        "gold_answer",
        "answer_no_retrieval",
        "answer_fixed_rag",
        "answer_metadata_assisted"
    ]
].copy()

# Step 2: Attach the correctness labels from the scored results.
scored_columns = [
    "question_id",
    "No retrieval + FLAN-T5-small",
    "Fixed BM25 RAG + FLAN-T5-small",
    "Metadata-assisted RAG + FLAN-T5-small"
]

error_analysis_df = error_analysis_df.merge(
    fresh_generation_df[scored_columns].drop_duplicates(
        subset=["question_id"]
    ),
    on="question_id",
    how="left",
    validate="one_to_one"
)

# Step 3: Calculate correct and incorrect counts by question type.
error_records = []

correctness_columns = {
    "No retrieval + FLAN-T5-small":
        "No retrieval + FLAN-T5-small",
    "Fixed BM25 RAG + FLAN-T5-small":
        "Fixed BM25 RAG + FLAN-T5-small",
    "Metadata-assisted RAG + FLAN-T5-small":
        "Metadata-assisted RAG + FLAN-T5-small"
}

for question_type in ["batter_runs", "innings_runs", "match_winner"]:
    subset = error_analysis_df[
        error_analysis_df["question_type"] == question_type
    ]

    for method_name, correctness_column in correctness_columns.items():
        correct_count = int(subset[correctness_column].sum())
        total_count = len(subset)

        error_records.append({
            "question_type": question_type,
            "method": method_name,
            "correct": correct_count,
            "incorrect": total_count - correct_count,
            "accuracy_percent": round(
                100 * correct_count / total_count, 2
            )
        })

error_by_type_df = pd.DataFrame(error_records)

# Step 4: Display the error summary.
display(error_by_type_df)

# Step 5: Show incorrect match-winner answers from fixed RAG.
winner_errors_df = error_analysis_df[
    (error_analysis_df["question_type"] == "match_winner")
    & (~error_analysis_df["Fixed BM25 RAG + FLAN-T5-small"].astype(bool))
][
    [
        "question_id",
        "question",
        "gold_answer",
        "answer_fixed_rag",
        "answer_metadata_assisted"
    ]
].copy()

print("INCORRECT FIXED-RAG MATCH-WINNER ANSWERS")
display(winner_errors_df.head(15))

# Step 6: Save the error-analysis tables.
error_by_type_df.to_csv(
    "/content/ipl_fresh_error_summary_by_type.csv",
    index=False
)

winner_errors_df.to_csv(
    "/content/ipl_fresh_match_winner_errors.csv",
    index=False
)

print("Step 82 complete. Error-analysis tables saved.")


,question_type,method,correct,incorrect,accuracy_percent
0,batter_runs,No retrieval + FLAN-T5-small,3,57,5.00
1,batter_runs,Fixed BM25 RAG + FLAN-T5-small,49,11,81.67
2,batter_runs,Metadata-assisted RAG + FLAN-T5-small,49,11,81.67
3,innings_runs,No retrieval + FLAN-T5-small,0,60,0.00
4,innings_runs,Fixed BM25 RAG + FLAN-T5-small,39,21,65.00
5,innings_runs,Metadata-assisted RAG + FLAN-T5-small,40,20,66.67
6,match_winner,No retrieval + FLAN-T5-small,0,60,0.00
7,match_winner,Fixed BM25 RAG + FLAN-T5-small,12,48,20.00
8,match_winner,Metadata-assisted RAG + FLAN-T5-small,12,48,20.00


INCORRECT FIXED-RAG MATCH-WINNER ANSWERS


,question_id,question,gold_answer,answer_fixed_rag,answer_metadata_assisted
120,1082636_winner,Which team won the IPL match between Royal Cha...,Kolkata Knight Riders,1082636,1082636
121,1136572_winner,Which team won the IPL match between Kings XI ...,Kings XI Punjab,0,0
122,1136578_winner,Which team won the IPL match between Kolkata K...,Kings XI Punjab,Kolkata Knight Riders,Kolkata Knight Riders
123,1136580_winner,Which team won the IPL match between Chennai S...,Chennai Super Kings,0,0
125,1136603_winner,Which team won the IPL match between Chennai S...,Rajasthan Royals,1136603.,1136603.
126,1136609_winner,Which team won the IPL match between Rajasthan...,Kolkata Knight Riders,1136609.,1136609.
127,1136615_winner,Which team won the IPL match between Delhi Dar...,Delhi Daredevils,1136615,1136615
128,1178404_winner,Which team won the IPL match between Kolkata K...,Chennai Super Kings,0,0
129,1178413_winner,Which team won the IPL match between Kolkata K...,Sunrisers Hyderabad,1178413,1178413
132,1254096_winner,Which team won the IPL match between Mumbai In...,Kolkata Knight Riders,Mumbai Indians,Mumbai Indians


Step 82 complete. Error-analysis tables saved.


In [93]:

# STEP 83: INSPECT EVIDENCE FOR INCORRECT MATCH-WINNER ANSWERS

# Step 1: Select the incorrectly answered winner questions.
winner_error_ids = set(
    winner_errors_df["question_id"].astype(str)
)

winner_error_evidence_df = (
    fresh_generation_df[
        fresh_generation_df["question_id"].astype(str).isin(
            winner_error_ids
        )
    ][
        [
            "question_id",
            "question",
            "gold_answer",
            "answer_fixed_rag"
        ]
    ]
    .merge(
        fresh_eval_df[
            ["question_id", "match_id", "question_type"]
        ],
        on="question_id",
        how="left",
        validate="one_to_one"
    )
)

# Step 2: Attach the exact fixed-BM25 passage used for generation.
fixed_evidence_lookup = (
    fresh_eval_df[
        ["question_id"]
    ]
    .merge(
        fresh_eval_df[["question_id"]],
        on="question_id",
        how="left"
    )
)

# Step 3: Retrieve evidence using the saved fresh-evaluation inputs.
fixed_input_lookup = generation_inputs_df.set_index("question_id")

winner_error_evidence_df["fixed_passage_id"] = (
    winner_error_evidence_df["question_id"].map(
        fixed_input_lookup["fixed_passage_id"]
    )
)

winner_error_evidence_df["fixed_evidence"] = (
    winner_error_evidence_df["question_id"].map(
        fixed_input_lookup["fixed_evidence"]
    )
)

# Step 4: Extract the winner explicitly from the retrieved passage.
def extract_winner_for_diagnostic(evidence):
    match = re.search(
        r"\bWinner:\s*([^.]*)",
        str(evidence),
        flags=re.IGNORECASE
    )
    if not match:
        return None

    winner = match.group(1).strip()
    return winner if winner and winner.lower() != "none" else None

winner_error_evidence_df["winner_in_evidence"] = (
    winner_error_evidence_df["fixed_evidence"].apply(
        extract_winner_for_diagnostic
    )
)

winner_error_evidence_df["evidence_matches_gold"] = (
    winner_error_evidence_df["winner_in_evidence"].fillna("").str.lower()
    == winner_error_evidence_df["gold_answer"].fillna("").str.lower()
)

# Step 5: Display a small sample for manual inspection.
display(
    winner_error_evidence_df[
        [
            "question_id",
            "gold_answer",
            "answer_fixed_rag",
            "winner_in_evidence",
            "evidence_matches_gold",
            "fixed_passage_id",
            "fixed_evidence"
        ]
    ].head(10)
)

# Step 6: Summarize whether the winner was present in the evidence.
print(
    "Incorrect winner answers whose evidence contains the gold winner:",
    int(winner_error_evidence_df["evidence_matches_gold"].sum()),
    "out of",
    len(winner_error_evidence_df)
)

# Step 7: Save the diagnostic table.
winner_error_evidence_df.to_csv(
    "/content/ipl_match_winner_evidence_diagnostic.csv",
    index=False
)

print("Step 83 complete. Evidence diagnostic saved.")


,question_id,gold_answer,answer_fixed_rag,winner_in_evidence,evidence_matches_gold,fixed_passage_id,fixed_evidence
0,1082636_winner,Kolkata Knight Riders,1082636,None,False,NaN,NaN
1,1136572_winner,Kings XI Punjab,0,None,False,NaN,NaN
2,1136578_winner,Kings XI Punjab,Kolkata Knight Riders,None,False,NaN,NaN
3,1136580_winner,Chennai Super Kings,0,None,False,NaN,NaN
4,1136603_winner,Rajasthan Royals,1136603.,None,False,NaN,NaN
5,1136609_winner,Kolkata Knight Riders,1136609.,None,False,NaN,NaN
6,1136615_winner,Delhi Daredevils,1136615,None,False,NaN,NaN
7,1178404_winner,Chennai Super Kings,0,None,False,NaN,NaN
8,1178413_winner,Sunrisers Hyderabad,1178413,None,False,NaN,NaN
9,1254096_winner,Kolkata Knight Riders,Mumbai Indians,None,False,NaN,NaN


Incorrect winner answers whose evidence contains the gold winner: 0 out of 48
Step 83 complete. Evidence diagnostic saved.


In [94]:

# STEP 84: CHECK THE AVAILABLE EVIDENCE DATAFRAMES AND COLUMNS

# Step 1: Inspect the relevant dataframe columns.
print("generation_inputs_df columns:")
print(generation_inputs_df.columns.tolist())

print("\nfresh_eval_df columns:")
print(fresh_eval_df.columns.tolist())

print("\nfresh_generation_df columns:")
print(fresh_generation_df.columns.tolist())

# Step 2: Check whether the winner-error question IDs match.
print("\nSample winner-error IDs:")
print(winner_errors_df["question_id"].head(5).tolist())

print("\nSample generation-input IDs:")
print(generation_inputs_df["question_id"].head(5).tolist())

# Step 3: Count matching question IDs.
winner_ids = set(winner_errors_df["question_id"].astype(str))
input_ids = set(generation_inputs_df["question_id"].astype(str))

print("\nWinner-error IDs:", len(winner_ids))
print("Matching evidence-input IDs:", len(winner_ids & input_ids))


generation_inputs_df columns:
['question_id', 'question_type', 'question', 'gold_answer', 'no_retrieval_evidence', 'fixed_passage_id', 'fixed_evidence', 'adaptive_passage_id', 'adaptive_evidence']

fresh_eval_df columns:
['question_id', 'match_id', 'split', 'question_type', 'question', 'gold_answer', 'target_passage_id', 'target_passage_exists', 'target_split_matches', 'target_evidence', 'evidence_available']

fresh_generation_df columns:
['question_id', 'question_type', 'question', 'gold_answer', 'fixed_passage_id', 'adaptive_passage_id', 'answer_no_retrieval', 'answer_fixed_rag', 'answer_metadata_assisted', 'No retrieval + FLAN-T5-small', 'Fixed BM25 RAG + FLAN-T5-small', 'Metadata-assisted RAG + FLAN-T5-small']

Sample winner-error IDs:
['1082636_winner', '1136572_winner', '1136578_winner', '1136580_winner', '1136603_winner']

Sample generation-input IDs:
['1359531_innings2_R Tewatia_runs', '1082650_innings2_DT Christian_runs', '1216512_innings3_runs', '1527681_winner', '598017_inni

In [95]:

# STEP 85: RECONSTRUCT WINNER EVIDENCE FROM THE FRESH EVALUATION

# Step 1: Select the incorrect match-winner answers.
winner_error_evidence_df = winner_errors_df[
    [
        "question_id",
        "question",
        "gold_answer",
        "answer_fixed_rag"
    ]
].copy()

# Step 2: Get the fixed-retrieval passage IDs from the fresh evaluation.
fixed_ids_lookup = fresh_generation_df[
    ["question_id", "fixed_passage_id"]
].drop_duplicates(subset=["question_id"])

winner_error_evidence_df = winner_error_evidence_df.merge(
    fixed_ids_lookup,
    on="question_id",
    how="left",
    validate="one_to_one"
)

# Step 3: Reconstruct evidence from the passage lookup dictionaries.
def get_fixed_evidence(passage_id):
    if pd.isna(passage_id):
        return None

    passage_id = str(passage_id)

    if passage_id in summary_text_lookup:
        return summary_text_lookup[passage_id]

    if passage_id in player_text_lookup:
        return player_text_lookup[passage_id]

    return None

winner_error_evidence_df["fixed_evidence"] = (
    winner_error_evidence_df["fixed_passage_id"].apply(
        get_fixed_evidence
    )
)

# Step 4: Extract the winner explicitly recorded in each passage.
winner_error_evidence_df["winner_in_evidence"] = (
    winner_error_evidence_df["fixed_evidence"].apply(
        extract_winner_for_diagnostic
    )
)

winner_error_evidence_df["evidence_matches_gold"] = (
    winner_error_evidence_df["winner_in_evidence"].fillna("").str.lower()
    == winner_error_evidence_df["gold_answer"].fillna("").str.lower()
)

# Step 5: Display a sample and count the evidence matches.
display(
    winner_error_evidence_df[
        [
            "question_id",
            "gold_answer",
            "answer_fixed_rag",
            "fixed_passage_id",
            "winner_in_evidence",
            "evidence_matches_gold",
            "fixed_evidence"
        ]
    ].head(10)
)

print(
    "Correct winner explicitly found in evidence:",
    int(winner_error_evidence_df["evidence_matches_gold"].sum()),
    "out of",
    len(winner_error_evidence_df)
)

print(
    "Missing evidence passages:",
    int(winner_error_evidence_df["fixed_evidence"].isna().sum())
)

# Step 6: Save the corrected diagnostic.
winner_error_evidence_df.to_csv(
    "/content/ipl_match_winner_evidence_diagnostic.csv",
    index=False
)

print("Step 85 complete. Corrected diagnostic saved.")


,question_id,gold_answer,answer_fixed_rag,fixed_passage_id,winner_in_evidence,evidence_matches_gold,fixed_evidence
0,1082636_winner,Kolkata Knight Riders,1082636,1082636_summary,Kolkata Knight Riders,True,IPL match ID: 1082636. Date: 2017-05-07. Seaso...
1,1136572_winner,Kings XI Punjab,0,1136572_summary,Kings XI Punjab,True,IPL match ID: 1136572. Date: 2018-04-15. Seaso...
2,1136578_winner,Kings XI Punjab,Kolkata Knight Riders,1136578_summary,Kings XI Punjab,True,IPL match ID: 1136578. Date: 2018-04-21. Seaso...
3,1136580_winner,Chennai Super Kings,0,1136580_summary,Chennai Super Kings,True,IPL match ID: 1136580. Date: 2018-04-22. Seaso...
4,1136603_winner,Rajasthan Royals,1136603.,1136603_summary,Rajasthan Royals,True,IPL match ID: 1136603. Date: 2018-05-11. Seaso...
5,1136609_winner,Kolkata Knight Riders,1136609.,1136609_summary,Kolkata Knight Riders,True,IPL match ID: 1136609. Date: 2018-05-15. Seaso...
6,1136615_winner,Delhi Daredevils,1136615,1136615_summary,Delhi Daredevils,True,IPL match ID: 1136615. Date: 2018-05-20. Seaso...
7,1178404_winner,Chennai Super Kings,0,1178404_summary,Chennai Super Kings,True,IPL match ID: 1178404. Date: 2019-04-14. Seaso...
8,1178413_winner,Sunrisers Hyderabad,1178413,1178413_summary,Sunrisers Hyderabad,True,IPL match ID: 1178413. Date: 2019-04-21. Seaso...
9,1254096_winner,Kolkata Knight Riders,Mumbai Indians,1254096_summary,Kolkata Knight Riders,True,IPL match ID: 1254096. Date: 2021-09-23. Seaso...


Correct winner explicitly found in evidence: 48 out of 48
Missing evidence passages: 0
Step 85 complete. Corrected diagnostic saved.


In [96]:

# STEP 86: CATEGORIZE FLAN-T5-SMALL MATCH-WINNER ERRORS

# Step 1: Copy the corrected winner diagnostic.
winner_error_categories_df = winner_error_evidence_df.copy()

# Step 2: Identify the most obvious output patterns.
def categorize_winner_error(row):
    answer = str(row["answer_fixed_rag"]).strip()
    gold = str(row["gold_answer"]).strip()

    # Detect empty or missing answers.
    if not answer or answer.lower() in {"none", "nan"}:
        return "Empty or missing answer"

    # Detect zero outputs.
    if answer == "0":
        return "Zero output"

    # Detect a match ID copied as the answer.
    match_id = str(row["question_id"]).split("_")[0]
    if answer.rstrip(".") == match_id:
        return "Match ID copied as answer"

    # Detect whether the answer names one of the two teams.
    question = str(row["question"])
    team_match = re.search(
        r"between (.+?) and (.+?)\?",
        question,
        flags=re.IGNORECASE
    )

    if team_match:
        team_a = team_match.group(1).strip()
        team_b = team_match.group(2).strip()

        if answer.lower() in {team_a.lower(), team_b.lower()}:
            return "Team name, but incorrect winner"

    # Detect the player name as the answer.
    evidence = str(row["fixed_evidence"])
    player_match = re.search(
        r"Player of the match:\s*([^.]*)",
        evidence,
        flags=re.IGNORECASE
    )

    if player_match and answer.lower() == player_match.group(1).strip().lower():
        return "Player of the match copied as answer"

    return "Other incorrect response"

# Step 3: Categorize the 48 incorrect fixed-RAG answers.
winner_error_categories_df["error_category"] = (
    winner_error_categories_df.apply(
        categorize_winner_error,
        axis=1
    )
)

# Step 4: Summarize error categories.
winner_error_category_summary_df = (
    winner_error_categories_df["error_category"]
    .value_counts()
    .rename_axis("error_category")
    .reset_index(name="count")
)

winner_error_category_summary_df["percent_of_errors"] = (
    100 * winner_error_category_summary_df["count"]
    / len(winner_error_categories_df)
).round(2)

display(winner_error_category_summary_df)

# Step 5: Save the categorized error analysis.
winner_error_category_summary_df.to_csv(
    "/content/ipl_match_winner_error_categories.csv",
    index=False
)

winner_error_categories_df.to_csv(
    "/content/ipl_match_winner_error_examples_categorized.csv",
    index=False
)

print("Step 86 complete. Error categories and examples saved.")


,error_category,count,percent_of_errors
0,Match ID copied as answer,14,29.17
1,Zero output,12,25.00
2,"Team name, but incorrect winner",10,20.83
3,Player of the match copied as answer,7,14.58
4,Other incorrect response,5,10.42


Step 86 complete. Error categories and examples saved.


In [97]:

# STEP 87: EVALUATE DETERMINISTIC MATCH-WINNER EXTRACTION

# Step 1: Select all 60 frozen match-winner questions.
winner_eval_df = fresh_eval_df[
    fresh_eval_df["question_type"] == "match_winner"
][["question_id", "question", "gold_answer"]].copy()

# Step 2: Attach the fixed BM25 evidence IDs from the frozen run.
winner_eval_df = winner_eval_df.merge(
    fresh_generation_df[
        ["question_id", "fixed_passage_id", "answer_fixed_rag"]
    ],
    on="question_id",
    how="left",
    validate="one_to_one"
)

# Step 3: Retrieve each passage from the saved passage dictionaries.
winner_eval_df["fixed_evidence"] = (
    winner_eval_df["fixed_passage_id"].apply(get_fixed_evidence)
)

# Step 4: Extract the explicitly recorded winner.
winner_eval_df["rule_extracted_answer"] = (
    winner_eval_df["fixed_evidence"].apply(
        extract_winner_for_diagnostic
    )
)

# Step 5: Score the rule-based and generated answers.
winner_eval_df["rule_correct"] = winner_eval_df.apply(
    lambda row: (
        normalize_answer_text(row["rule_extracted_answer"])
        == normalize_answer_text(row["gold_answer"])
        if pd.notna(row["rule_extracted_answer"])
        else False
    ),
    axis=1
)

winner_eval_df["llm_correct"] = winner_eval_df["question_id"].map(
    fresh_generation_df.set_index("question_id")[
        "Fixed BM25 RAG + FLAN-T5-small"
    ]
).astype(bool)

# Step 6: Display the comparison.
winner_comparison_df = pd.DataFrame([
    {
        "method": "Fixed BM25 RAG + FLAN-T5-small",
        "correct": int(winner_eval_df["llm_correct"].sum()),
        "total": len(winner_eval_df),
        "accuracy_percent": round(
            100 * winner_eval_df["llm_correct"].mean(), 2
        )
    },
    {
        "method": "Fixed BM25 + deterministic winner extraction",
        "correct": int(winner_eval_df["rule_correct"].sum()),
        "total": len(winner_eval_df),
        "accuracy_percent": round(
            100 * winner_eval_df["rule_correct"].mean(), 2
        )
    }
])

display(winner_comparison_df)

# Step 7: Save the question-level comparison.
winner_eval_df.to_csv(
    "/content/ipl_winner_extraction_comparison.csv",
    index=False
)

winner_comparison_df.to_csv(
    "/content/ipl_winner_extraction_summary.csv",
    index=False
)

print("Step 87 complete. Winner extraction comparison saved.")


,method,correct,total,accuracy_percent
0,Fixed BM25 RAG + FLAN-T5-small,12,60,20.0
1,Fixed BM25 + deterministic winner extraction,60,60,100.0


Step 87 complete. Winner extraction comparison saved.


In [98]:

# STEP 88: VERIFY THE SAVED RESEARCH OUTPUTS

# Step 1: Import the file utilities.
import os

# Step 2: List the main research artifacts expected from our experiments.
expected_files = [
    "/content/ipl_fresh_evaluation_180_questions.csv",
    "/content/ipl_fresh_evaluation_frozen_ids.csv",
    "/content/ipl_fresh_evaluation_frozen_config.json",
    "/content/ipl_fresh_evaluation_retrieval_summary.csv",
    "/content/ipl_fresh_evaluation_generation_summary.csv",
    "/content/ipl_fresh_evaluation_generation_by_type.csv",
    "/content/ipl_fresh_evaluation_extraction_summary.csv",
    "/content/ipl_fresh_five_method_summary.csv",
    "/content/ipl_fresh_five_method_by_type.csv",
    "/content/ipl_fresh_evaluation_confidence_intervals.csv",
    "/content/ipl_fresh_evaluation_mcnemar_tests.csv",
    "/content/ipl_paper_main_results_table.csv",
    "/content/ipl_fresh_error_summary_by_type.csv",
    "/content/ipl_match_winner_error_categories.csv",
    "/content/ipl_match_winner_evidence_diagnostic.csv",
    "/content/ipl_winner_extraction_comparison.csv",
    "/content/ipl_winner_extraction_summary.csv"
]

# Step 3: Report which files exist and their sizes.
artifact_status_df = pd.DataFrame([
    {
        "file": os.path.basename(path),
        "exists": os.path.exists(path),
        "size_kb": round(os.path.getsize(path) / 1024, 2)
        if os.path.exists(path) else None
    }
    for path in expected_files
])

display(artifact_status_df)

print(
    "\nFiles found:",
    int(artifact_status_df["exists"].sum()),
    "out of",
    len(artifact_status_df)
)


,file,exists,size_kb
0,ipl_fresh_evaluation_180_questions.csv,True,90.94
1,ipl_fresh_evaluation_frozen_ids.csv,True,4.01
2,ipl_fresh_evaluation_frozen_config.json,True,0.56
3,ipl_fresh_evaluation_retrieval_summary.csv,True,0.20
4,ipl_fresh_evaluation_generation_summary.csv,True,0.25
5,ipl_fresh_evaluation_generation_by_type.csv,True,0.65
6,ipl_fresh_evaluation_extraction_summary.csv,True,0.19
7,ipl_fresh_five_method_summary.csv,True,0.29
8,ipl_fresh_five_method_by_type.csv,True,0.92
9,ipl_fresh_evaluation_confidence_intervals.csv,True,0.37



Files found: 17 out of 17


In [99]:

# STEP 89: RESTORE THE MISSING CONFIDENCE-INTERVAL CSV

# Step 1: Load the saved main results table.
main_results_df = pd.read_csv(
    "/content/ipl_paper_main_results_table.csv"
)

# Step 2: Load the frozen evaluation totals from the main results.
# The existing table contains the correct counts and interval values.
restored_ci_df = main_results_df[
    [
        "method",
        "correct_answers",
        "total_questions",
        "accuracy_percent",
        "95% CI"
    ]
].copy()

# Step 3: Split the displayed confidence interval into numeric columns.
interval_parts = restored_ci_df["95% CI"].str.extract(
    r"([\d.]+)%\s*[–-]\s*([\d.]+)%"
)

restored_ci_df["ci_95_lower_percent"] = pd.to_numeric(
    interval_parts[0],
    errors="coerce"
)

restored_ci_df["ci_95_upper_percent"] = pd.to_numeric(
    interval_parts[1],
    errors="coerce"
)

# Step 4: Keep a clean statistical output schema.
restored_ci_df = restored_ci_df[
    [
        "method",
        "correct_answers",
        "total_questions",
        "accuracy_percent",
        "ci_95_lower_percent",
        "ci_95_upper_percent"
    ]
]

# Step 5: Save the restored file.
restored_ci_df.to_csv(
    "/content/ipl_fresh_evaluation_confidence_intervals.csv",
    index=False
)

# Step 6: Verify the saved file.
print("File exists:",
      os.path.exists(
          "/content/ipl_fresh_evaluation_confidence_intervals.csv"
      ))

display(restored_ci_df)


File exists: True


,method,correct_answers,total_questions,accuracy_percent,ci_95_lower_percent,ci_95_upper_percent
0,No retrieval + FLAN-T5-small,3,180,1.67,0.57,4.78
1,Fixed BM25 RAG + FLAN-T5-small,100,180,55.56,48.26,62.62
2,Metadata-assisted RAG + FLAN-T5-small,101,180,56.11,48.81,63.16
3,Fixed BM25 + rule extraction,179,180,99.44,96.92,99.90
4,Metadata-assisted retrieval + rule extraction,180,180,100.00,97.91,100.00


In [100]:

# STEP 90: VERIFY ALL RESEARCH ARTIFACTS BEFORE PACKAGING

# Step 1: Reuse the list of expected research outputs.
artifact_status_df = pd.DataFrame([
    {
        "file": os.path.basename(path),
        "exists": os.path.exists(path),
        "size_kb": round(os.path.getsize(path) / 1024, 2)
        if os.path.exists(path) else None
    }
    for path in expected_files
])

# Step 2: Display the verification results.
display(artifact_status_df)

# Step 3: Count missing files.
missing_files = artifact_status_df.loc[
    ~artifact_status_df["exists"], "file"
].tolist()

print("Verified:", int(artifact_status_df["exists"].sum()), "of", len(expected_files))
print("Missing files:", missing_files if missing_files else "None")


,file,exists,size_kb
0,ipl_fresh_evaluation_180_questions.csv,True,90.94
1,ipl_fresh_evaluation_frozen_ids.csv,True,4.01
2,ipl_fresh_evaluation_frozen_config.json,True,0.56
3,ipl_fresh_evaluation_retrieval_summary.csv,True,0.20
4,ipl_fresh_evaluation_generation_summary.csv,True,0.25
5,ipl_fresh_evaluation_generation_by_type.csv,True,0.65
6,ipl_fresh_evaluation_extraction_summary.csv,True,0.19
7,ipl_fresh_five_method_summary.csv,True,0.29
8,ipl_fresh_five_method_by_type.csv,True,0.92
9,ipl_fresh_evaluation_confidence_intervals.csv,True,0.38


Verified: 17 of 17
Missing files: None


In [101]:

# STEP 91: PACKAGE THE IPL ADAPTIVE RAG RESEARCH ARTIFACTS

# Step 1: Import the required libraries.
import os
import shutil
import zipfile
from pathlib import Path
from datetime import datetime

# Step 2: Define the package directory and archive path.
package_dir = Path("/content/ipl_adaptive_rag_research_package")
archive_path = Path("/content/ipl_adaptive_rag_research_package.zip")

package_dir.mkdir(parents=True, exist_ok=True)

# Step 3: List the verified frozen-evaluation and diagnostic artifacts.
package_files = [
    "/content/ipl_fresh_evaluation_180_questions.csv",
    "/content/ipl_fresh_evaluation_frozen_ids.csv",
    "/content/ipl_fresh_evaluation_frozen_config.json",
    "/content/ipl_fresh_evaluation_retrieval_summary.csv",
    "/content/ipl_fresh_evaluation_generation_summary.csv",
    "/content/ipl_fresh_evaluation_generation_by_type.csv",
    "/content/ipl_fresh_evaluation_extraction_summary.csv",
    "/content/ipl_fresh_five_method_summary.csv",
    "/content/ipl_fresh_five_method_by_type.csv",
    "/content/ipl_fresh_evaluation_confidence_intervals.csv",
    "/content/ipl_fresh_evaluation_mcnemar_tests.csv",
    "/content/ipl_paper_main_results_table.csv",
    "/content/ipl_fresh_error_summary_by_type.csv",
    "/content/ipl_match_winner_error_categories.csv",
    "/content/ipl_match_winner_evidence_diagnostic.csv",
    "/content/ipl_winner_extraction_comparison.csv",
    "/content/ipl_winner_extraction_summary.csv"
]

# Step 4: Verify all expected files before packaging.
missing_files = [path for path in package_files if not os.path.isfile(path)]

if missing_files:
    raise FileNotFoundError(
        "Cannot package research because these files are missing: "
        + ", ".join(missing_files)
    )

# Step 5: Copy the files into a clean package folder.
for file_path in package_files:
    shutil.copy2(file_path, package_dir / os.path.basename(file_path))

# Step 6: Write a README documenting the experiment.
readme_text = """
# Adaptive RAG for Historical IPL Question Answering

## Project overview

This project evaluates retrieval and answer-extraction approaches on
template-generated questions about historical Indian Premier League matches.

The frozen evaluation contains 180 questions:
- 60 batter-run questions
- 60 innings-total questions
- 60 match-winner questions

The evaluation questions were sampled from the held-out test split.
They exclude the earlier 60-question development pilot, but they are not
an independent external dataset.

## Methods compared

1. No retrieval with FLAN-T5-small
2. Fixed BM25 retrieval with FLAN-T5-small
3. Metadata-assisted retrieval with FLAN-T5-small
4. Fixed BM25 with deterministic rule extraction
5. Metadata-assisted retrieval with deterministic rule extraction

The language model was google/flan-t5-small. The retriever used BM25.
The rule-extraction methods use the structured passage format and should
not be interpreted as general-purpose natural-language reasoning systems.

## Main observed results

| Method | Correct | Questions | Accuracy |
|---|---:|---:|---:|
| No retrieval + FLAN-T5-small | 3 | 180 | 1.67% |
| Fixed BM25 RAG + FLAN-T5-small | 100 | 180 | 55.56% |
| Metadata-assisted RAG + FLAN-T5-small | 101 | 180 | 56.11% |
| Fixed BM25 + rule extraction | 179 | 180 | 99.44% |
| Metadata-assisted retrieval + rule extraction | 180 | 180 | 100.00% |

These are observed results on this benchmark, not evidence of perfect
generalization or universal hallucination reduction.

## Statistical analysis

The package includes Wilson confidence intervals and paired exact
McNemar tests. The methods were evaluated on the same questions, so
comparisons are paired.

The 1-question difference between fixed and metadata-assisted RAG
generation was not statistically significant. The 1-question difference
between the two rule-extraction methods was also not statistically
significant.

## Important limitations

- Questions are generated from structured historical match records.
- Rule extraction is tailored to the passage templates.
- The evaluation uses one domain and one frozen sample.
- Retrieval accuracy, answer accuracy, and evidence support are distinct.
- A correct answer alone does not establish that an answer was grounded.
- Further testing on independent datasets and less structured questions
  is needed before making broad claims about hallucination reduction.

## Files

- `ipl_fresh_evaluation_180_questions.csv`: frozen question sample
- `ipl_fresh_evaluation_frozen_ids.csv`: frozen question IDs
- `ipl_fresh_evaluation_frozen_config.json`: evaluation configuration
- `ipl_fresh_five_method_summary.csv`: overall method comparison
- `ipl_fresh_five_method_by_type.csv`: comparison by question type
- `ipl_fresh_evaluation_confidence_intervals.csv`: accuracy intervals
- `ipl_fresh_evaluation_mcnemar_tests.csv`: paired significance tests
- `ipl_fresh_error_summary_by_type.csv`: generation error summary
- `ipl_match_winner_error_categories.csv`: categorized winner errors
- `ipl_match_winner_evidence_diagnostic.csv`: evidence inspection
- `ipl_winner_extraction_comparison.csv`: question-level winner results
- `ipl_paper_main_results_table.csv`: paper-ready main results

Other CSV files contain supporting retrieval, generation, and extraction
summaries.

## Reproducibility notes

The frozen question IDs and configuration are included. This archive
contains evaluation artifacts, not the full source notebook, the complete
Cricsheet dataset, or all code needed to recreate every experiment.
Those should be preserved separately before claiming full reproducibility.
"""

(package_dir / "README.md").write_text(
    readme_text.strip() + "\n",
    encoding="utf-8"
)

# Step 7: Include the development pilot files separately if available.
pilot_files = [
    "/content/ipl_generation_pilot_60_questions.csv",
    "/content/ipl_generation_pilot_evidence_inputs.csv",
    "/content/ipl_generation_pilot_answers.csv",
    "/content/ipl_pilot_results_consolidated.csv"
]

pilot_dir = package_dir / "development_pilot"
available_pilot_files = [p for p in pilot_files if os.path.isfile(p)]

if available_pilot_files:
    pilot_dir.mkdir(exist_ok=True)
    for file_path in available_pilot_files:
        shutil.copy2(file_path, pilot_dir / os.path.basename(file_path))

# Step 8: Create the ZIP archive.
if archive_path.exists():
    archive_path.unlink()

with zipfile.ZipFile(
    archive_path, mode="w", compression=zipfile.ZIP_DEFLATED
) as zip_out:
    for file_path in package_dir.rglob("*"):
        if file_path.is_file():
            zip_out.write(
                file_path,
                arcname=str(file_path.relative_to(package_dir.parent))
            )

# Step 9: Verify the archive and report its size.
with zipfile.ZipFile(archive_path, "r") as zip_in:
    archive_members = zip_in.namelist()
    bad_member = zip_in.testzip()

if bad_member is not None:
    raise RuntimeError(f"Archive verification failed at: {bad_member}")

print("Archive created:", archive_path)
print("Archive size (MB):", round(archive_path.stat().st_size / (1024 ** 2), 2))
print("Files included:", len(archive_members))
print("Development pilot files included:", len(available_pilot_files))
print("Archive integrity check: passed")
print("\nContents:")
for member in archive_members:
    print("-", member)


Archive created: /content/ipl_adaptive_rag_research_package.zip
Archive size (MB): 0.04
Files included: 22
Development pilot files included: 4
Archive integrity check: passed

Contents:
- ipl_adaptive_rag_research_package/ipl_winner_extraction_summary.csv
- ipl_adaptive_rag_research_package/ipl_fresh_evaluation_frozen_ids.csv
- ipl_adaptive_rag_research_package/ipl_winner_extraction_comparison.csv
- ipl_adaptive_rag_research_package/ipl_fresh_five_method_by_type.csv
- ipl_adaptive_rag_research_package/ipl_fresh_evaluation_mcnemar_tests.csv
- ipl_adaptive_rag_research_package/ipl_fresh_evaluation_generation_by_type.csv
- ipl_adaptive_rag_research_package/ipl_fresh_evaluation_frozen_config.json
- ipl_adaptive_rag_research_package/ipl_fresh_error_summary_by_type.csv
- ipl_adaptive_rag_research_package/ipl_fresh_evaluation_generation_summary.csv
- ipl_adaptive_rag_research_package/ipl_fresh_evaluation_extraction_summary.csv
- ipl_adaptive_rag_research_package/ipl_fresh_five_method_summary.

In [102]:

# STEP 92: DOWNLOAD THE VERIFIED RESEARCH ARCHIVE

from google.colab import files

files.download("/content/ipl_adaptive_rag_research_package.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>